In [1]:
from ollama import chat, Client

"""
Configure register selection, LLM parameters, API access, and automatic recovery settings for specification processing.
"""

### CHANGE ###
LIST_WANT = [('SFENCE.VMA', [1])]
IS_USING_EVERYTHING = True # Set this to True if you want to use all instruction and register rule pairs
### CHANGE ###

MODEL = 'gpt-oss:120b'
IS_LOCAL = False
TEMPERATURE = 1
CONTEXT = 65536
API_KEY = ''
CLIENT = Client(host='https://ollama.com', timeout=1800, headers={'Authorization': API_KEY})
WIDTH = 120

reset_progress = True

# Automatic LLM recovery.
auto_retry = True
checkpoint_every = 10
max_retries = 10
retry_wait_seconds = 30
runner_stop_timeout_seconds = 60

print('====\nDONE\n====')

====
DONE
====


In [2]:
# 1 - CREATE - 1_TEST_CASE_INPUT.txt - START

"""
Combine RISC-V instruction and CSR specifications, select architectural targets and rules based on user configuration, and prepare the input for test case generation with optional filtering.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT_1 = '4_INSTRUCTION/1_OUTPUT/6_INSTRUCTION_TOTAL.txt'
FILE_INPUT_2 = '5_REGISTER/1_OUTPUT/8_REGISTER_TOTAL.txt'
FILE_OUTPUT = '6_TEST_CASE/1_TEST_CASE_INPUT.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_KEEP_ONLY_PYTHON_RULE(MAP_RULE):
    LIST_RULE_ID = sorted(MAP_RULE)

    for EACH_INDEX in range(len(LIST_RULE_ID) - 2):
        RULE_ID_1 = LIST_RULE_ID[EACH_INDEX]
        RULE_ID_3 = LIST_RULE_ID[EACH_INDEX + 2]

        STRING_RULE_1 = MAP_RULE[RULE_ID_1].strip()
        STRING_RULE_3 = MAP_RULE[RULE_ID_3].strip()

        if re.match(r'^In M-mode,', STRING_RULE_1) and re.match(r'^In U-mode,', STRING_RULE_3):
            return {
                EACH_RULE_ID: EACH_RULE_TEXT
                for EACH_RULE_ID, EACH_RULE_TEXT in MAP_RULE.items()
                if EACH_RULE_ID >= RULE_ID_1
            }

    return {}

def FUNCTION_PARSE_BLOCK(TEXT_VALUE: str):
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n').replace('\r', '\n')
    LIST_HEADER = [(MATCH_HEADER.group(1), MATCH_HEADER.group(2).strip(), MATCH_HEADER.end()) for MATCH_HEADER in REGEX_HEADER.finditer(TEXT_VALUE)]
    LIST_POSITION = [EACH_POS for _, _, EACH_POS in LIST_HEADER] + [len(TEXT_VALUE)]
    MAP_BLOCK = {'INSTRUCTION': {}, 'REGISTER': {}}
    for EACH_INDEX, (EACH_KIND, EACH_NAME, EACH_END) in enumerate(LIST_HEADER):
        STRING_BODY = TEXT_VALUE[EACH_END:LIST_POSITION[EACH_INDEX + 1]].strip()
        if EACH_KIND == 'INSTRUCTION':
            MATCH_FORMAT = re.search(r'^FORMAT:\s*(.+)$', STRING_BODY, re.M)
            MAP_RULE = {int(EACH_RULE_NUMBER): EACH_RULE_TEXT.strip() for EACH_RULE_NUMBER, EACH_RULE_TEXT in re.findall(r'RULE\s+(\d+):\s*(.*)', STRING_BODY)}
            if DOES_SKIP_PYTHON == 1:
                MAP_RULE = FUNCTION_REMOVE_PYTHON_RULE(MAP_RULE)
            elif DOES_SKIP_PYTHON == 2:
                MAP_RULE = FUNCTION_KEEP_ONLY_PYTHON_RULE(MAP_RULE)
            MAP_BLOCK[EACH_KIND][EACH_NAME] = {'format': MATCH_FORMAT.group(1).strip() if MATCH_FORMAT else '', 'rules': MAP_RULE}
        else:
            def FUNCTION_GET_VALUE(STRING_KEY):
                MATCH_VALUE = re.search(rf'^{STRING_KEY}:\s*(.+)$', STRING_BODY, re.M)
                return MATCH_VALUE.group(1).strip() if MATCH_VALUE else ''
            MAP_RULE = {int(EACH_RULE_NUMBER): EACH_RULE_TEXT.strip() for EACH_RULE_NUMBER, EACH_RULE_TEXT in re.findall(r'RULE\s+(\d+):\s*(.*)', STRING_BODY)}
            if DOES_SKIP_PYTHON == 1:
                MAP_RULE = FUNCTION_REMOVE_PYTHON_RULE(MAP_RULE)
            elif DOES_SKIP_PYTHON == 2:
                MAP_RULE = FUNCTION_KEEP_ONLY_PYTHON_RULE(MAP_RULE)
            STRING_EXTENSION = FUNCTION_GET_VALUE('EXTENSIONS')
            LIST_EXTENSION = ([EACH_EXTENSION.strip() for EACH_EXTENSION in STRING_EXTENSION.split(',')] if STRING_EXTENSION else [])
            MAP_BLOCK[EACH_KIND][EACH_NAME] = {'number': FUNCTION_GET_VALUE('NUMBER'), 'mode': FUNCTION_GET_VALUE('MODE'), 'width': FUNCTION_GET_VALUE('WIDTH'), 'fields': FUNCTION_GET_VALUE('FIELDS'), 'extensions': LIST_EXTENSION, 'rules': MAP_RULE}
    return MAP_BLOCK

def FUNCTION_RENDER_SELECTED(TEXT_VALUE: str, VALUE_WANT):
    MAP_BLOCK = FUNCTION_PARSE_BLOCK(TEXT_VALUE)
    MAP_BLOCK = FUNCTION_REMOVE_AMO_BLOCK(MAP_BLOCK)
    MAP_INDEX = {EACH_NAME.lower(): (EACH_KIND, EACH_NAME) for EACH_KIND, EACH_MAP_ITEM in MAP_BLOCK.items() for EACH_NAME in EACH_MAP_ITEM}
    LIST_OUTPUT_LINE = []
    for EACH_RAW_KEY, EACH_SELECTED in (VALUE_WANT.items() if isinstance(VALUE_WANT, dict) else VALUE_WANT):
        STRING_KEY = EACH_RAW_KEY.lower()
        if STRING_KEY not in MAP_INDEX:
            continue
        STRING_KIND, STRING_NAME = MAP_INDEX[STRING_KEY]
        MAP_ITEM = MAP_BLOCK[STRING_KIND][STRING_NAME]
        LIST_WANTED = ([EACH_SELECTED] if isinstance(EACH_SELECTED, int) else list(EACH_SELECTED))
        if STRING_KIND == 'INSTRUCTION':
            LIST_OUTPUT_LINE += [f'INSTRUCTION: {STRING_NAME}']
            if MAP_ITEM['format']:
                LIST_OUTPUT_LINE += [f"FORMAT: {MAP_ITEM['format']}"]
        else:
            LIST_OUTPUT_LINE += [f'REGISTER: {STRING_NAME}']
            for EACH_FIELD_KEY in ('number', 'mode', 'width', 'fields'):
                if MAP_ITEM.get(EACH_FIELD_KEY):
                    LIST_OUTPUT_LINE += [f'{EACH_FIELD_KEY.upper()}: {MAP_ITEM[EACH_FIELD_KEY]}']
            if MAP_ITEM.get('extensions'):
                LIST_OUTPUT_LINE += ['EXTENSIONS: ' + ', '.join(MAP_ITEM['extensions'])]
        for EACH_RULE_ID in LIST_WANTED:
            if EACH_RULE_ID in MAP_ITEM['rules']:
                LIST_OUTPUT_LINE += [f"RULE {EACH_RULE_ID}: {MAP_ITEM['rules'][EACH_RULE_ID]}"]
        LIST_OUTPUT_LINE += ['']
    while LIST_OUTPUT_LINE and LIST_OUTPUT_LINE[-1] == '':
        LIST_OUTPUT_LINE.pop()
    return '\n'.join(LIST_OUTPUT_LINE)

def FUNCTION_REMOVE_PYTHON_RULE(MAP_RULE):
    LIST_RULE_ID = sorted(MAP_RULE)
    for EACH_INDEX in range(len(LIST_RULE_ID) - 2):
        RULE_ID_1 = LIST_RULE_ID[EACH_INDEX]
        RULE_ID_3 = LIST_RULE_ID[EACH_INDEX + 2]

        STRING_RULE_1 = MAP_RULE[RULE_ID_1].strip()
        STRING_RULE_3 = MAP_RULE[RULE_ID_3].strip()

        if re.match(r'^In M-mode,', STRING_RULE_1) and re.match(r'^In U-mode,', STRING_RULE_3):
            return {
                EACH_RULE_ID: EACH_RULE_TEXT
                for EACH_RULE_ID, EACH_RULE_TEXT in MAP_RULE.items()
                if EACH_RULE_ID < RULE_ID_1
            }
    return MAP_RULE

def FUNCTION_RENDER_ALL(TEXT_VALUE: str):
    MAP_BLOCK = FUNCTION_PARSE_BLOCK(TEXT_VALUE)
    MAP_BLOCK = FUNCTION_REMOVE_AMO_BLOCK(MAP_BLOCK)
    LIST_OUTPUT_LINE = []
    for EACH_KIND in ('INSTRUCTION', 'REGISTER'):
        for EACH_NAME, MAP_ITEM in MAP_BLOCK[EACH_KIND].items():
            if EACH_KIND == 'INSTRUCTION':
                LIST_OUTPUT_LINE += [f'INSTRUCTION: {EACH_NAME}']
                if MAP_ITEM['format']:
                    LIST_OUTPUT_LINE += [f"FORMAT: {MAP_ITEM['format']}"]
            else:
                LIST_OUTPUT_LINE += [f'REGISTER: {EACH_NAME}']
                for EACH_FIELD_KEY in ('number', 'mode', 'width', 'fields'):
                    if MAP_ITEM.get(EACH_FIELD_KEY):
                        LIST_OUTPUT_LINE += [f'{EACH_FIELD_KEY.upper()}: {MAP_ITEM[EACH_FIELD_KEY]}']
                if MAP_ITEM.get('extensions'):
                    LIST_OUTPUT_LINE += ['EXTENSIONS: ' + ', '.join(MAP_ITEM['extensions'])]
            for EACH_RULE_ID in sorted(MAP_ITEM['rules']):
                LIST_OUTPUT_LINE += [f"RULE {EACH_RULE_ID}: {MAP_ITEM['rules'][EACH_RULE_ID]}"]
            LIST_OUTPUT_LINE += ['']
    while LIST_OUTPUT_LINE and LIST_OUTPUT_LINE[-1] == '':
        LIST_OUTPUT_LINE.pop()
    return '\n'.join(LIST_OUTPUT_LINE)

def FUNCTION_REMOVE_AMO_BLOCK(MAP_BLOCK):
    if not DOES_SKIP_AMO:
        return MAP_BLOCK

    MAP_BLOCK['INSTRUCTION'] = {
        EACH_NAME: EACH_ITEM
        for EACH_NAME, EACH_ITEM in MAP_BLOCK['INSTRUCTION'].items()
        if not EACH_NAME.upper().startswith('AMO')
    }

    return MAP_BLOCK
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_HEADER = re.compile(r'^(INSTRUCTION|REGISTER):\s*(.+?)\s*$', re.M)
DOES_SKIP_PYTHON = 0 # 0 is ALL, 1 is NO PYTHON, 2 is PYTHON ONLY
DOES_SKIP_AMO = False
# VARIABLE - FINISH

# CODE - BEGIN
with open(FILE_INPUT_1, encoding='utf-8') as FILE_HANDLE:
    STRING_CONTENT_1 = FILE_HANDLE.read().strip()

with open(FILE_INPUT_2, encoding='utf-8') as FILE_HANDLE:
    STRING_CONTENT_2 = FILE_HANDLE.read().strip()

STRING_TOTAL = STRING_CONTENT_1 + '\n\n' + STRING_CONTENT_2
STRING_FINAL = FUNCTION_RENDER_ALL(STRING_TOTAL) if IS_USING_EVERYTHING else FUNCTION_RENDER_SELECTED(STRING_TOTAL, LIST_WANT)

#with open(FILE_OUTPUT, 'w', encoding='utf-8') as FILE_HANDLE:
    #FILE_HANDLE.write(STRING_FINAL)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 1 - CREATE - 1_TEST_CASE_INPUT.txt - END

0.00 second(s)

====
DONE
====


In [3]:
# 2 - CREATE - 2_TEST_CASE_RAW.txt & 3_TEST_CASE_REVISED.txt - START

"""
Generate RISC-V assembly test cases with expected results from architectural rules using LLMs, enriched with relevant instruction and CSR metadata, semantic summaries, and test harness information. Normalize the generated test cases and save raw outputs, revised outputs, and prompt histories with checkpointing and automatic recovery support.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from __future__ import annotations
from dataclasses import dataclass
from ollama import chat, Client, ResponseError
from httpx import TransportError
from pathlib import Path
import hashlib, json, os, subprocess
from typing import Dict, List, Optional, Tuple
import re
# IMPORT - FINISH

# PATH - BEGIN
FILE_INPUT = '6_TEST_CASE/1_TEST_CASE_INPUT.txt'
FILE_OUTPUT_REVISED = '6_TEST_CASE/3_TEST_CASE_REVISED.txt'
FILE_OUTPUT_RAW = '6_TEST_CASE/2_TEST_CASE_RAW.txt'
FILE_OUTPUT_HISTORY = '6_TEST_CASE/4_HISTORY_TEST_CASE.txt'
FILE_REGISTER_ALL = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
FILE_INSTRUCTION_ALL = '4_INSTRUCTION/1_OUTPUT/1_INSTRUCTION_ALL.txt'
FILE_INSTRUCTION_SUMMARY = '4_INSTRUCTION/1_OUTPUT/7_INSTRUCTION_SUMMARY.txt'
FILE_REGISTER_SUMMARY = '5_REGISTER/1_OUTPUT/9_REGISTER_SUMMARY.txt'
FILE_CHECKPOINT = '6_TEST_CASE/5_WHICH_LOOP_TEST_CASE.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_DETECT_EXTENSION(EACH_ITEM: RuleItem, STRING_EXISTING_EXTENSION: str = 'NONE') -> str:
    STRING_NAME = (EACH_ITEM.name or '')
    STRING_NAME_LOWER = STRING_NAME.lower()
    STRING_NAME_UPPER = STRING_NAME.upper()
    STRING_RULE_TEXT = EACH_ITEM.rule_text or ''
    STRING_RULE_TEXT_UPPER = STRING_RULE_TEXT.upper()

    STRING_EXISTING_EXTENSION = (STRING_EXISTING_EXTENSION or 'NONE').strip()

    if 'SMDBLTRP' in STRING_RULE_TEXT_UPPER or 'MDT' in STRING_NAME_UPPER:
        return 'Smdbltrp'

    #if (
        #STRING_NAME_UPPER in SET_VECTOR_INSTRUCTION
        #or STRING_NAME in SET_VECTOR_REGISTER
        #or STRING_NAME_LOWER in SET_VECTOR_REGISTER_LOWER
    #):
        #if STRING_EXISTING_EXTENSION.upper() == 'NONE':
            #return 'V'
        #return STRING_EXISTING_EXTENSION

    return STRING_EXISTING_EXTENSION

def FUNCTION_GET_EXTENSION_FROM_TEXT(STRING_TEXT: str) -> str:
    MATCH_EXTENSION = re.search(
        r'(?im)^#\s*(?:\d+\s*)?(?:#\s*)?Extension\s+Must\s+Be\s+Active\s*:\s*(.+?)\s*$',
        STRING_TEXT
    )

    if MATCH_EXTENSION:
        return MATCH_EXTENSION.group(1).strip()

    return 'NONE'

def FUNCTION_SANITIZE_TEXT(STRING_VALUE: str) -> str:
    for EACH_CHAR_REMOVE in ['\u200b', '\u200c', '\u200d', '\ufeff']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_REMOVE, '')
    for EACH_CHAR_SPACE in ['\u00a0', '\u202f']:
        STRING_VALUE = STRING_VALUE.replace(EACH_CHAR_SPACE, ' ')
    for EACH_CODEPOINT in range(0xE000, 0xF900):
        STRING_VALUE = STRING_VALUE.replace(chr(EACH_CODEPOINT), '')
    STRING_VALUE = (STRING_VALUE.replace('“', '"').replace('”', '"').replace('←', '<-').replace('→', '->').replace('■', '+'))
    return STRING_VALUE.strip()

def FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR):
    if isinstance(ERROR, (ConnectionError, TransportError)):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    STRING_ERROR = str(ERROR).lower()
    LIST_RETRY_KEYWORD = ['cuda', 'gpu', 'launch timed out', 'ggml_backend_cuda', 'cudastreamsynchronize', 'cannot access memory', 'runner']
    return (STATUS_CODE is not None and STATUS_CODE >= 500) or any(EACH_KEYWORD in STRING_ERROR for EACH_KEYWORD in LIST_RETRY_KEYWORD)

def FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR):
    if isinstance(ERROR, (ConnectionError, TransportError)):
        return True
    STATUS_CODE = getattr(ERROR, 'status_code', None)
    return STATUS_CODE in (408, 429) or (STATUS_CODE is not None and STATUS_CODE >= 500)

def FUNCTION_STOP_LOCAL_RUNNER(MODEL):
    print(f'[RECOVERY] Stopping Ollama runner for {MODEL!r}...', flush=True)
    try:
        COMPLETED_PROCESS = subprocess.run(['ollama', 'stop', MODEL], check=False, capture_output=True, text=True, timeout=runner_stop_timeout_seconds)
        if COMPLETED_PROCESS.returncode != 0:
            STRING_ERROR = COMPLETED_PROCESS.stderr.strip()
            if STRING_ERROR:
                print(f'[RECOVERY] ollama stop returned {COMPLETED_PROCESS.returncode}: {STRING_ERROR}', flush=True)
    except Exception as ERROR:
        print(f'[RECOVERY] Could not stop runner cleanly: {ERROR}', flush=True)

def FUNCTION_CALL_LLM(STRING_PROMPT: str) -> str:
    for NUMBER_ATTEMPT in range(1, max_retries + 1):
        try:
            if IS_LOCAL:
                return chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content
            return CLIENT.chat(model=MODEL, messages=[{'role': 'user', 'content': STRING_PROMPT}], options={'temperature': TEMPERATURE, 'num_ctx': CONTEXT}, keep_alive='30m').message.content
        except (ResponseError, ConnectionError, TransportError) as ERROR:
            IS_RETRYABLE = FUNCTION_IS_RETRYABLE_LOCAL_ERROR(ERROR) if IS_LOCAL else FUNCTION_IS_RETRYABLE_ONLINE_ERROR(ERROR)
            if not auto_retry or not IS_RETRYABLE or NUMBER_ATTEMPT >= max_retries:
                print(f'[RECOVERY] Giving up after attempt {NUMBER_ATTEMPT}/{max_retries}.', flush=True)
                raise
            STRING_MODE = 'LOCAL' if IS_LOCAL else 'ONLINE'
            print(f'[RECOVERY] {STRING_MODE} request failed on attempt {NUMBER_ATTEMPT}/{max_retries}: {ERROR}', flush=True)
            if IS_LOCAL:
                FUNCTION_STOP_LOCAL_RUNNER(MODEL)
            print(f'[RECOVERY] Waiting {retry_wait_seconds} seconds, then retrying the SAME request automatically...', flush=True)
            time.sleep(retry_wait_seconds)

def FUNCTION_WHOLE_WORD_PATTERN(STRING_WORD: str, FLAGS: int = 0) -> re.Pattern:
    return re.compile(
        r'(?<![A-Za-z0-9_])' + re.escape(STRING_WORD) + r'(?![A-Za-z0-9_])',
        FLAGS
    )

def FUNCTION_PARSE_REGISTER_NAME_FILE(TEXT_VALUE: str) -> Dict[str, Tuple[str, str, Optional[str], Optional[str]]]:
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n').strip()
    LIST_BLOCK = re.split(r'\n\s*\n', TEXT_VALUE)
    MAP_OUTPUT: Dict[str, Tuple[str, str, Optional[str], Optional[str]]] = {}
    for EACH_BLOCK in LIST_BLOCK:
        LIST_LINE = [EACH_LINE.strip() for EACH_LINE in EACH_BLOCK.splitlines() if EACH_LINE.strip()]
        if not LIST_LINE:
            continue
        STRING_REGISTER = None
        STRING_NUMBER = None
        STRING_FIELDS = None
        STRING_MODE = None  # NEW
        for EACH_LINE in LIST_LINE:
            if EACH_LINE.startswith('REGISTER:') and not EACH_LINE.startswith('REGISTER: mcause'):
                STRING_REGISTER = EACH_LINE.split(':', 1)[1].strip()
            elif EACH_LINE.startswith('NUMBER:'):
                STRING_NUMBER = EACH_LINE.split(':', 1)[1].strip()
            elif EACH_LINE.startswith('MODE:'):  # NEW
                STRING_MODE = EACH_LINE.split(':', 1)[1].strip()
            elif EACH_LINE.startswith('FIELDS:'):
                STRING_VALUE = EACH_LINE.split(':', 1)[1].strip()
                if STRING_VALUE != '-':
                    STRING_FIELDS = STRING_VALUE
        if STRING_REGISTER and STRING_NUMBER:
            STRING_HEX = STRING_NUMBER
            try:
                VALUE_INT = int(STRING_NUMBER, 0)
                STRING_HEX = f'0x{VALUE_INT:X}'
            except ValueError:
                pass
            MAP_OUTPUT[STRING_REGISTER.lower()] = (STRING_REGISTER, STRING_HEX, STRING_FIELDS, STRING_MODE)  # CHANGED
    return MAP_OUTPUT

def FUNCTION_PARSE_INSTRUCTION_NAME_FILE(TEXT_VALUE: str) -> Dict[str, Tuple[str, str]]:
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n').strip()
    LIST_BLOCK = re.split(r'\n\s*\n', TEXT_VALUE)
    MAP_OUTPUT: Dict[str, Tuple[str, str]] = {}
    for EACH_BLOCK in LIST_BLOCK:
        LIST_LINE = [EACH_LINE.strip() for EACH_LINE in EACH_BLOCK.splitlines() if EACH_LINE.strip()]
        STRING_INSTRUCTION = None
        STRING_FORMAT = None
        for EACH_LINE in LIST_LINE:
            if EACH_LINE.startswith('INSTRUCTION:'):
                STRING_INSTRUCTION = EACH_LINE.split(':', 1)[1].strip()
            elif EACH_LINE.startswith('FORMAT:'):
                STRING_FORMAT = EACH_LINE.split(':', 1)[1].strip()
        if STRING_INSTRUCTION and STRING_FORMAT:
            MAP_OUTPUT[STRING_INSTRUCTION.lower()] = (STRING_INSTRUCTION, STRING_FORMAT)
    return MAP_OUTPUT

def FUNCTION_PARSE_SUMMARY_FILE_INSTRUCTION(TEXT_VALUE: str) -> Dict[str, str]:
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n')
    MAP_OUTPUT: Dict[str, str] = {}
    INSTRUCTION_CURRENT: Optional[str] = None
    IS_COLLECTING = False
    LIST_BUFFER: List[str] = []
    def FUNCTION_FLUSH():
        nonlocal INSTRUCTION_CURRENT, IS_COLLECTING, LIST_BUFFER
        if INSTRUCTION_CURRENT and LIST_BUFFER:
            MAP_OUTPUT[INSTRUCTION_CURRENT.lower()] = ' '.join(EACH_LINE.strip() for EACH_LINE in LIST_BUFFER if EACH_LINE.strip()).strip()
        INSTRUCTION_CURRENT = None
        IS_COLLECTING = False
        LIST_BUFFER = []
    for EACH_RAW_LINE in TEXT_VALUE.splitlines():
        EACH_LINE = EACH_RAW_LINE.strip()
        if not EACH_LINE:
            if IS_COLLECTING:
                FUNCTION_FLUSH()
            continue
        if EACH_LINE.startswith('INSTRUCTION:'):
            FUNCTION_FLUSH()
            INSTRUCTION_CURRENT = EACH_LINE.split(':', 1)[1].strip()
            continue
        if EACH_LINE.startswith('SUMMARY:'):
            IS_COLLECTING = True
            LIST_BUFFER.append(EACH_LINE.split(':', 1)[1].strip())
            continue
        if IS_COLLECTING:
            LIST_BUFFER.append(EACH_LINE)
    FUNCTION_FLUSH()
    return MAP_OUTPUT

def FUNCTION_PARSE_SUMMARY_FILE_REGISTER(TEXT_VALUE: str) -> Dict[str, str]:
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n').strip()
    LIST_BLOCK = re.split(r'\n\s*\n', TEXT_VALUE)
    MAP_OUTPUT: Dict[str, str] = {}
    for EACH_BLOCK in LIST_BLOCK:
        LIST_LINE = [EACH_LINE.strip() for EACH_LINE in EACH_BLOCK.splitlines() if EACH_LINE.strip()]
        STRING_REGISTER = None
        STRING_SUMMARY = None
        for EACH_LINE in LIST_LINE:
            if EACH_LINE.startswith('REGISTER:'):
                STRING_REGISTER = EACH_LINE.split(':', 1)[1].strip()
            elif EACH_LINE.startswith('SUMMARY:'):
                STRING_SUMMARY = EACH_LINE.split(':', 1)[1].strip()
        if STRING_REGISTER and STRING_SUMMARY:
            MAP_OUTPUT[STRING_REGISTER.lower()] = STRING_SUMMARY
    return MAP_OUTPUT

def FUNCTION_PARSE_RULE_ITEM(TEXT_VALUE: str) -> List[RuleItem]:
    TEXT_VALUE = TEXT_VALUE.replace('\r\n', '\n')
    LIST_LINE = TEXT_VALUE.splitlines()
    KIND_CURRENT: Optional[str] = None
    NAME_CURRENT: Optional[str] = None
    NUMBER_CURRENT: Optional[str] = None
    FIELDS_CURRENT: Optional[str] = None
    FORMAT_CURRENT: Optional[str] = None
    LIST_ITEM: List[RuleItem] = []
    EACH_INDEX = 0
    while EACH_INDEX < len(LIST_LINE):
        STRING_LINE = LIST_LINE[EACH_INDEX].strip()
        if not STRING_LINE:
            EACH_INDEX += 1
            continue
        if STRING_LINE.startswith('REGISTER:'):
            KIND_CURRENT = 'REGISTER'
            NAME_CURRENT = STRING_LINE.split(':', 1)[1].strip()
            NUMBER_CURRENT = None
            FIELDS_CURRENT = None
            FORMAT_CURRENT = None
            EACH_INDEX += 1
            continue
        if STRING_LINE.startswith('INSTRUCTION:'):
            KIND_CURRENT = 'INSTRUCTION'
            NAME_CURRENT = STRING_LINE.split(':', 1)[1].strip()
            FORMAT_CURRENT = None
            NUMBER_CURRENT = None
            FIELDS_CURRENT = None
            EACH_INDEX += 1
            continue
        if STRING_LINE.startswith('NUMBER:') and KIND_CURRENT == 'REGISTER':
            NUMBER_CURRENT = STRING_LINE.split(':', 1)[1].strip()
            EACH_INDEX += 1
            continue
        if STRING_LINE.startswith('FIELDS:') and KIND_CURRENT == 'REGISTER':
            STRING_VALUE = STRING_LINE.split(':', 1)[1].strip()
            FIELDS_CURRENT = None if STRING_VALUE == '-' else STRING_VALUE
            EACH_INDEX += 1
            continue
        if STRING_LINE.startswith('FORMAT:') and KIND_CURRENT == 'INSTRUCTION':
            FORMAT_CURRENT = STRING_LINE.split(':', 1)[1].strip()
            EACH_INDEX += 1
            continue
        MATCH_RULE = REGEX_RULE_LINE.match(STRING_LINE)
        if MATCH_RULE and KIND_CURRENT and NAME_CURRENT:
            VALUE_RULE_NUMBER = int(MATCH_RULE.group(1))
            LIST_PART = [MATCH_RULE.group(2).strip()]
            EACH_NEXT_INDEX = EACH_INDEX + 1
            while EACH_NEXT_INDEX < len(LIST_LINE):
                STRING_NEXT = LIST_LINE[EACH_NEXT_INDEX].strip()
                if not STRING_NEXT:
                    break
                if (
                    STRING_NEXT.startswith('RULE ')
                    or STRING_NEXT.startswith('REGISTER:')
                    or STRING_NEXT.startswith('INSTRUCTION:')
                    or STRING_NEXT.startswith('NUMBER:')
                    or STRING_NEXT.startswith('FIELDS:')
                    or STRING_NEXT.startswith('FORMAT:')
                ):
                    break
                LIST_PART.append(STRING_NEXT)
                EACH_NEXT_INDEX += 1
            STRING_RULE_TEXT = ' '.join(
                [EACH_PART for EACH_PART in LIST_PART if EACH_PART]
            ).strip()
            LIST_ITEM.append(
                RuleItem(
                    kind=KIND_CURRENT,
                    name=NAME_CURRENT,
                    number=NUMBER_CURRENT,
                    fields=FIELDS_CURRENT,
                    fmt=FORMAT_CURRENT,
                    rule_number=VALUE_RULE_NUMBER,
                    rule_text=STRING_RULE_TEXT,
                )
            )
            EACH_INDEX = EACH_NEXT_INDEX
            continue
        EACH_INDEX += 1
    return LIST_ITEM

def FUNCTION_BUILD_BACKGROUND(ITEM: RuleItem, MAP_REGISTER: Dict[str, Tuple[str, str, Optional[str], Optional[str]]], MAP_INSTRUCTION: Dict[str, Tuple[str, str]], MAP_REGISTER_SUMMARY: Dict[str, str], MAP_INSTRUCTION_SUMMARY: Dict[str, str], LIST_PATTERN_REGISTER: List[Tuple[str, re.Pattern]], LIST_PATTERN_INSTRUCTION: List[Tuple[str, re.Pattern]]) -> Tuple[str, str]:
    LIST_BACKGROUND_LINE: List[str] = []
    SET_FIELD_NAME = FUNCTION_PARSE_FIELD_NAME_SET(ITEM.fields if ITEM.kind == 'REGISTER' else None)
    if ITEM.kind == 'REGISTER':
        if ITEM.name:
            LIST_BACKGROUND_LINE.append(f'REGISTER: {ITEM.name}')
        if ITEM.number:
            LIST_BACKGROUND_LINE.append(f'CSR ADDRESS: {ITEM.number}')
        if ITEM.fields:
            LIST_BACKGROUND_LINE.append(f'FIELDS: {ITEM.fields}')
        STRING_KEY = ITEM.name.lower()
        if STRING_KEY in MAP_REGISTER:
            _, _, _, STRING_MODE = MAP_REGISTER[STRING_KEY]
            if STRING_MODE:
                LIST_BACKGROUND_LINE.append(f'MINIMUM PRIVILEGE REQUIRED TO ACCESS THIS REGISTER: {STRING_MODE}')
        if STRING_KEY in MAP_REGISTER_SUMMARY:
            LIST_BACKGROUND_LINE.append(f'SUMMARY: {MAP_REGISTER_SUMMARY[STRING_KEY]}')
    else:
        LIST_BACKGROUND_LINE.append(f'INSTRUCTION: {ITEM.name}')
        if ITEM.fmt:
            LIST_BACKGROUND_LINE.append(f'FORMAT: {ITEM.fmt}')
        STRING_KEY = ITEM.name.lower()
        STRING_RANGE = FUNCTION_FORMAT_IMMEDIATE_RANGE(STRING_KEY)
        if STRING_RANGE:
            LIST_BACKGROUND_LINE.append(STRING_RANGE)
        if STRING_KEY in MAP_INSTRUCTION_SUMMARY:
            LIST_BACKGROUND_LINE.append(f'SUMMARY: {MAP_INSTRUCTION_SUMMARY[STRING_KEY]}')
    STRING_RULE_LOWER = ITEM.rule_text.lower()
    STRING_MAIN_REGISTER = ITEM.name.lower() if ITEM.kind == 'REGISTER' else None
    for EACH_REGISTER_LOWER, EACH_PATTERN in LIST_PATTERN_REGISTER:
        if STRING_MAIN_REGISTER and EACH_REGISTER_LOWER == STRING_MAIN_REGISTER:
            continue
        if EACH_REGISTER_LOWER in SET_FIELD_NAME:
            continue
        if EACH_PATTERN.search(STRING_RULE_LOWER):
            STRING_OTHER_NAME, STRING_OTHER_HEX, STRING_OTHER_FIELDS, STRING_OTHER_MODE = MAP_REGISTER[EACH_REGISTER_LOWER]
            LIST_BACKGROUND_LINE.append('')
            LIST_BACKGROUND_LINE.append(f'REGISTER: {STRING_OTHER_NAME}')
            LIST_BACKGROUND_LINE.append(f'CSR ADDRESS: {STRING_OTHER_HEX}')
            if STRING_OTHER_FIELDS:
                LIST_BACKGROUND_LINE.append(f'FIELDS: {STRING_OTHER_FIELDS}')
            if STRING_OTHER_MODE:
                LIST_BACKGROUND_LINE.append(
                    f'MINIMUM PRIVILEGE REQUIRED TO ACCESS THIS REGISTER: {STRING_OTHER_MODE}'
                )
            if EACH_REGISTER_LOWER in MAP_REGISTER_SUMMARY:
                LIST_BACKGROUND_LINE.append(f'SUMMARY: {MAP_REGISTER_SUMMARY[EACH_REGISTER_LOWER]}')
    STRING_MAIN_INSTRUCTION = ITEM.name.lower() if ITEM.kind == 'INSTRUCTION' else None
    for EACH_INSTRUCTION_LOWER, EACH_PATTERN in LIST_PATTERN_INSTRUCTION:
        if STRING_MAIN_INSTRUCTION and EACH_INSTRUCTION_LOWER == STRING_MAIN_INSTRUCTION:
            continue
        if EACH_INSTRUCTION_LOWER in SET_FIELD_NAME:
            continue
        if EACH_PATTERN.search(ITEM.rule_text):
            STRING_INSTRUCTION_NAME, STRING_INSTRUCTION_FORMAT = MAP_INSTRUCTION[EACH_INSTRUCTION_LOWER]
            LIST_BACKGROUND_LINE.append('')
            LIST_BACKGROUND_LINE.append(f'INSTRUCTION: {STRING_INSTRUCTION_NAME}')
            LIST_BACKGROUND_LINE.append(f'FORMAT: {STRING_INSTRUCTION_FORMAT}')
            STRING_SUMMARY_KEY = STRING_INSTRUCTION_NAME.lower()
            STRING_RANGE = FUNCTION_FORMAT_IMMEDIATE_RANGE(STRING_SUMMARY_KEY)
            if STRING_RANGE:
                LIST_BACKGROUND_LINE.append(STRING_RANGE)
            if STRING_SUMMARY_KEY in MAP_INSTRUCTION_SUMMARY:
                LIST_BACKGROUND_LINE.append(f'SUMMARY: {MAP_INSTRUCTION_SUMMARY[STRING_SUMMARY_KEY]}')
    STRING_BACKGROUND = '\n'.join(LIST_BACKGROUND_LINE).strip()
    STRING_EXAMPLE_NAME = ITEM.name if ITEM.name else 'UNKNOWN'
    return STRING_BACKGROUND, STRING_EXAMPLE_NAME

def FUNCTION_REVISED_JOIN_ONLY(STRING_MODEL_TEXT: str, STRING_RULE_TEXT: str) -> str:
    STRING_TEXT = STRING_MODEL_TEXT.replace('\r\n', '\n')
    def FUNCTION_NORMALIZE_INLINE_COMMENT(LINE: str) -> str:
        LINE = ' '.join(LINE.strip().split())
        if not LINE:
            return ''
        if LINE.startswith('#'):
            return LINE
        POS = LINE.find('#')
        if POS == -1:
            return LINE
        CODE = LINE[:POS].rstrip()
        CMT = LINE[POS+1:].strip()
        if not CODE:
            return ''
        if not CMT:
            return CODE
        return f'{CODE} # {CMT}'
    LIST_LINE = STRING_TEXT.splitlines()
    LIST_BLOCK = []
    HEADER_CURRENT = None
    INSTR_CURRENT = []
    RESULT_CURRENT = None
    REGEX_EXTENSION_ANY = re.compile(
        r'(?i)^#\s*(?:\d+\s*)?(?:#\s*)?Extension\s+Must\s+Be\s+Active:\s*(.+?)\s*$'
    )
    REGEX_RESULT_ANY = re.compile(
        r'(?i)^#\s*Result\s*:\s*(.+?)\s*$'
    )
    def FUNCTION_FLUSH():
        nonlocal HEADER_CURRENT, INSTR_CURRENT, RESULT_CURRENT
        if HEADER_CURRENT and RESULT_CURRENT:
            BODY = [LN for LN in INSTR_CURRENT if LN.strip()]
            LIST_BLOCK.append((HEADER_CURRENT, BODY, RESULT_CURRENT))
        HEADER_CURRENT = None
        INSTR_CURRENT = []
        RESULT_CURRENT = None
    for RAW in LIST_LINE:
        LINE = RAW.strip()
        if not LINE:
            continue
        M_EXT = REGEX_EXTENSION_ANY.match(LINE)
        if M_EXT:
            FUNCTION_FLUSH()
            EXT = M_EXT.group(1).strip()
            HEADER_CURRENT = f'# Extension Must Be Active: {EXT}'
            continue
        M_RES = REGEX_RESULT_ANY.match(LINE)
        if M_RES:
            if RESULT_CURRENT is None:
                RESULT_CURRENT = f'# Result: {M_RES.group(1).strip()}'
            continue
        if LINE.startswith('#'):
            continue
        NORMAL = FUNCTION_NORMALIZE_INLINE_COMMENT(LINE)
        if NORMAL:
            INSTR_CURRENT.append(NORMAL)
    FUNCTION_FLUSH()
    STRING_RULE_BODY = (STRING_RULE_TEXT or '')
    LIST_FIXED = []
    for HEADER, BODY, RESULT in LIST_BLOCK:
        EXT = HEADER.split(':', 1)[1].strip() if ':' in HEADER else 'NONE'
        if EXT.upper() != 'NONE':
            if not re.search(r'\b' + re.escape(EXT) + r'\b', STRING_RULE_BODY):
                HEADER = '# Extension Must Be Active: NONE'
        LIST_FIXED.append((HEADER, BODY, RESULT))
    OUT = []
    for HEADER, BODY, RESULT in LIST_FIXED:
        OUT.append(HEADER)
        OUT.extend(BODY)
        OUT.append(RESULT)
    return '\n'.join(OUT).strip()

def FUNCTION_HISTORY_WRAP(GLOBAL_NUMBER: int, LOCAL_NUMBER: int, STRING_NAME: str, STRING_PROMPT: str, STRING_RESPONSE: str) -> str:
    STRING_INTRO_FIRST = f' START - {GLOBAL_NUMBER} '.center(WIDTH_PROMPT, '=')
    STRING_INTRO_END = f' END - {GLOBAL_NUMBER} '.center(WIDTH_PROMPT, '=')
    STRING_MID = f'''GLOBAL NUMBER: {GLOBAL_NUMBER}
LOCAL NUMBER: {LOCAL_NUMBER}
NAME: {STRING_NAME}
{'=' * (WIDTH_PROMPT // 2)}
=======
PROMPT:
=======
{'=' * (WIDTH_PROMPT // 2)}
'''
    STRING_TAIL = f'''
{'=' * (WIDTH_PROMPT // 2)}
=========
RESPONSE:
=========
{'=' * (WIDTH_PROMPT // 2)}
{STRING_RESPONSE}
'''
    return STRING_INTRO_FIRST + '\n' + STRING_MID + STRING_PROMPT + STRING_TAIL + STRING_INTRO_END + STRING_SUFFIX

def FUNCTION_RENUMBER_EXTENSION_HEADER_INPLACE(FILE_PATH: str):
    STRING_TEXT = open(FILE_PATH, 'r', encoding='utf-8').read().replace('\r\n', '\n')
    COUNTER_INDEX = 0
    def FUNCTION_REPLACE(MATCH_VALUE: re.Match) -> str:
        nonlocal COUNTER_INDEX
        COUNTER_INDEX += 1
        return f'# {COUNTER_INDEX} {MATCH_VALUE.group(0)}'
    STRING_REVISED = re.sub(r'(?m)^#\s*Extension Must Be Active:.*$', FUNCTION_REPLACE, STRING_TEXT)
    with open(FILE_PATH, 'w', encoding='utf-8', newline='\n') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_REVISED)

def FUNCTION_PARSE_FIELD_NAME_SET(STRING_FIELDS: Optional[str]) -> set[str]:
    if not STRING_FIELDS:
        return set()
    SET_NAME: set[str] = set()
    for EACH_PART in STRING_FIELDS.split(','):
        EACH_PART = EACH_PART.strip()
        if not EACH_PART:
            continue
        if '=' in EACH_PART:
            NAME = EACH_PART.split('=', 1)[1].strip()
            if NAME:
                SET_NAME.add(NAME.lower())
    return SET_NAME

def FUNCTION_GET_IMMEDIATE_RANGE(STRING_MNEMONIC_LOWER: str):
    TUP = MAP_IMMEDIATE_RANGE.get(STRING_MNEMONIC_LOWER)
    if TUP:
        return TUP
    if STRING_MNEMONIC_LOWER.endswith('.vi'):
        return MAP_VECTOR_IMM_RANGE['.vi']
    return None

def FUNCTION_FORMAT_IMMEDIATE_RANGE(STRING_MNEMONIC_LOWER: str):
    TUP = FUNCTION_GET_IMMEDIATE_RANGE(STRING_MNEMONIC_LOWER)
    if not TUP:
        return None
    MIN_V, MAX_V = TUP
    return f'RANGE FOR IMMEDIATE: {MIN_V} until {MAX_V}'

def FUNCTION_DOES_RULE_CONTAIN_LIST_NOT_IMPLEMENTED(STRING_RULE_TEXT: str, LIST_NOT_IMPLEMENTED: List[str]) -> bool:
    STRING_RULE_LOWER = STRING_RULE_TEXT.lower()
    for EACH_WORD in LIST_NOT_IMPLEMENTED:
        if FUNCTION_WHOLE_WORD_PATTERN(EACH_WORD.lower()).search(STRING_RULE_LOWER):
            return True
    return False

def FUNCTION_DOES_RULE_CONTAIN_RESERVED(STRING_RULE_TEXT: str) -> bool:
    HAS_RESERVED = FUNCTION_WHOLE_WORD_PATTERN('reserved').search(STRING_RULE_TEXT) is not None
    HAS_MODE = FUNCTION_WHOLE_WORD_PATTERN('MODE').search(STRING_RULE_TEXT) is not None
    return HAS_RESERVED and not HAS_MODE

def FUNCTION_ADD_TEST_CASE_NAME(STRING_TEXT: str, STRING_NAME: str, STRING_KIND, STRING_RULE, STRING_RULE_NUMBER) -> str:
    LIST_LINE = STRING_TEXT.replace('\r\n', '\n').splitlines()
    LIST_OUTPUT = []
    HAS_INSERTED = False

    for EACH_LINE in LIST_LINE:
        LIST_OUTPUT.append(EACH_LINE)
        if (
            not HAS_INSERTED
            and REGEX_EXTENSION_HEADER.match(EACH_LINE.strip())
        ):
            LIST_OUTPUT.append('# Many important RISC-V extensions are currently active.')
            LIST_OUTPUT.append(f'# Test Case Name: {STRING_NAME}')
            LIST_OUTPUT.append(f'# Test Case Type: {STRING_KIND}')
            LIST_OUTPUT.append(f'# Test Case Rule: RULE {STRING_RULE_NUMBER}: {STRING_RULE}')
            LIST_OUTPUT.append('##### ASSEMBLY #####')
            HAS_INSERTED = True

    return '\n'.join(LIST_OUTPUT).strip()
def FUNCTION_ATOMIC_WRITE_JSON(FILE_PATH, OBJECT_VALUE):
    FILE_PATH = Path(FILE_PATH)
    FILE_TEMP = FILE_PATH.with_suffix(FILE_PATH.suffix + '.tmp')
    with open(FILE_TEMP, 'w', encoding='utf-8') as FILE_HANDLE:
        json.dump(OBJECT_VALUE, FILE_HANDLE, indent=2)
        FILE_HANDLE.write('\n')
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())
    os.replace(FILE_TEMP, FILE_PATH)

def FUNCTION_APPEND_AND_SYNC(FILE_PATH, STRING_VALUE):
    with open(FILE_PATH, 'a', encoding='utf-8', newline='\n') as FILE_HANDLE:
        FILE_HANDLE.write(STRING_VALUE)
        FILE_HANDLE.flush()
        os.fsync(FILE_HANDLE.fileno())

def FUNCTION_RESTORE_COMMITTED_SIZE(FILE_PATH, COMMITTED_SIZE):
    FILE_PATH = Path(FILE_PATH)
    if not FILE_PATH.exists():
        if COMMITTED_SIZE != 0:
            raise RuntimeError(f'Cannot resume: {FILE_PATH} is missing.')
        FILE_PATH.touch()
        return
    CURRENT_SIZE = FILE_PATH.stat().st_size
    if CURRENT_SIZE < COMMITTED_SIZE:
        raise RuntimeError(f'Cannot resume: {FILE_PATH} is shorter than the checkpoint.')
    if CURRENT_SIZE > COMMITTED_SIZE:
        with open(FILE_PATH, 'r+b') as FILE_HANDLE:
            FILE_HANDLE.truncate(COMMITTED_SIZE)
            FILE_HANDLE.flush()
            os.fsync(FILE_HANDLE.fileno())

def FUNCTION_WORK_SHA256():
    HASH_OBJECT = hashlib.sha256()
    for EACH_FILE_PATH in [FILE_INPUT, FILE_REGISTER_ALL, FILE_INSTRUCTION_ALL, FILE_INSTRUCTION_SUMMARY, FILE_REGISTER_SUMMARY]:
        FILE_HASH = hashlib.sha256()
        with open(EACH_FILE_PATH, 'rb') as FILE_HANDLE:
            for EACH_CHUNK in iter(lambda: FILE_HANDLE.read(1024 * 1024), b''):
                FILE_HASH.update(EACH_CHUNK)
        HASH_OBJECT.update(EACH_FILE_PATH.encode('utf-8') + b'\0' + FILE_HASH.digest())
    MAP_CONFIG = {
        'model': MODEL, 'temperature': TEMPERATURE, 'context': CONTEXT,
        'prompt': STRING_PROMPT_TEMPLATE, 'pc_initial': PC_INITIAL, 'width_prompt': WIDTH_PROMPT,
        'filter_flags': [DOES_SKIP_VIRTUAL, DOES_WANT_VIRTUAL, DOES_SKIP_LIST_NOT_IMPLEMENTED, DOES_SKIP_RESERVE, DOES_SKIP_RULE_IMPLEMENTATION, DOES_SKIP_ARBITRARY, DOES_SKIP_FETCH, DOES_SKIP_RESTORE],
        'not_implemented': LIST_NOT_IMPLEMENTED,
        'virtual_pattern': PATTERN_VIRTUAL,
        'immediate_range': MAP_IMMEDIATE_RANGE, 'vector_imm_range': MAP_VECTOR_IMM_RANGE,
        #'vector_instruction': sorted(SET_VECTOR_INSTRUCTION),
        #'vector_register': sorted(SET_VECTOR_REGISTER),
        #'vector_register_lower': sorted(SET_VECTOR_REGISTER_LOWER)
    }
    HASH_OBJECT.update(json.dumps(MAP_CONFIG, sort_keys=True).encode('utf-8'))
    return HASH_OBJECT.hexdigest()

def FUNCTION_NUMBER_RAW_HEADERS(STRING_TEXT):
    global RAW_HEADER_COUNTER
    def FUNCTION_REPLACE(MATCH_VALUE):
        global RAW_HEADER_COUNTER
        RAW_HEADER_COUNTER += 1
        return f'# {RAW_HEADER_COUNTER} {MATCH_VALUE.group(0)}'
    return re.sub(r'(?m)^#\s*Extension Must Be Active:.*$', FUNCTION_REPLACE, STRING_TEXT)

def FUNCTION_CHECKPOINT(NEXT_INDEX):
    global LIST_OUTPUT_RAW, LIST_OUTPUT_CLEAN, STRING_PROMPT_HISTORY, LAST_CHECKPOINT_INDEX
    NUMBER_PENDING = len(LIST_OUTPUT_RAW)
    if NUMBER_PENDING:
        NUMBER_PREVIOUS = GLOBAL_COUNTER - 1 - NUMBER_PENDING
        STRING_SEPARATOR = '\n\n' if NUMBER_PREVIOUS else ''
        STRING_RAW_BATCH = STRING_SEPARATOR + '\n\n'.join(LIST_OUTPUT_RAW)
        STRING_REVISED_BATCH = STRING_SEPARATOR + '\n\n'.join(LIST_OUTPUT_CLEAN)
        STRING_HISTORY_BATCH = STRING_PROMPT_HISTORY.removesuffix(STRING_SUFFIX)
        if NUMBER_PREVIOUS:
            STRING_HISTORY_BATCH = STRING_SUFFIX + STRING_HISTORY_BATCH
        FUNCTION_APPEND_AND_SYNC(FILE_OUTPUT_RAW, FUNCTION_NUMBER_RAW_HEADERS(STRING_RAW_BATCH))
        FUNCTION_APPEND_AND_SYNC(FILE_OUTPUT_REVISED, STRING_REVISED_BATCH)
        FUNCTION_APPEND_AND_SYNC(FILE_OUTPUT_HISTORY, STRING_HISTORY_BATCH)
    NEW_STATE = {
        'next_index': NEXT_INDEX,
        'next_loop': NEXT_INDEX + 1,
        'global_counter': GLOBAL_COUNTER,
        'local_counter': LOCAL_COUNTER,
        'string_name_current': STRING_NAME_CURRENT,
        'is_first_seen': IS_FIRST_SEEN,
        'raw_header_counter': RAW_HEADER_COUNTER,
        'work_sha256': WORK_SHA256,
        'paths': {'raw': FILE_OUTPUT_RAW, 'revised': FILE_OUTPUT_REVISED, 'history': FILE_OUTPUT_HISTORY},
        'file_sizes': {'raw': os.path.getsize(FILE_OUTPUT_RAW), 'revised': os.path.getsize(FILE_OUTPUT_REVISED), 'history': os.path.getsize(FILE_OUTPUT_HISTORY)}
    }
    FUNCTION_ATOMIC_WRITE_JSON(FILE_CHECKPOINT, NEW_STATE)
    LIST_OUTPUT_RAW = []
    LIST_OUTPUT_CLEAN = []
    STRING_PROMPT_HISTORY = ''
    LAST_CHECKPOINT_INDEX = NEXT_INDEX
    print(f'[CHECKPOINT] Scanned {NEXT_INDEX}/{len(LIST_ITEM)} rules; saved {GLOBAL_COUNTER - 1} test case responses.', flush=True)

# FUNCTION - FINISH

# VARIABLE - BEGIN
MAP_IMMEDIATE_RANGE = {
    'addi': (-2048, 2047), 'addiw': (-2048, 2047),
    'xori': (-2048, 2047), 'ori': (-2048, 2047), 'andi': (-2048, 2047),
    'slti': (-2048, 2047), 'sltiu': (-2048, 2047),
    'slli': (0, 63), 'srli': (0, 63), 'srai': (0, 63),
    'slliw': (0, 31), 'srliw': (0, 31), 'sraiw': (0, 31),
    'lui': (0, (1 << 20) - 1), 'auipc': (0, (1 << 20) - 1),
}

MAP_IMMEDIATE_RANGE.update({
    'aes32dsi': (0, 3),
    'aes32dsmi': (0, 3),
    'aes32esi': (0, 3),
    'aes32esmi': (0, 3),
})

MAP_VECTOR_IMM_RANGE = {
    '.vi': (-16, 15),
}

DOES_SKIP_VIRTUAL = False
DOES_WANT_VIRTUAL = False
DOES_SKIP_LIST_NOT_IMPLEMENTED = True
DOES_SKIP_RESERVE = True
DOES_SKIP_RULE_IMPLEMENTATION = True
DOES_SKIP_ARBITRARY = True
DOES_SKIP_FETCH = True
DOES_SKIP_RESTORE = True
WIDTH_PROMPT = 120
STRING_DIVIDER = (('#' * WIDTH_PROMPT) + '\n') * 3
STRING_DIVIDER = STRING_DIVIDER.rstrip('\n')
STRING_SUFFIX = '\n' + STRING_DIVIDER + '\n'
REGEX_REGISTER_HEADER = re.compile(r'(?m)^REGISTER:\s*(.+?)\s*$')
REGEX_NUMBER = re.compile(r'(?m)^NUMBER:\s*(.+?)\s*$')
REGEX_FIELDS = re.compile(r'(?m)^FIELDS:\s*(.+?)\s*$')
REGEX_INSTRUCTION_HEADER = re.compile(r'(?m)^INSTRUCTION:\s*(.+?)\s*$')
REGEX_FORMAT = re.compile(r'(?m)^FORMAT:\s*(.+?)\s*$')
REGEX_RULE_LINE = re.compile(r'^RULE\s+(\d+):\s*(.*)$')
REGEX_EXTENSION_HEADER = re.compile(r'(?i)^#\s*Extension\s+Must\s+Be\s+Active:\s*(.+?)\s*$')
REGEX_RESULT_LINE = re.compile(r'(?i)^#\s*Result\s*:\s*(.+?)\s*$')
STRING_INPUT_TEXT = open(FILE_INPUT, 'r', encoding='utf-8').read().strip()
STRING_REGISTER_ALL_TEXT = open(FILE_REGISTER_ALL, 'r', encoding='utf-8').read().strip()
STRING_INSTRUCTION_ALL_TEXT = open(FILE_INSTRUCTION_ALL, 'r', encoding='utf-8').read().strip()
STRING_INSTRUCTION_SUMMARY_TEXT = open(FILE_INSTRUCTION_SUMMARY, 'r', encoding='utf-8').read()
STRING_REGISTER_SUMMARY_TEXT = open(FILE_REGISTER_SUMMARY, 'r', encoding='utf-8').read()
MAP_REGISTER = FUNCTION_PARSE_REGISTER_NAME_FILE(STRING_REGISTER_ALL_TEXT)
MAP_INSTRUCTION = FUNCTION_PARSE_INSTRUCTION_NAME_FILE(STRING_INSTRUCTION_ALL_TEXT)
MAP_INSTRUCTION_SUMMARY = FUNCTION_PARSE_SUMMARY_FILE_INSTRUCTION(STRING_INSTRUCTION_SUMMARY_TEXT)
MAP_REGISTER_SUMMARY = FUNCTION_PARSE_SUMMARY_FILE_REGISTER(STRING_REGISTER_SUMMARY_TEXT)
LIST_PATTERN_REGISTER = [(EACH_KEY, FUNCTION_WHOLE_WORD_PATTERN(EACH_KEY)) for EACH_KEY in MAP_REGISTER.keys()]
LIST_PATTERN_INSTRUCTION = []

for EACH_KEY in MAP_INSTRUCTION.keys():
    if EACH_KEY in ('and', 'or'):
        LIST_PATTERN_INSTRUCTION.append(
            (EACH_KEY, FUNCTION_WHOLE_WORD_PATTERN(EACH_KEY.upper(), 0))
        )
    else:
        LIST_PATTERN_INSTRUCTION.append(
            (EACH_KEY, FUNCTION_WHOLE_WORD_PATTERN(EACH_KEY, re.IGNORECASE))
        )
        
LIST_OUTPUT_CLEAN: List[str] = []
LIST_OUTPUT_RAW: List[str] = []
STRING_PROMPT_HISTORY = ''
GLOBAL_COUNTER = 1
LOCAL_COUNTER = 1
STRING_NAME_CURRENT: Optional[str] = None
IS_FIRST_SEEN = True

PATTERN_VIRTUAL = r'''
(?:VS|VU)(?:\s*-\s*|\s+)mode
|virtual\s*-?\s*mode
|virtual(?:\s*-\s*|\s+)(?:supervisor|user)
|virtualization
|privilege\s+mode\s+is\s+(?:VS|VU)
|virtual(?:\s*-\s*|\s+)instruction
|\b(?:VS\s*-\s*stage|VU\s*-\s*stage|V=1)\b
'''

REGEX_VIRTUAL = re.compile(PATTERN_VIRTUAL, re.IGNORECASE | re.VERBOSE)
PC_INITIAL = 0x800001f2

LIST_NOT_IMPLEMENTED = ['sireg', 'sireg2', 'sireg3', 'sireg4', 'sireg5', 'sireg6', 'vsireg', 'vsireg2', 'vsireg3', 'vsireg4', 'vsireg5', 'vsireg6', 'jvt', 'mnstatus', 'mncause', 'tcontrol', 'mireg', 'mireg2', 'mireg3', 'mireg4', 'mireg5', 'mireg6', 'mtopei', *[f'hpmcounter{i}' for i in range(19, 32)], *[f'hpmcounter{i}h' for i in range(3, 32)], *[f'mhpmcounter{i}' for i in range(19, 32)], *[f'mhpmcounter{i}h' for i in range(3, 32)], 'pmpcfg1', 'pmpcfg3', 'pmpcfg5', 'pmpcfg7', 'pmpcfg9', 'pmpcfg11', 'pmpcfg13', 'pmpcfg15', 'mctrctl', 'mcyclecfg', 'mcyclecfgh', 'mcycleh', 'medelegh', 'menvcfgh', *[f'mhpmevent{i}h' for i in range(3, 32)], 'midelegh', 'mieh', 'minstretcfg', 'minstretcfgh', 'minstreth', 'mintstatus', 'miph', 'mnepc', 'mnscratch', 'mnxti', 'mscratchcsw', 'mscratchcswl', 'mseccfgh', 'mstatush', 'mtvt', 'mvienh', 'mviph', 'hcontext', 'hedelegh', 'henvcfgh', 'hidelegh', 'htimedeltah', 'hvienh', 'hviph', 'hviprio1h', 'hviprio2h', 'mscontext', 'scontext', 'sctrctl', 'sctrdepth', 'sctrstatus', 'sieh', 'sintstatus', 'siph', 'snxti', 'srmcfg', 'sscratchcsw', 'sscratchcswl', 'stimecmph', 'stopei', 'stvt', 'cycleh', 'instreth', 'timeh', 'uintstatus', 'unxti', 'uscratchcsw', 'uscratchcswl', 'utvt', 'vsieh', 'vsiph', 'vstimecmph', 'vstopei', 'scountovf', 'stimecmp', 'vstimecmp', 'hstateen0h', 'hstateen1h', 'hstateen2h', 'hstateen3h', 'mstateen0h', 'mstateen1h', 'mstateen2h', 'mstateen3h', 'aes32dsi', 'aes32dsmi', 'aes32esi', 'aes32esmi']

STRING_PROMPT_TEMPLATE = '''For RISC-V, based on the rule below, generate me 1 test case in this format:
# Extension Must Be Active: <extension>
<contiguous assembly code>
# Result: <result>

Below is the instruction / register, the rule, and the background information that you might need:

####################
THE INSTRUCTION / REGISTER:
{EACH_ITEM}

THE RULE:
{rule}{bg_block}
####################

Additional notes:
+ Self-check before answering:
    - Identify the exact input value seen by the target instruction.
    - Identify the exact architectural effect of that instruction.
    - Ensure "# Result" matches that exact effect and not a different value.
    - If not, regenerate the test case.'''
# VARIABLE - FINISH

# CODE - BEGIN
@dataclass
class RuleItem:
    kind: str
    name: str
    number: Optional[str]
    fields: Optional[str]
    fmt: Optional[str]
    rule_number: Optional[int]
    rule_text: str

LIST_ITEM = FUNCTION_PARSE_RULE_ITEM(STRING_INPUT_TEXT)

if checkpoint_every < 1 or max_retries < 1:
    raise ValueError('checkpoint_every and max_retries must be at least 1.')

for EACH_FILE_PATH in [FILE_OUTPUT_RAW, FILE_OUTPUT_REVISED, FILE_OUTPUT_HISTORY, FILE_CHECKPOINT]:
    Path(EACH_FILE_PATH).parent.mkdir(parents=True, exist_ok=True)

WORK_SHA256 = FUNCTION_WORK_SHA256()
RAW_HEADER_COUNTER = 0
MAP_OUTPUT_PATH = {'raw': FILE_OUTPUT_RAW, 'revised': FILE_OUTPUT_REVISED, 'history': FILE_OUTPUT_HISTORY}

if reset_progress and Path(FILE_CHECKPOINT).exists():
    Path(FILE_CHECKPOINT).unlink()

if Path(FILE_CHECKPOINT).exists():
    with open(FILE_CHECKPOINT, encoding='utf-8') as FILE_HANDLE:
        STATE = json.load(FILE_HANDLE)
    if STATE.get('work_sha256') != WORK_SHA256:
        raise RuntimeError('Input files or generation settings changed. Set reset_progress = True for a fresh run.')
    if STATE.get('paths') != MAP_OUTPUT_PATH:
        raise RuntimeError('Checkpoint output paths do not match.')
    START_INDEX = STATE['next_index']
    if not 0 <= START_INDEX <= len(LIST_ITEM):
        raise RuntimeError('Checkpoint index is out of range.')
    for EACH_KEY, EACH_FILE_PATH in MAP_OUTPUT_PATH.items():
        FUNCTION_RESTORE_COMMITTED_SIZE(EACH_FILE_PATH, STATE['file_sizes'][EACH_KEY])
    GLOBAL_COUNTER = STATE['global_counter']
    LOCAL_COUNTER = STATE['local_counter']
    STRING_NAME_CURRENT = STATE['string_name_current']
    IS_FIRST_SEEN = STATE['is_first_seen']
    RAW_HEADER_COUNTER = STATE['raw_header_counter']
    LAST_CHECKPOINT_INDEX = START_INDEX
    print(f'[RESUME] Scanned {START_INDEX}/{len(LIST_ITEM)} rules; restored {GLOBAL_COUNTER - 1} test case responses.', flush=True)
else:
    for EACH_FILE_PATH in MAP_OUTPUT_PATH.values():
        with open(EACH_FILE_PATH, 'w', encoding='utf-8'):
            pass
    START_INDEX = 0
    FUNCTION_CHECKPOINT(0)

for EACH_INDEX_ITEM in range(START_INDEX, len(LIST_ITEM)):
    EACH_ITEM = LIST_ITEM[EACH_INDEX_ITEM]
    IS_VIRTUAL = bool(REGEX_VIRTUAL.search(EACH_ITEM.rule_text))
    if DOES_SKIP_VIRTUAL and IS_VIRTUAL:
        continue
    if DOES_WANT_VIRTUAL and not IS_VIRTUAL:
        continue
    if DOES_SKIP_LIST_NOT_IMPLEMENTED and FUNCTION_DOES_RULE_CONTAIN_LIST_NOT_IMPLEMENTED(EACH_ITEM.rule_text, LIST_NOT_IMPLEMENTED):
        continue
    if DOES_SKIP_RESERVE and FUNCTION_DOES_RULE_CONTAIN_RESERVED(EACH_ITEM.rule_text):
        continue
    if DOES_SKIP_RULE_IMPLEMENTATION and 'implementation' in EACH_ITEM.rule_text.lower():
        continue
    if DOES_SKIP_ARBITRARY and 'arbitrary' in EACH_ITEM.rule_text.lower():
        continue
    if DOES_SKIP_FETCH and 'instruction fetch' in EACH_ITEM.rule_text.lower():
        continue
    if DOES_SKIP_RESTORE and (
        'context restore' in EACH_ITEM.rule_text.lower()
        or 'context save' in EACH_ITEM.rule_text.lower()
    ):
        continue
    STRING_BACKGROUND, STRING_EXAMPLE_NAME = FUNCTION_BUILD_BACKGROUND(ITEM=EACH_ITEM, MAP_REGISTER=MAP_REGISTER, MAP_INSTRUCTION=MAP_INSTRUCTION, MAP_REGISTER_SUMMARY=MAP_REGISTER_SUMMARY, MAP_INSTRUCTION_SUMMARY=MAP_INSTRUCTION_SUMMARY, LIST_PATTERN_REGISTER=LIST_PATTERN_REGISTER, LIST_PATTERN_INSTRUCTION=LIST_PATTERN_INSTRUCTION)
    if IS_FIRST_SEEN:
        STRING_NAME_CURRENT = STRING_EXAMPLE_NAME
        LOCAL_COUNTER = 1
        IS_FIRST_SEEN = False
    else:
        if STRING_NAME_CURRENT != STRING_EXAMPLE_NAME:
            STRING_NAME_CURRENT = STRING_EXAMPLE_NAME
            LOCAL_COUNTER = 1
    STRING_BACKGROUND_BLOCK = ''
    if STRING_BACKGROUND:
        STRING_BACKGROUND_BLOCK = (f'\n\nTHE BACKGROUND INFORMATION:\n##########\n{STRING_BACKGROUND}\n##########')
    STRING_PROMPT = STRING_PROMPT_TEMPLATE.format(EACH_ITEM=EACH_ITEM.name, rule=EACH_ITEM.rule_text, bg_block=STRING_BACKGROUND_BLOCK, PC_INITIAL=PC_INITIAL, LIST_NOT_IMPLEMENTED=LIST_NOT_IMPLEMENTED)
    STRING_RAW = FUNCTION_CALL_LLM(STRING_PROMPT)
    STRING_RAW = FUNCTION_SANITIZE_TEXT(STRING_RAW)
    STRING_EXISTING_EXTENSION = FUNCTION_GET_EXTENSION_FROM_TEXT(STRING_RAW)
    STRING_DETECTED_EXTENSION = FUNCTION_DETECT_EXTENSION(EACH_ITEM, STRING_EXISTING_EXTENSION)
    STRING_RAW = re.sub(
        r'(?im)^#\s*(?:\d+\s*)?(?:#\s*)?Extension\s+Must\s+Be\s+Active\s*:\s*.*$',
        f'# Extension Must Be Active: {STRING_DETECTED_EXTENSION}',
        STRING_RAW,
        count=1
    )
    if not re.search(r'(?im)^#\s*Extension\s+Must\s+Be\s+Active\s*:', STRING_RAW):
        STRING_RAW = f'# Extension Must Be Active: {STRING_DETECTED_EXTENSION}\n' + STRING_RAW
    LIST_OUTPUT_RAW.append('\n'.join(' '.join(EACH_LINE.split()) for EACH_LINE in STRING_RAW.splitlines()))
    STRING_REVISED = FUNCTION_REVISED_JOIN_ONLY(STRING_RAW, EACH_ITEM.rule_text)
    
    STRING_REVISED = FUNCTION_ADD_TEST_CASE_NAME(STRING_REVISED, EACH_ITEM.name, EACH_ITEM.kind, EACH_ITEM.rule_text, EACH_ITEM.rule_number)
    LIST_OUTPUT_CLEAN.append(STRING_REVISED)
    STRING_PROMPT_HISTORY += FUNCTION_HISTORY_WRAP(GLOBAL_COUNTER, LOCAL_COUNTER, STRING_EXAMPLE_NAME, STRING_PROMPT, STRING_RAW)
    GLOBAL_COUNTER += 1
    LOCAL_COUNTER += 1

    if len(LIST_OUTPUT_RAW) >= checkpoint_every:
        FUNCTION_CHECKPOINT(EACH_INDEX_ITEM + 1)

if LAST_CHECKPOINT_INDEX != len(LIST_ITEM):
    FUNCTION_CHECKPOINT(len(LIST_ITEM))
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 2 - CREATE - 2_TEST_CASE_RAW.txt & 3_TEST_CASE_REVISED.txt - END

[CHECKPOINT] Scanned 0/408 rules; saved 0 test case responses.


[RECOVERY] ONLINE request failed on attempt 1/10: you (adilahmad17) have reached your session usage limit, upgrade for higher limits: https://ollama.com/upgrade or add usage credits: https://ollama.com/settings (ref: dc0a20de-66ba-4d2e-8399-e3d7035e0757) (status code: 429)


[RECOVERY] Waiting 30 seconds, then retrying the SAME request automatically...


[RECOVERY] ONLINE request failed on attempt 2/10: you (adilahmad17) have reached your session usage limit, upgrade for higher limits: https://ollama.com/upgrade or add usage credits: https://ollama.com/settings (ref: 59f54364-ee0c-4539-ae0b-e77e70c0b5cb) (status code: 429)


[RECOVERY] Waiting 30 seconds, then retrying the SAME request automatically...


[RECOVERY] ONLINE request failed on attempt 3/10: you (adilahmad17) have reached your session usage limit, upgrade for higher limits: https://ollama.com/upgrade or add usage credits: https://ollama.com/settings (ref: 5cfaa4f3-ec45-422f-8818-06f7f8ea78ba) (status code: 429)


... (37 outputs left out)

[CHECKPOINT] Scanned 390/408 rules; saved 350 test case responses.


[CHECKPOINT] Scanned 400/408 rules; saved 360 test case responses.


[CHECKPOINT] Scanned 408/408 rules; saved 368 test case responses.


1 hour(s) 1 minute(s) 54.03 second(s)

====
DONE
====


In [4]:
# 3 - CREATE - 3_TEST_CASE_REVISED_1.txt - START

"""
Refine LLM-generated RISC-V assembly test cases using deterministic Python-based transformations to correct syntax, operands, immediate values, CSR accesses, and vector configurations. Recalculate expected results through lightweight instruction simulation where supported and remove test cases with invalid assembly syntax.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
import re
import ast
import operator
# IMPORT - FINISH

# PATH - BEGIN
FILE_REGISTER_ALL = '5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt'
the_input = '6_TEST_CASE/3_TEST_CASE_REVISED.txt'
the_output = '6_TEST_CASE/3_TEST_CASE_REVISED_1.txt'
# PATH - FINISH

# FUNCTION - BEGIN
def encode_vtype_from_parts(parts):
    vsew_map = {
        8: 0,
        16: 1,
        32: 2,
        64: 3,
    }
    vlmul_map = {
        'm1': 0,
        'm2': 1,
        'm4': 2,
        'm8': 3,
        'mf2': 7,
        'mf4': 6,
        'mf8': 5,
    }
    sew = parse_vtype_sew(parts)
    lmul = parse_vtype_lmul(parts)
    vsew = vsew_map.get(sew, 0)
    vlmul = vlmul_map.get(lmul, 0)
    vta = 0
    vma = 0
    for tok in parts[1:]:
        t = tok.strip().lower()
        if t == 'ta':
            vta = 1
        elif t == 'tu':
            vta = 0
        elif t == 'ma':
            vma = 1
        elif t == 'mu':
            vma = 0
    return (vma << 7) | (vta << 6) | (vsew << 3) | vlmul

def is_privilege_transition_call_line(line: str) -> bool:
    parts = tokenize_asm(line)
    if not parts:
        return False
    return (
        len(parts) >= 2
        and parts[0].lower() == 'call'
        and parts[1].strip() in PRIVILEGE_TRANSITION_CALLS
    )

def is_simple_csr_name(tok: str) -> bool:
    tok = tok.strip()
    if is_reg_name(tok):
        return False
    canonical = canonicalize_csr_name(tok).strip()
    canonical_low = canonical.lower()
    if canonical_low in CSR_CANONICAL_NAME:
        return True
    if is_numeric_token(tok):
        try:
            value = parse_int_token(tok)
            return hex(value).lower() in CSR_CANONICAL_NAME or str(value) in CSR_CANONICAL_NAME
        except Exception:
            return False
    return False

def get_result_lhs(lines):
    for each_line in lines:
        match_result = RESULT_RE2.match(each_line)
        if not match_result:
            continue
        return match_result.group(1).strip()
    return None

def prune_trailing_unused_lines_for_csr_result(lines):
    result_lhs = get_result_lhs(lines)
    if result_lhs is None:
        return lines
    if not is_simple_csr_name(result_lhs):
        return lines
    result_csr = canonicalize_csr_name(result_lhs).strip().lower()
    new_lines = lines[:]
    while True:
        last_asm_index = None
        last_asm_parts = None
        for each_index in range(len(new_lines) - 1, -1, -1):
            parts = tokenize_asm(new_lines[each_index])
            if parts:
                last_asm_index = each_index
                last_asm_parts = parts
                break
        if last_asm_index is None or not last_asm_parts:
            break
        mnem = last_asm_parts[0].lower()
        if mnem == 'csrr' and len(last_asm_parts) >= 3:
            csr_tok = canonicalize_csr_name(last_asm_parts[2].strip()).strip().lower()
            if csr_tok == result_csr:
                if count_asm_lines(new_lines) <= 1:
                    break
                new_lines.pop(last_asm_index)
                continue
        
        if is_privilege_transition_call_line(new_lines[last_asm_index]):
            if count_asm_lines(new_lines) <= 1:
                break
            new_lines.pop(last_asm_index)
            continue
        break
    return new_lines

def fix_scalar_gpr_result_line(line: str, regs) -> str:
    if not line.strip().lower().startswith('# result:'):
        return line
    body = line.split(':', 1)[1].strip()
    if '|' in body:
        return line

    new_parts = []
    changed = False

    for cond in body.split('&'):
        cond = cond.strip()
        m = re.fullmatch(r'([A-Za-z][A-Za-z0-9]*)\s*=\s*([^\s&|#]+)', cond)
        if not m:
            new_parts.append(cond)
            continue

        lhs, rhs = m.group(1), m.group(2)
        if not is_reg_name(lhs):
            new_parts.append(cond)
            continue

        actual = regs[reg_id(lhs)]
        new_rhs = hex(actual) if rhs.lower().startswith('0x') else str(s64(actual))

        if rhs.lower() != new_rhs.lower():
            changed = True

        new_parts.append(f'{lhs} = {new_rhs}')

    if not changed:
        return line

    return '# Result: ' + ' & '.join(new_parts)

def sign_extend_to_xlen(value: int, sew: int) -> int:
    return u64(sext(value, sew))

def count_asm_lines(lines) -> int:
    return sum(1 for EACH_LINE in lines if tokenize_asm(EACH_LINE))

def mask_for_sew(sew: int) -> int:
    return (1 << sew) - 1

def signed_lane(value: int, sew: int) -> int:
    return sext(value, sew)

def unsigned_lane(value: int, sew: int) -> int:
    return value & mask_for_sew(sew)

def normalize_lane(value: int, sew: int) -> int:
    return value & mask_for_sew(sew)

def parse_vtype_sew(parts):
    for tok in parts[1:]:
        tok_low = tok.lower().strip()
        if re.fullmatch(r'e(8|16|32|64)', tok_low):
            return int(tok_low[1:])
    return 64

def parse_masked_flag(parts):
    return any(tok.strip().lower() == 'v0.t' for tok in parts[1:])

def vector_imm_value(tok: str) -> int:
    val = parse_int_token(tok)
    return sext(val, 5)

def has_invalid_amo_offset_addressing(lines):
    for line in lines:
        parts = tokenize_asm(line)
        if not parts:
            continue
        mnem = parts[0].lower()
        if not mnem.startswith('amo'):
            continue
        if len(parts) >= 4 and re.fullmatch(r'-?\d+\([^)]+\)', parts[3]):
            return True
    return False
    
def roundoff_unsigned(value: int, shift: int, vxrm: int) -> int:
    if shift == 0:
        return value
    base = value >> shift
    round_bits = value & ((1 << shift) - 1)
    d = shift
    if vxrm == 0:      # rnu
        r = 1 if ((value >> (d - 1)) & 1) else 0
    elif vxrm == 1:    # rne
        r = 1 if (((value >> (d - 1)) & 1) and (((value >> (d - 2)) & 1) if d > 1 else 0 or (value & ((1 << (d - 1)) - 1) != 0 or (base & 1)))) else 0
    elif vxrm == 2:    # rdn
        r = 0
    else:              # rod
        r = 1 if (base & 1) == 0 and round_bits != 0 else 0
    return base + r

def roundoff_signed(value: int, shift: int, vxrm: int) -> int:
    if shift == 0:
        return value
    base = value >> shift
    low = value & ((1 << shift) - 1)
    if vxrm == 0:      # rnu
        r = 1 if ((value >> (shift - 1)) & 1) else 0
    elif vxrm == 1:    # rne
        bit_d1 = (value >> (shift - 1)) & 1
        lower_nonzero = 1 if shift > 1 and (value & ((1 << (shift - 1)) - 1)) != 0 else 0
        r = 1 if (bit_d1 and (lower_nonzero or (base & 1))) else 0
    elif vxrm == 2:    # rdn
        r = 0
    else:              # rod
        r = 1 if ((base & 1) == 0 and low != 0) else 0
    return base + r

def safe_eval_int_expr(expr: str) -> int:
    def visit(node):
        if isinstance(node, ast.Expression):
            return visit(node.body)
        if isinstance(node, ast.Constant):
            if isinstance(node.value, int):
                return int(node.value)
            raise ValueError('Non-integer constant')
        if isinstance(node, ast.BinOp):
            op_type = type(node.op)
            if op_type not in SAFE_BIN_OP:
                raise ValueError('Unsupported binary operator')
            left = visit(node.left)
            right = visit(node.right)
            return SAFE_BIN_OP[op_type](left, right)
        if isinstance(node, ast.UnaryOp):
            op_type = type(node.op)
            if op_type not in SAFE_UNARY_OP:
                raise ValueError('Unsupported unary operator')
            operand = visit(node.operand)
            return SAFE_UNARY_OP[op_type](operand)
        raise ValueError('Unsupported expression')
    tree = ast.parse(expr, mode='eval')
    return visit(tree)

def normalize_result_rhs(rhs: str) -> str:
    rhs = rhs.strip()
    if is_numeric_token(rhs):
        return rhs
    if not re.fullmatch(r'[\s0-9a-fA-FxX_()+\-~<>&|^]+', rhs):
        return rhs
    try:
        value = safe_eval_int_expr(rhs)
        return hex(u64(value))
    except Exception:
        return rhs

def normalize_result_line(line: str) -> str:
    m = RESULT_RE2.match(line)
    if not m:
        return line
    lhs = m.group(1).strip()
    rhs = m.group(2).strip()
    rhs_new = normalize_result_rhs(rhs)
    return f'# Result: {lhs} = {rhs_new}'

def block_has_float(lines) -> bool:
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        if parts[0].lower().startswith('f'):
            return True
    return False

def fix_vmv_numeric_source_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    mnem = parts[0].lower()
    if mnem != 'vmv.v.x':
        return line
    if len(parts) < 3:
        return line
    src = parts[2].strip()
    if is_numeric_token(src):
        parts[0] = 'vmv.v.i'
        rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
        return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()
    return line

def u64(x: int) -> int:
    return x & MASK64

def u32(x: int) -> int:
    return x & MASK32

def s64(x: int) -> int:
    x &= MASK64
    return x - (1 << 64) if (x >> 63) & 1 else x

def s32(x: int) -> int:
    x &= MASK32
    return x - (1 << 32) if (x >> 31) & 1 else x

def sext(x: int, bits: int) -> int:
    mask = (1 << bits) - 1
    x &= mask
    sign = 1 << (bits - 1)
    return (x ^ sign) - sign

def sext32_to_64(x: int) -> int:
    return u64(sext(x, 32))

def mul_low_xlen(a: int, b: int) -> int:
    return u64(u64(a) * u64(b))

def mulh_signed(a: int, b: int) -> int:
    prod = s64(a) * s64(b)
    return u64(prod >> 64)

def mulh_su(a: int, b: int) -> int:
    prod = s64(a) * u64(b)
    return u64(prod >> 64)

def mulh_unsigned(a: int, b: int) -> int:
    prod = u64(a) * u64(b)
    return u64(prod >> 64)

def div_signed(a: int, b: int) -> int:
    a_s = s64(a)
    b_s = s64(b)
    if b_s == 0:
        return u64(-1)
    if a_s == -(1 << 63) and b_s == -1:
        return u64(a_s)  # overflow case
    return u64(int(a_s / b_s))  # trunc toward 0 in Python for ints

def div_unsigned(a: int, b: int) -> int:
    a_u = u64(a)
    b_u = u64(b)
    if b_u == 0:
        return MASK64
    return u64(a_u // b_u)

def rem_signed(a: int, b: int) -> int:
    a_s = s64(a)
    b_s = s64(b)
    if b_s == 0:
        return u64(a_s)
    if a_s == -(1 << 63) and b_s == -1:
        return u64(0)
    q = int(a_s / b_s)
    r = a_s - q * b_s
    return u64(r)

def rem_unsigned(a: int, b: int) -> int:
    a_u = u64(a)
    b_u = u64(b)
    if b_u == 0:
        return u64(a_u)
    return u64(a_u % b_u)

def divw_signed(a: int, b: int) -> int:
    a_s = s32(a)
    b_s = s32(b)
    if b_s == 0:
        return u64(-1)  # -1 sign-extended to 64 => 0xffff...
    if a_s == -(1 << 31) and b_s == -1:
        return sext32_to_64(a_s)
    return sext32_to_64(int(a_s / b_s))

def divuw(a: int, b: int) -> int:
    a_u = u32(a)
    b_u = u32(b)
    if b_u == 0:
        return sext32_to_64(0xFFFFFFFF)
    return sext32_to_64(a_u // b_u)

def remw_signed(a: int, b: int) -> int:
    a_s = s32(a)
    b_s = s32(b)
    if b_s == 0:
        return sext32_to_64(a_s)
    if a_s == -(1 << 31) and b_s == -1:
        return sext32_to_64(0)
    q = int(a_s / b_s)
    r = a_s - q * b_s
    return sext32_to_64(r)

def remuw(a: int, b: int) -> int:
    a_u = u32(a)
    b_u = u32(b)
    if b_u == 0:
        return sext32_to_64(a_u)
    return sext32_to_64(a_u % b_u)

def parse_int_token(tok: str) -> int:
    tok = tok.strip()
    m = _NUM_HEX_RE.match(tok)
    if m:
        sign = -1 if m.group(1) == '-' else 1
        return sign * int(m.group(2).replace('_', ''), 16)
    return int(tok, 10)

def is_hex_token(tok: str) -> bool:
    return bool(_NUM_HEX_RE.match(tok.strip()))

def format_int_like(tok_style: str, value: int) -> str:
    if is_hex_token(tok_style):
        return hex(u64(value))
    return str(s64(value))

def clamp(v: int, lo: int, hi: int) -> int:
    return lo if v < lo else hi if v > hi else v

def normalize_imm_for_insn(insn: str, imm_tok: str) -> int:
    insn = insn.lower()
    lo, hi = MAP_IMMEDIATE_RANGE[insn]
    if insn in ('lui', 'auipc'):
        raw = parse_int_token(imm_tok)
        return clamp(raw, lo, hi)
    if insn in ('slli', 'srli', 'srai', 'slliw', 'srliw', 'sraiw'):
        raw = parse_int_token(imm_tok)
        return clamp(raw, lo, hi)
    raw = parse_int_token(imm_tok)
    if is_hex_token(imm_tok):
        if 0 <= raw <= 0xFFF:
            return sext(raw, 12)
        return clamp(raw, lo, hi)
    return clamp(raw, lo, hi)

def format_imm_preserve_style(insn: str, imm_tok_style: str, imm_value_semantic: int) -> str:
    insn = insn.lower()
    if not is_hex_token(imm_tok_style):
        return str(int(imm_value_semantic))
    if insn in ('addi', 'addiw', 'andi', 'ori', 'xori', 'slti', 'sltiu'):
        if imm_value_semantic < 0:
            return str(int(imm_value_semantic))
        return hex(int(imm_value_semantic))
    return hex(int(imm_value_semantic))

def is_numeric_token(tok: str) -> bool:
    tok = tok.strip()
    return bool(_NUM_HEX_RE.match(tok) or _NUM_DEC_RE.match(tok))

def build_csr_canonical_map(path: str) -> dict[str, str]:
    mapping = {}
    try:
        with open(path, 'r', encoding='utf-8') as io:
            text = io.read()
    except FileNotFoundError:
        return mapping
    current_name = None
    current_number = None
    for line in text.splitlines():
        m_name = REGISTER_NAME_RE.match(line)
        if m_name:
            current_name = m_name.group(1).strip()
            current_number = None
            continue
        m_num = REGISTER_NUMBER_RE.match(line)
        if m_num and current_name is not None:
            current_number = m_num.group(1).strip()
            canonical = current_name.strip()
            mapping[canonical.lower()] = canonical
            if is_numeric_token(current_number):
                try:
                    value = parse_int_token(current_number)
                    mapping[hex(value).lower()] = canonical
                    mapping[str(value)] = canonical
                except Exception:
                    pass
            current_name = None
            current_number = None
    return mapping

def canonicalize_csr_name(tok: str) -> str:
    tok = tok.strip()
    tok_low = tok.lower()
    if tok_low in CSR_CANONICAL_NAME:
        return CSR_CANONICAL_NAME[tok_low]
    if is_numeric_token(tok):
        try:
            value = parse_int_token(tok)
            if hex(value).lower() in CSR_CANONICAL_NAME:
                return CSR_CANONICAL_NAME[hex(value).lower()]
            if str(value) in CSR_CANONICAL_NAME:
                return CSR_CANONICAL_NAME[str(value)]
        except Exception:
            pass
    return tok

def split_blocks(text: str):
    return [
        b for b in re.split(
            r'\n\s*\n(?=\s*#\s*(?:\d+\s*#\s*)?Extension\s+Must\s+Be\s+Active\s*:)',
            text.strip(),
            flags=re.I | re.M
        )
        if b.strip()
    ]

def fix_la_expression_to_li_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    mnem = parts[0].lower()
    if mnem not in ('la', 'li'):
        return line
    if len(parts) < 3:
        return line
    operand = ' '.join(parts[2:]).strip()
    if not re.fullmatch(r'[\s0-9a-fA-FxX_()+\-~<>&|^*]+', operand):
        return line
    inner = operand
    while inner.startswith('(') and inner.endswith(')'):
        inner = inner[1:-1].strip()
    try:
        value = safe_eval_int_expr(inner if inner else operand)
    except Exception:
        try:
            value = safe_eval_int_expr(operand)
        except Exception:
            return line
    parts[0] = 'li'
    parts = parts[:2] + [hex(u64(value))]
    rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
    return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()

def fix_la_expression_to_li_in_text(text: str) -> str:
    return '\n'.join(fix_la_expression_to_li_in_line(line) for line in text.splitlines())

def strip_comment(line: str) -> str:
    return line.split('#', 1)[0].strip()

def tokenize_asm(line: str):
    line = strip_comment(line)
    if not line:
        return None
    line = re.sub(r'[,\t]+', ' ', line.strip())
    parts = [p for p in line.split(' ') if p]
    if not parts:
        return None
    return parts

def reg_id(r: str) -> int:
    r = r.strip()
    if r not in REG_ALIAS:
        raise ValueError(f'Unknown register: {r}')
    return REG_ALIAS[r]

def block_has_mem(lines):
    for ln in lines:
        parts = tokenize_asm(ln)
        if parts and parts[0].lower() in MEM_MNEMS:
            return True
    return False

def block_has_vector(lines) -> bool:
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        m = parts[0]
        if m.startswith('v'):
            return True
    return False

def get_vector_imm_range(mnem: str):
    for rx, lo, hi in VECTOR_IMM_RULES:
        if rx.match(mnem):
            return lo, hi
    return None

def format_vector_imm_preserve_style(old_tok: str, new_val: int) -> str:
    if is_hex_token(old_tok):
        if new_val < 0:
            return str(int(new_val))
        return hex(int(new_val))
    return str(int(new_val))

def fix_vector_immediate_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    mnem = parts[0]
    if not mnem.startswith('v'):
        return line
    rng = get_vector_imm_range(mnem)
    if not rng:
        return line
    lo, hi = rng
    imm_idx = None
    for i in range(len(parts) - 1, 0, -1):
        t = parts[i]
        if t in ('vm', 'v0'):
            continue
        if is_numeric_token(t):
            imm_idx = i
            break
    if imm_idx is None:
        return line
    old_imm_tok = parts[imm_idx]
    try:
        raw = parse_int_token(old_imm_tok)
    except Exception:
        return line
    new_val = clamp(raw, lo, hi)
    if new_val == raw:
        return line
    parts[imm_idx] = format_vector_imm_preserve_style(old_imm_tok, new_val)
    rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
    return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()

def get_compressed_imm_range(mnem: str):
    m = mnem.lower()
    for rx, lo, hi in COMPRESSED_IMM_RULES:
        if rx.match(m):
            return lo, hi
    return None

def fix_compressed_immediate_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    mnem = parts[0]
    if not mnem.lower().startswith('c.'):
        return line
    rng = get_compressed_imm_range(mnem)
    if not rng:
        return line
    lo, hi = rng
    imm_idx = None
    for i in range(len(parts) - 1, 0, -1):
        if is_numeric_token(parts[i]):
            imm_idx = i
            break
    if imm_idx is None:
        return line
    old = parts[imm_idx]
    try:
        raw = parse_int_token(old)
    except Exception:
        return line
    new_val = clamp(raw, lo, hi)
    if new_val == raw:
        return line
    if is_hex_token(old):
        parts[imm_idx] = str(int(new_val)) if new_val < 0 else hex(int(new_val))
    else:
        parts[imm_idx] = str(int(new_val))
    rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
    return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()

def block_has_compressed(lines) -> bool:
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        if parts[0].lower().startswith('c.'):
            return True
    return False

def exec_block(lines, PC_START: int = 0):
    regs = [0] * 32
    pc = u64(PC_START)
    sim_csrs = {}
    def r(i):
        return 0 if i == 0 else regs[i]
    def w(i, v):
        if i != 0:
            regs[i] = u64(v)
    for raw_line in lines:
        parts = tokenize_asm(raw_line)
        if not parts:
            continue
        mnem = parts[0].lower()
        if mnem == 'li':
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            if -(1 << 31) <= imm <= (1 << 31) - 1:
                w(rd, sext32_to_64(imm))
            else:
                w(rd, imm)
        
            pc = u64(pc + 4)
            continue
        if mnem == 'lui':
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            w(rd, u64(imm << 12))
            pc = u64(pc + 4)
            continue
        if mnem == 'auipc':
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            val32 = u32(pc + (imm << 12))
            w(rd, val32)
            pc = u64(pc + 4)
            continue
        if mnem in ('add.uw', 'sh1add', 'sh2add', 'sh3add',
                    'sh1add.uw', 'sh2add.uw', 'sh3add.uw'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            rs2 = reg_id(parts[3])
            a = r(rs1)
            b = r(rs2)
            if mnem == 'add.uw':
                w(rd, u64(u32(a) + b))
            elif mnem == 'sh1add':
                w(rd, u64((a << 1) + b))
            elif mnem == 'sh2add':
                w(rd, u64((a << 2) + b))
            elif mnem == 'sh3add':
                w(rd, u64((a << 3) + b))
            elif mnem == 'sh1add.uw':
                w(rd, u64((u32(a) << 1) + b))
            elif mnem == 'sh2add.uw':
                w(rd, u64((u32(a) << 2) + b))
            elif mnem == 'sh3add.uw':
                w(rd, u64((u32(a) << 3) + b))
            pc = u64(pc + 4)
            continue
        if mnem == 'slli.uw':
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            sh = parse_int_token(parts[3]) & 0x3F
            w(rd, u64(u32(r(rs1)) << sh))
            pc = u64(pc + 4)
            continue
        if mnem in ('clz', 'ctz', 'cpop', 'clzw', 'ctzw', 'cpopw'):
            rd = reg_id(parts[1])
            a = r(reg_id(parts[2]))
            if mnem == 'clz':
                a64 = u64(a)
                res = 64 if a64 == 0 else (63 - a64.bit_length() + 1)
                w(rd, res)
            elif mnem == 'ctz':
                a64 = u64(a)
                if a64 == 0:
                    res = 64
                else:
                    res = 0
                    while (a64 >> res) & 1 == 0:
                        res += 1
                w(rd, res)
            elif mnem == 'cpop':
                w(rd, bin(u64(a)).count('1'))
            elif mnem == 'clzw':
                a32 = u32(a)
                res = 32 if a32 == 0 else (31 - a32.bit_length() + 1)
                w(rd, res)
            elif mnem == 'ctzw':
                a32 = u32(a)
                if a32 == 0:
                    res = 32
                else:
                    res = 0
                    while (a32 >> res) & 1 == 0:
                        res += 1
                w(rd, res)
            elif mnem == 'cpopw':
                w(rd, bin(u32(a)).count('1'))
            pc = u64(pc + 4)
            continue
        if mnem in ('max', 'maxu', 'min', 'minu'):
            rd = reg_id(parts[1])
            a = r(reg_id(parts[2]))
            b = r(reg_id(parts[3]))
            if mnem == 'max':
                w(rd, u64(max(s64(a), s64(b))))
            elif mnem == 'maxu':
                w(rd, max(u64(a), u64(b)))
            elif mnem == 'min':
                w(rd, u64(min(s64(a), s64(b))))
            elif mnem == 'minu':
                w(rd, min(u64(a), u64(b)))
            pc = u64(pc + 4)
            continue
        if mnem in ('rol', 'ror', 'rolw', 'rorw'):
            rd = reg_id(parts[1])
            a = r(reg_id(parts[2]))
            shamt_width = 0x3F if mnem in ('rol', 'ror') else 0x1F
            sh = u64(r(reg_id(parts[3]))) & shamt_width
            if mnem == 'rol':
                a64 = u64(a)
                w(rd, u64(((a64 << sh) | (a64 >> (64 - sh))) if sh else a64))
            elif mnem == 'ror':
                a64 = u64(a)
                w(rd, u64(((a64 >> sh) | (a64 << (64 - sh))) if sh else a64))
            elif mnem == 'rolw':
                a32 = u32(a)
                res32 = a32 if sh == 0 else (((a32 << sh) | (a32 >> (32 - sh))) & MASK32)
                w(rd, sext32_to_64(res32))
            elif mnem == 'rorw':
                a32 = u32(a)
                res32 = a32 if sh == 0 else (((a32 >> sh) | (a32 << (32 - sh))) & MASK32)
                w(rd, sext32_to_64(res32))
            pc = u64(pc + 4)
            continue
        if mnem in ('rori', 'roriw'):
            rd = reg_id(parts[1])
            a = r(reg_id(parts[2]))
            sh = parse_int_token(parts[3]) & (0x3F if mnem == 'rori' else 0x1F)
            if mnem == 'rori':
                a64 = u64(a)
                w(rd, u64(((a64 >> sh) | (a64 << (64 - sh))) if sh else a64))
            else:
                a32 = u32(a)
                res32 = a32 if sh == 0 else (((a32 >> sh) | (a32 << (32 - sh))) & MASK32)
                w(rd, sext32_to_64(res32))
            pc = u64(pc + 4)
            continue
        if mnem in ('sext.b', 'sext.h', 'zext.h', 'sext.w'):
            rd = reg_id(parts[1])
            a = r(reg_id(parts[2]))
            if mnem == 'sext.b':
                w(rd, u64(sext(a & 0xFF, 8)))
            elif mnem == 'sext.h':
                w(rd, u64(sext(a & 0xFFFF, 16)))
            elif mnem == 'zext.h':
                w(rd, a & 0xFFFF)
            elif mnem == 'sext.w':
                w(rd, sext32_to_64(u32(a)))
            pc = u64(pc + 4)
            continue
        if mnem == 'rev8':
            rd = reg_id(parts[1])
            a = u64(r(reg_id(parts[2])))
            res = 0
            for i in range(8):
                res |= ((a >> (i * 8)) & 0xFF) << ((7 - i) * 8)
            w(rd, u64(res))
            pc = u64(pc + 4)
            continue
        if mnem == 'orc.b':
            rd = reg_id(parts[1])
            a = u64(r(reg_id(parts[2])))
            res = 0
            for i in range(8):
                if (a >> (i * 8)) & 0xFF:
                    res |= 0xFF << (i * 8)
            w(rd, u64(res))
            pc = u64(pc + 4)
            continue
        if mnem in ('bclr', 'bset', 'binv', 'bext'):
            rd = reg_id(parts[1])
            a = u64(r(reg_id(parts[2])))
            sh = u64(r(reg_id(parts[3]))) & 0x3F
            if mnem == 'bclr':
                w(rd, a & ((MASK64) ^ (1 << sh)))
            elif mnem == 'bset':
                w(rd, a | (1 << sh))
            elif mnem == 'binv':
                w(rd, a ^ (1 << sh))
            elif mnem == 'bext':
                w(rd, (a >> sh) & 1)
            pc = u64(pc + 4)
            continue
        if mnem in ('bclri', 'bseti', 'binvi', 'bexti'):
            rd = reg_id(parts[1])
            a = u64(r(reg_id(parts[2])))
            sh = parse_int_token(parts[3]) & 0x3F
            if mnem == 'bclri':
                w(rd, a & ((MASK64) ^ (1 << sh)))
            elif mnem == 'bseti':
                w(rd, a | (1 << sh))
            elif mnem == 'binvi':
                w(rd, a ^ (1 << sh))
            elif mnem == 'bexti':
                w(rd, (a >> sh) & 1)
            pc = u64(pc + 4)
            continue
        if mnem in ('clmul', 'clmulh', 'clmulr'):
            rd = reg_id(parts[1])
            a = u64(r(reg_id(parts[2])))
            b = u64(r(reg_id(parts[3])))
            full = 0
            for i in range(64):
                if (b >> i) & 1:
                    full ^= a << i
            if mnem == 'clmul':
                w(rd, full & MASK64)
            elif mnem == 'clmulh':
                w(rd, (full >> 64) & MASK64)
            elif mnem == 'clmulr':
                w(rd, (full >> 63) & MASK64)
            pc = u64(pc + 4)
            continue
        if mnem in ('czero.eqz', 'czero.nez'):
            rd = reg_id(parts[1])
            a = r(reg_id(parts[2]))
            cond = u64(r(reg_id(parts[3])))
            if mnem == 'czero.eqz':
                w(rd, 0 if cond == 0 else a)
            else:
                w(rd, 0 if cond != 0 else a)
            pc = u64(pc + 4)
            continue
        if mnem == 'nop':
            pc = u64(pc + 4)
            continue
        if mnem == 'mv':
            rd = reg_id(parts[1])
            w(rd, r(reg_id(parts[2])))
            pc = u64(pc + 4)
            continue
        if mnem == 'not':
            rd = reg_id(parts[1])
            w(rd, u64(~r(reg_id(parts[2]))))
            pc = u64(pc + 4)
            continue
        if mnem == 'neg':
            rd = reg_id(parts[1])
            w(rd, u64(-s64(r(reg_id(parts[2])))))
            pc = u64(pc + 4)
            continue
        if mnem == 'negw':
            rd = reg_id(parts[1])
            res32 = (-s32(r(reg_id(parts[2])))) & MASK32
            w(rd, sext32_to_64(res32))
            pc = u64(pc + 4)
            continue
        if mnem == 'seqz':
            rd = reg_id(parts[1])
            w(rd, 1 if u64(r(reg_id(parts[2]))) == 0 else 0)
            pc = u64(pc + 4)
            continue
        if mnem == 'snez':
            rd = reg_id(parts[1])
            w(rd, 1 if u64(r(reg_id(parts[2]))) != 0 else 0)
            pc = u64(pc + 4)
            continue
        if mnem == 'sltz':
            rd = reg_id(parts[1])
            w(rd, 1 if s64(r(reg_id(parts[2]))) < 0 else 0)
            pc = u64(pc + 4)
            continue
        if mnem == 'sgtz':
            rd = reg_id(parts[1])
            w(rd, 1 if s64(r(reg_id(parts[2]))) > 0 else 0)
            pc = u64(pc + 4)
            continue
        if mnem in ('addi', 'andi', 'ori', 'xori', 'slti', 'sltiu'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            imm_tok = parts[3]
            imm = normalize_imm_for_insn(mnem, imm_tok)
            if mnem == 'addi':
                w(rd, r(rs1) + imm)
            elif mnem == 'andi':
                w(rd, r(rs1) & u64(imm))
            elif mnem == 'ori':
                w(rd, r(rs1) | u64(imm))
            elif mnem == 'xori':
                w(rd, r(rs1) ^ u64(imm))
            elif mnem == 'slti':
                w(rd, 1 if s64(r(rs1)) < int(imm) else 0)
            elif mnem == 'sltiu':
                w(rd, 1 if u64(r(rs1)) < u64(imm) else 0)
            pc = u64(pc + 4)
            continue
        if mnem == 'addiw':
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            imm_tok = parts[3]
            imm = normalize_imm_for_insn(mnem, imm_tok)
            res32 = (r(rs1) + imm) & MASK32
            w(rd, sext32_to_64(res32))
            pc = u64(pc + 4)
            continue
        if mnem in ('slli', 'srli', 'srai', 'slliw', 'srliw', 'sraiw'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            sh_tok = parts[3]
            sh = normalize_imm_for_insn(mnem, sh_tok)
            if mnem == 'slli':
                w(rd, r(rs1) << sh)
            elif mnem == 'srli':
                w(rd, u64(r(rs1)) >> sh)
            elif mnem == 'srai':
                w(rd, u64(s64(r(rs1)) >> sh))
            elif mnem == 'slliw':
                res32 = (u32(r(rs1)) << sh) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'srliw':
                res32 = (u32(r(rs1)) >> sh) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'sraiw':
                res32 = s32(r(rs1)) >> sh
                w(rd, sext32_to_64(res32))
            pc = u64(pc + 4)
            continue
        if mnem in ('sll', 'srl', 'sra') and len(parts) >= 4 and is_numeric_token(parts[3]):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            if rd is None or rs1 is None:
                pc = u64(pc + 4)
                continue
            sh = parse_int_token(parts[3]) & 0x3F  # RV64
            if mnem == 'sll':
                w(rd, u64(r(rs1)) << sh)
            elif mnem == 'srl':
                w(rd, u64(r(rs1)) >> sh)
            else:  # sra
                w(rd, u64(s64(r(rs1)) >> sh))
            pc = u64(pc + 4)
            continue
        if mnem in ('add', 'sub', 'and', 'or', 'xor', 'sll', 'srl', 'sra', 'slt', 'sltu'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            rs2 = reg_id(parts[3])
            a = r(rs1)
            b = r(rs2)
            if mnem == 'add':
                w(rd, a + b)
            elif mnem == 'sub':
                w(rd, a - b)
            elif mnem == 'and':
                w(rd, a & b)
            elif mnem == 'or':
                w(rd, a | b)
            elif mnem == 'xor':
                w(rd, a ^ b)
            elif mnem == 'sll':
                w(rd, u64(a) << (u64(b) & 0x3F))
            elif mnem == 'srl':
                w(rd, u64(a) >> (u64(b) & 0x3F))
            elif mnem == 'sra':
                w(rd, u64(s64(a) >> (u64(b) & 0x3F)))
            elif mnem == 'slt':
                w(rd, 1 if s64(a) < s64(b) else 0)
            elif mnem == 'sltu':
                w(rd, 1 if u64(a) < u64(b) else 0)
            pc = u64(pc + 4)
            continue
        if mnem in ('andn', 'orn', 'xnor'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            rs2 = reg_id(parts[3])
            a = r(rs1)
            b = r(rs2)
            if mnem == 'andn':
                w(rd, a & u64(~b))
            elif mnem == 'orn':
                w(rd, a | u64(~b))
            elif mnem == 'xnor':
                w(rd, u64(~(a ^ b)))
            pc = u64(pc + 4)
            continue
        if mnem in ('addw', 'subw', 'sllw', 'srlw', 'sraw'):
            if len(parts) < 4:
                pc = u64(pc + 4)
                continue
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            rs2 = reg_id(parts[3])
            a = r(rs1)
            b = r(rs2)
            sh = u64(b) & 0x1F
            if mnem == 'addw':
                res32 = (u32(a) + u32(b)) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'subw':
                res32 = (u32(a) - u32(b)) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'sllw':
                res32 = (u32(a) << sh) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'srlw':
                res32 = (u32(a) >> sh) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'sraw':
                res32 = s32(a) >> sh
                w(rd, sext32_to_64(res32))
            pc = u64(pc + 4)
            continue
        if mnem == 'c.add':
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, r(rd) + r(rs2))
            pc = u64(pc + 2); continue
        if mnem == 'c.addi':
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            w(rd, r(rd) + imm)
            pc = u64(pc + 2); continue
        if mnem == 'c.addiw':
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            res32 = (u32(r(rd)) + imm) & MASK32
            w(rd, sext32_to_64(res32))
            pc = u64(pc + 2); continue
        if mnem == 'c.addw':
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            res32 = (u32(r(rd)) + u32(r(rs2))) & MASK32
            w(rd, sext32_to_64(res32))
            pc = u64(pc + 2); continue
        if mnem == 'c.sub':
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, r(rd) - r(rs2))
            pc = u64(pc + 2); continue
        if mnem == 'c.subw':
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            res32 = (u32(r(rd)) - u32(r(rs2))) & MASK32
            w(rd, sext32_to_64(res32))
            pc = u64(pc + 2); continue
        if mnem == 'c.and':
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, r(rd) & r(rs2))
            pc = u64(pc + 2); continue
        if mnem == 'c.andi':
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            w(rd, r(rd) & u64(imm))
            pc = u64(pc + 2); continue
        if mnem == 'c.or':
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, r(rd) | r(rs2))
            pc = u64(pc + 2); continue
        if mnem == 'c.xor':
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, r(rd) ^ r(rs2))
            pc = u64(pc + 2); continue
        if mnem == 'c.slli':
            rd = reg_id(parts[1])
            sh = parse_int_token(parts[2]) & 0x3F
            w(rd, u64(r(rd)) << sh)
            pc = u64(pc + 2); continue
        if mnem == 'c.srli':
            rd = reg_id(parts[1])
            sh = parse_int_token(parts[2]) & 0x3F
            w(rd, u64(r(rd)) >> sh)
            pc = u64(pc + 2); continue
        if mnem == 'c.srai':
            rd = reg_id(parts[1])
            sh = parse_int_token(parts[2]) & 0x3F
            w(rd, u64(s64(r(rd)) >> sh))
            pc = u64(pc + 2); continue
        if mnem == 'c.lui':
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            w(rd, u64(sext(imm, 18) << 12))
            pc = u64(pc + 2); continue
        if mnem == 'c.not':
            rd = reg_id(parts[1])
            w(rd, ~u64(r(rd)))
            pc = u64(pc + 2); continue
        if mnem == 'c.mul':
            rd = reg_id(parts[1]); rs2 = reg_id(parts[2])
            w(rd, mul_low_xlen(r(rd), r(rs2)))
            pc = u64(pc + 2); continue
        if mnem in ('mul', 'mulh', 'mulhsu', 'mulhu',
                    'div', 'divu', 'rem', 'remu',
                    'mulw', 'divw', 'divuw', 'remw', 'remuw'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            rs2 = reg_id(parts[3])
            a = r(rs1)
            b = r(rs2)
            if mnem == 'mul':
                w(rd, mul_low_xlen(a, b))
            elif mnem == 'mulh':
                w(rd, mulh_signed(a, b))
            elif mnem == 'mulhsu':
                w(rd, mulh_su(a, b))
            elif mnem == 'mulhu':
                w(rd, mulh_unsigned(a, b))
            elif mnem == 'div':
                w(rd, div_signed(a, b))
            elif mnem == 'divu':
                w(rd, div_unsigned(a, b))
            elif mnem == 'rem':
                w(rd, rem_signed(a, b))
            elif mnem == 'remu':
                w(rd, rem_unsigned(a, b))
            elif mnem == 'mulw':
                res32 = (s32(a) * s32(b)) & MASK32
                w(rd, sext32_to_64(res32))
            elif mnem == 'divw':
                w(rd, divw_signed(a, b))
            elif mnem == 'divuw':
                w(rd, divuw(a, b))
            elif mnem == 'remw':
                w(rd, remw_signed(a, b))
            elif mnem == 'remuw':
                w(rd, remuw(a, b))
            pc = u64(pc + 4)
            continue
        if mnem == 'csrw':
            csr = canonicalize_csr_name(parts[1]).strip().lower()
            if csr in SIMULATABLE_CSRS:
                src = parts[2].strip()
                if is_reg_name(src):
                    sim_csrs[csr] = r(reg_id(src))
                elif is_numeric_token(src):
                    sim_csrs[csr] = u64(parse_int_token(src))
            pc = u64(pc + 4)
            continue
        if mnem == 'csrr':
            rd = reg_id(parts[1])
            csr = canonicalize_csr_name(parts[2]).strip().lower()
            if csr in SIMULATABLE_CSRS and csr in sim_csrs:
                w(rd, sim_csrs[csr])
            pc = u64(pc + 4)
            continue
        pc = u64(pc + 4)
    return regs

def block_has_only_supported_scalar_mnems(lines):
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        mnem = parts[0].lower()
        if mnem.startswith('v'):
            continue
        if mnem not in SUPPORTED_SCALAR_MNEMS:
            return False
    return True
    
def fix_csr_immediate_write_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    mnem = parts[0].lower()
    if mnem != 'csrw':
        return line
    if len(parts) < 3:
        return line
    csr = parts[1].strip()
    src = parts[2].strip()
    if not is_numeric_token(src):
        return line
    parts[0] = 'csrwi'
    rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
    return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()

def exec_vector_block(lines, PC_START: int = 0):
    regs = [0] * 32
    pc = u64(PC_START)
    VECTOR = {f'v{i}': [0] * 256 for i in range(32)}
    VL = 0
    SEW = 64
    VXSAT = 0
    VXRM = 0
    VTYPE = 0xD0
    VLENB = 0x10
    def r(i):
        return 0 if i == 0 else regs[i]
    def w(i, v):
        if i != 0:
            regs[i] = u64(v)
    def active_mask(i: int) -> bool:
        if i >= VL:
            return False
        return (VECTOR['v0'][i] & 1) != 0
    def set_all_lanes(vd: str, value: int):
        lane = normalize_lane(value, SEW)
        for i in range(VL):
            VECTOR[vd][i] = lane
    def set_lane0(vd: str, value: int):
        if VL > 0:
            VECTOR[vd][0] = normalize_lane(value, SEW)
    for raw_line in lines:
        parts = tokenize_asm(raw_line)
        if not parts:
            continue
        mnem = parts[0].lower()
        if mnem == 'li':
            rd = reg_id(parts[1])
            imm = parse_int_token(parts[2])
            if -(1 << 31) <= imm <= (1 << 31) - 1:
                w(rd, sext32_to_64(imm))
            else:
                w(rd, imm)
            pc = u64(pc + 4)
            continue
        if mnem in ('addi', 'andi', 'ori', 'xori', 'slti', 'sltiu'):
            rd = reg_id(parts[1])
            rs1 = reg_id(parts[2])
            imm_tok = parts[3]
            imm = normalize_imm_for_insn(mnem, imm_tok)
            if mnem == 'addi':
                w(rd, r(rs1) + imm)
            elif mnem == 'andi':
                w(rd, r(rs1) & u64(imm))
            elif mnem == 'ori':
                w(rd, r(rs1) | u64(imm))
            elif mnem == 'xori':
                w(rd, r(rs1) ^ u64(imm))
            elif mnem == 'slti':
                w(rd, 1 if s64(r(rs1)) < int(imm) else 0)
            elif mnem == 'sltiu':
                w(rd, 1 if u64(r(rs1)) < u64(imm) else 0)
            pc = u64(pc + 4)
            continue
        if mnem == 'csrr':
            rd = reg_id(parts[1])
            csr = canonicalize_csr_name(parts[2]).lower()
            if csr == 'vtype':
                w(rd, VTYPE)
            elif csr == 'vlenb':
                w(rd, VLENB)
            elif csr == 'vl':
                w(rd, VL)
            elif csr == 'vxrm':
                w(rd, VXRM)
            elif csr == 'vxsat':
                w(rd, VXSAT & 1)
            elif csr == 'vcsr':
                w(rd, ((VXRM & 0x3) << 1) | (VXSAT & 0x1))
            else:
                w(rd, 0)
            pc = u64(pc + 4)
            continue
        if mnem == 'csrw':
            csr = canonicalize_csr_name(parts[1]).lower()
            src = parts[2].strip()
            if is_reg_name(src):
                value = r(reg_id(src))
            elif is_numeric_token(src):
                value = parse_int_token(src)
            else:
                value = 0
            if csr == 'vxrm':
                VXRM = value & 0x3
            elif csr == 'vxsat':
                VXSAT = value & 0x1
            elif csr == 'vcsr':
                VXSAT = value & 0x1
                VXRM = (value >> 1) & 0x3
            pc = u64(pc + 4)
            continue
        if mnem == 'csrwi':
            csr = canonicalize_csr_name(parts[1]).lower()
            imm = parse_int_token(parts[2])
            if csr == 'vxrm':
                VXRM = imm & 0x3
            elif csr == 'vxsat':
                VXSAT = imm & 0x1
            elif csr == 'vcsr':
                VXSAT = imm & 0x1
                VXRM = (imm >> 1) & 0x3
            pc = u64(pc + 4)
            continue
        if mnem == 'vsetvli':
            rd = reg_id(parts[1])
            rs1_name = parts[2].strip()
            rs1 = reg_id(rs1_name)
            SEW = parse_vtype_sew(parts)
            LMUL = parse_vtype_lmul(parts)
            VTYPE = encode_vtype_from_parts(parts)
            vlmax = compute_vlmax(SEW, LMUL)
            if rs1_name in ('x0', 'zero'):
                VL = vlmax
            else:
                VL = min(max(0, r(rs1)), vlmax)
            w(rd, VL)
            pc = u64(pc + 4)
            continue
        if mnem == 'vmset.m':
            vd = normalize_vector_register_name(parts[1])
            for i in range(VL):
                VECTOR[vd][i] = 1
            pc = u64(pc + 4)
            continue
        if mnem == 'vadc.vim':
            vd = normalize_vector_register_name(parts[1])
            vs2 = normalize_vector_register_name(parts[2])
            imm = vector_imm_value(parts[3])
            old_vd = VECTOR[vd][:]
            src = VECTOR[vs2][:]
            for i in range(VL):
                carry = 1 if active_mask(i) else 0
                out = unsigned_lane(src[i], SEW) + unsigned_lane(imm, SEW) + carry
                VECTOR[vd][i] = normalize_lane(out, SEW)
            pc = u64(pc + 4)
            continue
        if mnem == 'vmv.v.x':
            vd = normalize_vector_register_name(parts[1])
            src = parts[2].strip()
            if is_reg_name(src):
                set_all_lanes(vd, r(reg_id(src)))
            elif is_numeric_token(src):
                set_all_lanes(vd, parse_int_token(src))
            else:
                pc = u64(pc + 4)
                continue
            pc = u64(pc + 4)
            continue
        if mnem == 'vmv.v.i':
            vd = normalize_vector_register_name(parts[1])
            imm = parse_int_token(parts[2])
            set_all_lanes(vd, imm)
            pc = u64(pc + 4)
            continue
        if mnem in ('vssra.vv', 'vssrl.vv'):
            vd = normalize_vector_register_name(parts[1])
            vs2 = normalize_vector_register_name(parts[2])
            vs1 = normalize_vector_register_name(parts[3])
            masked = parse_masked_flag(parts)
            old_vd = VECTOR[vd][:]
            a2 = VECTOR[vs2][:]
            a1 = VECTOR[vs1][:]

            for i in range(VL):
                if masked and not active_mask(i):
                    VECTOR[vd][i] = old_vd[i]
                    continue

                sh = unsigned_lane(a1[i], SEW) & (SEW - 1)

                if mnem == 'vssra.vv':
                    value = signed_lane(a2[i], SEW)
                    out = roundoff_signed(value, sh, VXRM)
                else:
                    value = unsigned_lane(a2[i], SEW)
                    out = roundoff_unsigned(value, sh, VXRM)

                VECTOR[vd][i] = normalize_lane(out, SEW)

            pc = u64(pc + 4)
            continue
        if mnem in ('vssra.vx', 'vssrl.vx'):
            vd = normalize_vector_register_name(parts[1])
            vs2 = normalize_vector_register_name(parts[2])
            rs1 = reg_id(parts[3])
            masked = parse_masked_flag(parts)
            old_vd = VECTOR[vd][:]
            a2 = VECTOR[vs2][:]
            sh_scalar = r(rs1) & (SEW - 1)

            for i in range(VL):
                if masked and not active_mask(i):
                    VECTOR[vd][i] = old_vd[i]
                    continue

                if mnem == 'vssra.vx':
                    value = signed_lane(a2[i], SEW)
                    out = roundoff_signed(value, sh_scalar, VXRM)
                else:
                    value = unsigned_lane(a2[i], SEW)
                    out = roundoff_unsigned(value, sh_scalar, VXRM)

                VECTOR[vd][i] = normalize_lane(out, SEW)

            pc = u64(pc + 4)
            continue

        if mnem in ('vssra.vi', 'vssrl.vi'):
            vd = normalize_vector_register_name(parts[1])
            vs2 = normalize_vector_register_name(parts[2])
            sh_imm = parse_int_token(parts[3]) & (SEW - 1)
            masked = parse_masked_flag(parts)
            old_vd = VECTOR[vd][:]
            a2 = VECTOR[vs2][:]

            for i in range(VL):
                if masked and not active_mask(i):
                    VECTOR[vd][i] = old_vd[i]
                    continue

                if mnem == 'vssra.vi':
                    value = signed_lane(a2[i], SEW)
                    out = roundoff_signed(value, sh_imm, VXRM)
                else:
                    value = unsigned_lane(a2[i], SEW)
                    out = roundoff_unsigned(value, sh_imm, VXRM)

                VECTOR[vd][i] = normalize_lane(out, SEW)

            pc = u64(pc + 4)
            continue
        if mnem == 'vmv.s.x':
            vd = normalize_vector_register_name(parts[1])
            rs1 = reg_id(parts[2])
            set_lane0(vd, r(rs1))
            pc = u64(pc + 4)
            continue
        if mnem == 'vmv.x.s':
            rd = reg_id(parts[1])
            vs2 = normalize_vector_register_name(parts[2])
            value = VECTOR[vs2][0] if (VL > 0 and vs2 in VECTOR) else 0
            w(rd, sign_extend_to_xlen(value, SEW))
            pc = u64(pc + 4)
            continue
        if mnem == 'vfirst.m':
            if len(parts) < 3:
                pc = u64(pc + 4)
                continue
            rd_name = parts[1].strip()
            vs2 = normalize_vector_register_name(parts[2])
            if not is_reg_name(rd_name):
                pc = u64(pc + 4)
                continue
            rd = reg_id(rd_name)
            result = -1
            for i in range(VL):
                if (VECTOR[vs2][i] & 1) != 0:
                    result = i
                    break
            w(rd, result)
            pc = u64(pc + 4)
            continue
        if mnem == 'vmclr.m':
            vd = normalize_vector_register_name(parts[1])
            for i in range(VL):
                VECTOR[vd][i] = 0
            pc = u64(pc + 4)
            continue
        if mnem == 'vslidedown.vi':
            vd, vs2 = normalize_vector_register_name(parts[1]), normalize_vector_register_name(parts[2])
            imm = vector_imm_value(parts[3])
            offset = max(0, imm)
            old = VECTOR[vs2][:]
            for i in range(VL):
                src = i + offset
                VECTOR[vd][i] = old[src] if src < VL else 0
            pc = u64(pc + 4)
            continue
        if mnem == 'vmsne.vi':
            vd, vs2 = normalize_vector_register_name(parts[1]), normalize_vector_register_name(parts[2])
            imm = vector_imm_value(parts[3])
            src = VECTOR[vs2][:]
            for i in range(VL):
                VECTOR[vd][i] = 1 if signed_lane(src[i], SEW) != imm else 0
            pc = u64(pc + 4)
            continue
        if mnem == 'vmslt.vi':
            vd, vs2 = normalize_vector_register_name(parts[1]), normalize_vector_register_name(parts[2])
            imm = vector_imm_value(parts[3])
            src = VECTOR[vs2][:]
            for i in range(VL):
                VECTOR[vd][i] = 1 if signed_lane(src[i], SEW) < imm else 0
            pc = u64(pc + 4)
            continue
        if mnem == 'vmsltu.vi':
            vd, vs2 = normalize_vector_register_name(parts[1]), normalize_vector_register_name(parts[2])
            imm = vector_imm_value(parts[3]) & mask_for_sew(SEW)
            src = VECTOR[vs2][:]
            for i in range(VL):
                VECTOR[vd][i] = 1 if unsigned_lane(src[i], SEW) < imm else 0
            pc = u64(pc + 4)
            continue
        if mnem in ('vadd.vi', 'vand.vi'):
            vd, vs2 = normalize_vector_register_name(parts[1]), normalize_vector_register_name(parts[2])
            imm = vector_imm_value(parts[3])
            masked = parse_masked_flag(parts)
            old_vd = VECTOR[vd][:]
            src = VECTOR[vs2][:]
            for i in range(VL):
                if masked and not active_mask(i):
                    VECTOR[vd][i] = old_vd[i]
                    continue
                a = src[i]
                if mnem == 'vadd.vi':
                    VECTOR[vd][i] = normalize_lane(signed_lane(a, SEW) + imm, SEW)
                else:
                    VECTOR[vd][i] = normalize_lane(unsigned_lane(a, SEW) & unsigned_lane(imm, SEW), SEW)
            pc = u64(pc + 4)
            continue
        if mnem in ('vadd.vv', 'vand.vv', 'vsub.vv', 'vaadd.vv', 'vaaddu.vv', 'vasub.vv', 'vasubu.vv'):
            vd, vs2, vs1 = normalize_vector_register_name(parts[1]), normalize_vector_register_name(parts[2]), normalize_vector_register_name(parts[3])
            masked = parse_masked_flag(parts)
            old_vd = VECTOR[vd][:]
            a2 = VECTOR[vs2][:]
            a1 = VECTOR[vs1][:]
            for i in range(VL):
                if masked and not active_mask(i):
                    VECTOR[vd][i] = old_vd[i]
                    continue
                x = a2[i]
                y = a1[i]
                if mnem == 'vadd.vv':
                    out = signed_lane(x, SEW) + signed_lane(y, SEW)
                elif mnem == 'vsub.vv':
                    out = signed_lane(x, SEW) - signed_lane(y, SEW)
                elif mnem == 'vand.vv':
                    out = unsigned_lane(x, SEW) & unsigned_lane(y, SEW)
                elif mnem == 'vaadd.vv':
                    out = roundoff_signed(signed_lane(x, SEW) + signed_lane(y, SEW), 1, VXRM)
                elif mnem == 'vaaddu.vv':
                    out = roundoff_unsigned(unsigned_lane(x, SEW) + unsigned_lane(y, SEW), 1, VXRM)
                elif mnem == 'vasub.vv':
                    out = roundoff_signed(signed_lane(x, SEW) - signed_lane(y, SEW), 1, VXRM)
                else:
                    out = roundoff_unsigned((unsigned_lane(x, SEW) - unsigned_lane(y, SEW)) & ((1 << (SEW + 1)) - 1), 1, VXRM)
                VECTOR[vd][i] = normalize_lane(out, SEW)
            pc = u64(pc + 4)
            continue
        def signed_saturation_bounds(sew):
            return -(1 << (sew - 1)), (1 << (sew - 1)) - 1
        if mnem == 'vsadd.vv':
            vd = normalize_vector_register_name(parts[1])
            vs2 = normalize_vector_register_name(parts[2])
            vs1 = normalize_vector_register_name(parts[3])
            masked = parse_masked_flag(parts)
            old_vd = VECTOR[vd][:]
            a2 = VECTOR[vs2][:]
            a1 = VECTOR[vs1][:]
            lo, hi = signed_saturation_bounds(SEW)

            for i in range(VL):
                if masked and not active_mask(i):
                    VECTOR[vd][i] = old_vd[i]
                    continue

                raw = signed_lane(a2[i], SEW) + signed_lane(a1[i], SEW)
                sat = clamp(raw, lo, hi)

                if sat != raw:
                    VXSAT = 1

                VECTOR[vd][i] = normalize_lane(sat, SEW)

            pc = u64(pc + 4)
            continue
        if mnem in ('vadd.vx', 'vand.vx', 'vsub.vx', 'vaadd.vx', 'vaaddu.vx', 'vasub.vx', 'vasubu.vx'):
            vd, vs2 = normalize_vector_register_name(parts[1]), normalize_vector_register_name(parts[2])
            rs1 = reg_id(parts[3])
            masked = parse_masked_flag(parts)
            old_vd = VECTOR[vd][:]
            a2 = VECTOR[vs2][:]
            y = r(rs1)
            for i in range(VL):
                if masked and not active_mask(i):
                    VECTOR[vd][i] = old_vd[i]
                    continue
                x = a2[i]
                if mnem == 'vadd.vx':
                    out = signed_lane(x, SEW) + signed_lane(y, SEW)
                elif mnem == 'vsub.vx':
                    out = signed_lane(x, SEW) - signed_lane(y, SEW)
                elif mnem == 'vand.vx':
                    out = unsigned_lane(x, SEW) & unsigned_lane(y, SEW)
                elif mnem == 'vaadd.vx':
                    out = roundoff_signed(signed_lane(x, SEW) + signed_lane(y, SEW), 1, VXRM)
                elif mnem == 'vaaddu.vx':
                    out = roundoff_unsigned(unsigned_lane(x, SEW) + unsigned_lane(y, SEW), 1, VXRM)
                elif mnem == 'vasub.vx':
                    out = roundoff_signed(signed_lane(x, SEW) - signed_lane(y, SEW), 1, VXRM)
                else:
                    out = roundoff_unsigned((unsigned_lane(x, SEW) - unsigned_lane(y, SEW)) & ((1 << (SEW + 1)) - 1), 1, VXRM)
                VECTOR[vd][i] = normalize_lane(out, SEW)
            pc = u64(pc + 4)
            continue
        pc = u64(pc + 4)
    return regs

def fix_vector_csr_and_mnemonic_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    changed = False
    for i, tok in enumerate(parts):
        if tok in VECTOR_TOKEN_FIX_MAP:
            parts[i] = VECTOR_TOKEN_FIX_MAP[tok]
            changed = True
    if not changed:
        return line
    rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
    return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()

def is_mcause_token(tok: str) -> bool:
    t = tok.strip().lower()
    if t == 'mcause':
        return True
    if is_numeric_token(t):
        try:
            return parse_int_token(t) == 0x342
        except Exception:
            return False
    return False

def try_rewrite_mcause_readback_block(lines):
    last_asm_idx = None
    last_asm_parts = None
    for i in range(len(lines) - 1, -1, -1):
        parts = tokenize_asm(lines[i])
        if parts:
            last_asm_idx = i
            last_asm_parts = parts
            break
    if last_asm_idx is None:
        return False, lines
    mnem = last_asm_parts[0].lower()
    if mnem != 'csrr':
        return False, lines
    if len(last_asm_parts) < 3:
        return False, lines
    rd = last_asm_parts[1].strip()
    csr_tok = last_asm_parts[2].strip()
    if not is_mcause_token(csr_tok):
        return False, lines
    res_idx = None
    res_reg = None
    res_val = None
    for i, ln in enumerate(lines):
        if not ln.strip().lower().startswith('# result:'):
            continue
        m = RESULT_RE2.match(ln)
        if not m:
            continue
        res_idx = i
        res_reg = m.group(1).strip()
        res_val = m.group(2).strip()
        break
    if res_idx is None:
        return False, lines
    if res_reg != rd:
        return False, lines
    if count_asm_lines(lines) <= 1:
        return False, lines
    new_lines = lines[:]
    new_lines.pop(last_asm_idx)
    new_lines[res_idx if res_idx < last_asm_idx else res_idx - 1] = f'# Result: mcause = {res_val}'
    return True, new_lines

def block_has_csr(lines):
    for ln in lines:
        parts = tokenize_asm(ln)
        if parts and parts[0].lower() in CSR_MNEMS:
            return True
    return False

def block_has_only_simulatable_csrs(lines):
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        mnem = parts[0].lower()
        if mnem not in CSR_MNEMS:
            continue
        if mnem == 'csrw' and len(parts) >= 3:
            csr = canonicalize_csr_name(parts[1]).strip().lower()
        elif mnem == 'csrr' and len(parts) >= 3:
            csr = canonicalize_csr_name(parts[2]).strip().lower()
        else:
            return False
        if csr not in SIMULATABLE_CSRS:
            return False
    return True

def fix_immediates_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    mnem = parts[0].lower()
    if mnem not in MAP_IMMEDIATE_RANGE:
        return line
    if mnem in ('lui', 'auipc'):
        if len(parts) < 3:
            return line
        imm_idx = 2
    else:
        if len(parts) < 4:
            return line
        imm_idx = 3
    old_imm_tok = parts[imm_idx]
    if not is_numeric_token(old_imm_tok):
        return line
    imm_sem = normalize_imm_for_insn(mnem, old_imm_tok)
    new_imm_tok = format_imm_preserve_style(mnem, old_imm_tok, imm_sem)
    parts[imm_idx] = new_imm_tok
    rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
    return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()

def is_reg_name(name: str) -> bool:
    name = name.strip()
    return name in REG_ALIAS

def try_rewrite_generic_csr_readback_block(lines):
    INDEX_RESULT = None
    STRING_RESULT_REGISTER = None
    STRING_RESULT_VALUE = None
    for EACH_INDEX, EACH_LINE in enumerate(lines):
        MATCH_RESULT = RESULT_RE2.match(EACH_LINE)
        if not MATCH_RESULT:
            continue
        INDEX_RESULT = EACH_INDEX
        STRING_RESULT_REGISTER = MATCH_RESULT.group(1).strip()
        STRING_RESULT_VALUE = MATCH_RESULT.group(2).strip()
        break

    if INDEX_RESULT is None:
        return False, lines

    if count_asm_lines(lines) <= 1:
        return False, lines

    LAST_ASM_INDEX = None
    LAST_ASM_PARTS = None
    for EACH_INDEX in range(len(lines) - 1, -1, -1):
        PARTS = tokenize_asm(lines[EACH_INDEX])
        if PARTS:
            LAST_ASM_INDEX = EACH_INDEX
            LAST_ASM_PARTS = PARTS
            break

    if LAST_ASM_INDEX is None or not LAST_ASM_PARTS:
        return False, lines
    if LAST_ASM_PARTS[0].lower() != 'csrr':
        return False, lines
    if len(LAST_ASM_PARTS) < 3:
        return False, lines

    STRING_DEST_REGISTER = LAST_ASM_PARTS[1].strip()
    STRING_CSR = LAST_ASM_PARTS[2].strip()

    if STRING_RESULT_REGISTER != STRING_DEST_REGISTER:
        return False, lines

    STRING_CSR = canonicalize_csr_name(STRING_CSR)
    NEW_LINES = lines[:]
    NEW_LINES.pop(LAST_ASM_INDEX)
    NEW_RESULT_INDEX = INDEX_RESULT if INDEX_RESULT < LAST_ASM_INDEX else INDEX_RESULT - 1

    STRING_FULL = f'{STRING_RESULT_REGISTER} = {STRING_RESULT_VALUE}'
    for sep in ('|', '&'):
        parts_r = STRING_FULL.split(sep)
        rebuilt = []
        for p in parts_r:
            p = p.strip()
            m2 = re.match(r'^([A-Za-z_][A-Za-z0-9_]*)\s*(!=|=)\s*(.+)$', p)
            if m2 and m2.group(1).strip() == STRING_DEST_REGISTER:
                rebuilt.append(f'{STRING_CSR} {m2.group(2)} {m2.group(3).strip()}')
            else:
                rebuilt.append(p)
        STRING_FULL = f' {sep} '.join(rebuilt)

    NEW_LINES[NEW_RESULT_INDEX] = f'# Result: {STRING_FULL}'
    return True, NEW_LINES

def rewrite_vector_block_structure(lines):
    new_lines = []
    temp_reg = find_safe_scalar_temp_register(lines)
    for idx, line in enumerate(lines):
        parts = tokenize_asm(line)
        if not parts:
            new_lines.append(line)
            continue
        mnem = parts[0].lower()
        if mnem == 'vmv.v.i' and len(parts) >= 3 and is_numeric_token(parts[2]):
            imm = parse_int_token(parts[2])
            if imm < -16 or imm > 15:
                vd = parts[1]
                new_lines.append(f'li {temp_reg}, {parts[2]}')
                new_lines.append(f'vmv.v.x {vd}, {temp_reg}')
                continue
        if mnem in VECTOR_MEM_MNEMS and len(parts) >= 3:
            mem = parts[2].strip()
            parsed = parse_offset_base_operand(mem)
            if parsed is not None:
                offset, base = parsed
                if offset == 0:
                    parts[2] = f'({base})'
                    new_lines.append(parts[0] + ' ' + ', '.join(parts[1:]))
                else:
                    new_lines.append(f'addi {temp_reg}, {base}, {offset}')
                    parts[2] = f'({temp_reg})'
                    new_lines.append(parts[0] + ' ' + ', '.join(parts[1:]))
                continue
        if mnem.startswith('vse') and len(parts) >= 3:
            vs3 = normalize_vector_register_name(parts[1]).lower()
            mem = parts[2].strip()
            if vs3 == 'v0':
                parsed = parse_offset_base_operand(mem)
                if parsed is not None:
                    offset, base = parsed
                    if offset == 0:
                        new_lines.append(f'vsm.v v0, ({base})')
                    else:
                        new_lines.append(f'addi {temp_reg}, {base}, {offset}')
                        new_lines.append(f'vsm.v v0, ({temp_reg})')
                    continue

                if mem.startswith('(') and mem.endswith(')'):
                    new_lines.append(f'vsm.v v0, {mem}')
                    continue
        reg_to_init = needs_zero_init_for_vector_op(parts)
        if reg_to_init and not vector_register_written_before(new_lines, len(new_lines), reg_to_init):
            new_lines.append(f'vmv.v.i {reg_to_init}, 0')
            new_lines.append(line)
            continue
        new_lines.append(line)
    return new_lines

def block_touches_mstatus(lines) -> bool:
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        for tok in parts[1:]:
            if canonicalize_csr_name(tok).strip().lower() == 'mstatus':
                return True
    return False

def fix_vector_mcause_result_line(
    line: str,
    has_vector: bool,
    keep_mcause_2: bool,
    block_has_mstatus: bool,
) -> str:
    if not has_vector:
        return line
    if keep_mcause_2:
        return line
    if block_has_mstatus:
        return line

    m = re.match(r'^\s*#\s*Result\s*:\s*mcause\s*=\s*(.+?)\s*$', line, re.I)
    if not m:
        return line

    rhs = m.group(1).strip()
    try:
        rhs_norm = normalize_result_rhs(rhs).lower()
    except Exception:
        rhs_norm = rhs.lower()

    if rhs_norm in ('2', '0x2'):
        return '# Result: mcause = 0'
    return line

def parse_offset_base_operand(tok: str):
    tok = tok.strip()
    m = re.fullmatch(r'([+-]?(?:0[xX][0-9a-fA-F_]+|\d+))\(([^()]+)\)', tok)
    if not m:
        return None
    try:
        offset = parse_int_token(m.group(1))
    except Exception:
        return None
    base = m.group(2).strip()
    return offset, base

def vector_block_should_keep_mcause_2(lines) -> bool:
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        mnem = parts[0].lower()
        if mnem in ('vadc.vim', 'vadc.vvm', 'vadc.vxm'):
            if len(parts) >= 2:
                vd = normalize_vector_register_name(parts[1]).lower()
                if vd == 'v0':
                    return True
    return False

def instruction_writes_vector_register(parts, reg_name: str) -> bool:
    if not parts or len(parts) < 2:
        return False
    reg_name = normalize_vector_register_name(reg_name).lower()
    mnem = parts[0].lower()
    if not is_vector_register(parts[1]):
        return False
    vd = normalize_vector_register_name(parts[1]).lower()
    if vd != reg_name:
        return False
    if (
        mnem.startswith('vmv.')
        or mnem.startswith('vle')
        or mnem.startswith('vl')
        or mnem.startswith('vadd')
        or mnem.startswith('vsub')
        or mnem.startswith('vand')
        or mnem.startswith('vor')
        or mnem.startswith('vxor')
        or mnem.startswith('vaadd')
        or mnem.startswith('vasub')
        or mnem.startswith('vadc')
        or mnem.startswith('vms')
        or mnem.startswith('vmclr')
        or mnem.startswith('vmset')
        or mnem.startswith('vmerge')
        or mnem.startswith('vrgather')
        or mnem.startswith('vslide')
    ):
        return True

    return False

def find_last_result_line_index(lines):
    for i, ln in enumerate(lines):
        if ln.strip().lower().startswith('# result:'):
            return i
    return None

def block_has_vfirst_on_v0_without_prior_write(lines) -> bool:
    for i, ln in enumerate(lines):
        parts = tokenize_asm(ln)
        if not parts:
            continue
        mnem = parts[0].lower()
        if mnem != 'vfirst.m':
            continue
        if len(parts) < 3:
            continue
        src_mask = normalize_vector_register_name(parts[2]).lower()
        if src_mask != 'v0':
            continue
        wrote_v0 = False
        for j in range(i):
            prev = tokenize_asm(lines[j])
            if not prev:
                continue
            if instruction_writes_vector_register(prev, 'v0'):
                wrote_v0 = True
                break
        if not wrote_v0:
            return True
    return False

def fix_vfirst_result_if_v0_never_written(lines):
    if not block_has_vfirst_on_v0_without_prior_write(lines):
        return lines
    new_lines = lines[:]
    idx = find_last_result_line_index(new_lines)
    if idx is None:
        return lines
    m = RESULT_RE2.match(new_lines[idx])
    if not m:
        return lines
    lhs = m.group(1).strip()
    new_lines[idx] = f'# Result: {lhs} = -1'
    return new_lines

def fix_vsetvli_x0_x0_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    if parts[0].lower() != 'vsetvli':
        return line
    if len(parts) < 4:
        return line
    rd = parts[1].strip().lower()
    rs1 = parts[2].strip().lower()
    if rd == 'x0' and rs1 in ('x0', 'zero'):
        parts[1] = 't0'
        parts[2] = 'zero'
        rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
        return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()
    return line

def should_drop_block_due_to_bad_syntax(exc: Exception) -> bool:
    return isinstance(exc, (ValueError, IndexError, KeyError))

VECTOR_CSR_NAMES = {'vstart', 'vtype', 'vl', 'vlenb', 'vxrm', 'vxsat', 'vcsr'}

def block_touches_vector_csr(lines):
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        for tok in parts[1:]:
            if canonicalize_csr_name(tok).strip().lower() in VECTOR_CSR_NAMES:
                return True
    return False

def force_vector_extension_header(lines):
    if not (block_has_vector(lines) or block_touches_vector_csr(lines)):
        return lines

    out = []
    replaced = False
    for ln in lines:
        if EXT_HEADER_RE.match(ln):
            out.append('# Extension Must Be Active: V')
            replaced = True
        else:
            out.append(ln)

    if not replaced:
        out.insert(0, '# Extension Must Be Active: V')

    return out

def rewrite_vector_csr_result_to_a0(lines):
    out = []
    for ln in lines:
        m = RESULT_RE2.match(ln)
        if not m:
            out.append(ln)
            continue

        lhs = canonicalize_csr_name(m.group(1).strip()).strip().lower()
        rhs = m.group(2).strip()

        if lhs in VECTOR_CSR_NAMES:
            out.append(f'csrr a0, {lhs}')
            out.append(f'# Result: a0 = {rhs}')
        else:
            out.append(ln)

    return out
    
def rewrite_block(block_text: str, PC_START: int = 0) -> str | None:
    try:
        lines = block_text.splitlines()
        lines = force_vector_extension_header(lines)
        lines = rewrite_vector_block_structure(lines)
        lines = fix_masked_vector_op_with_uninitialized_v0(lines)
        HAS_MSTATUS = block_touches_mstatus(lines)
        HAS_FLOAT = block_has_float(lines)
        HAS_VECTOR = block_has_vector(lines)
        KEEP_VECTOR_MCAUSE_2 = vector_block_should_keep_mcause_2(lines)
        fixed_lines = []
        asm_lines = []
        for ln in lines:
            if ln.strip().lower().startswith('# result:'):
                fixed = normalize_result_line(ln)
                if not HAS_FLOAT:
                    fixed = fix_vector_mcause_result_line(
                        fixed,
                        HAS_VECTOR,
                        KEEP_VECTOR_MCAUSE_2,
                        HAS_MSTATUS,
                    )
                fixed_lines.append(fixed)
            else:
                fixed = fix_immediates_in_line(ln)
                fixed = fix_compressed_immediate_in_line(fixed)
                fixed = fix_vector_immediate_in_line(fixed)
                fixed = fix_vector_csr_and_mnemonic_in_line(fixed)
                fixed = fix_vmv_numeric_source_in_line(fixed)
                fixed = fix_vector_vm_operand_in_line(fixed)
                fixed = fix_vadc_mask_operand_in_line(fixed)
                fixed = fix_vector_mask_suffix_in_line(fixed)
                fixed = fix_invalid_vector_4th_operand_in_line(fixed)
                fixed = fix_vsetvli_x0_x0_in_line(fixed)
                fixed = fix_csr_immediate_write_in_line(fixed)
                fixed_lines.append(fixed)
                if tokenize_asm(fixed):
                    asm_lines.append(fixed)
        fixed_lines = fix_vfirst_result_if_v0_never_written(fixed_lines)
        fixed_lines = rewrite_vector_csr_result_to_a0(fixed_lines)
        if not HAS_FLOAT:
            _, fixed_lines = try_rewrite_mcause_readback_block(fixed_lines)
            if not (block_has_csr(asm_lines) and block_has_only_simulatable_csrs(asm_lines)):
                _, fixed_lines = try_rewrite_generic_csr_readback_block(fixed_lines)
                fixed_lines = prune_trailing_unused_lines_for_csr_result(fixed_lines)
        if HAS_FLOAT:
            return '\n'.join(fixed_lines).rstrip()
        if HAS_VECTOR:
            asm_lines = [ln for ln in fixed_lines if tokenize_asm(ln)]
        
            if block_has_only_supported_vector_mnems(asm_lines):
                ok, regs = try_exec_vector_block(asm_lines, PC_START=PC_START)
                if ok and regs is not None:
                    fixed_lines = [
                        fix_scalar_gpr_result_line(ln, regs)
                        if ln.strip().lower().startswith('# result:')
                        else ln
                        for ln in fixed_lines
                    ]
        asm_lines = [ln for ln in fixed_lines if tokenize_asm(ln)]
        if block_has_vector(asm_lines) and block_has_only_supported_vector_mnems(asm_lines):
            ok, regs = try_exec_vector_block(asm_lines, PC_START=PC_START)
            if not ok:
                return '\n'.join(fixed_lines).rstrip()
            out_lines = []
            for ln in fixed_lines:
                if not ln.strip().lower().startswith('# result:'):
                    out_lines.append(ln)
                    continue
                m = RESULT_RE2.match(ln)
                if not m:
                    out_lines.append(ln)
                    continue
                reg_name = m.group(1).strip()
                old_val_tok = m.group(2).strip()
                if not is_reg_name(reg_name):
                    out_lines.append(normalize_result_line(ln))
                    continue
                conditions = [x.strip() for x in re.split(r'\s*&\s*', old_val_tok)]
                new_conditions = []
                can_rewrite_all = True
                for cond in conditions:
                    mm = re.match(r'^([a-zA-Z0-9]+)\s*=\s*(.+)$', cond)
                    if mm:
                        lhs = mm.group(1).strip()
                        rhs = mm.group(2).strip()
                        if not is_reg_name(lhs):
                            can_rewrite_all = False
                            break
                        rid = reg_id(lhs)
                        rhs_new = format_int_like(rhs, regs[rid])
                        new_conditions.append(f'{lhs} = {rhs_new}')
                    else:
                        rid = reg_id(reg_name)
                        rhs_new = format_int_like(old_val_tok, regs[rid])
                        new_conditions = [f'{reg_name} = {rhs_new}']
                        break
                if can_rewrite_all:
                    out_lines.append('# Result: ' + ' & '.join(new_conditions))
                else:
                    out_lines.append(normalize_result_line(ln))
            return '\n'.join(out_lines).rstrip()
        if has_invalid_amo_offset_addressing(fixed_lines):
            return None
        if (
            (block_has_csr(asm_lines) and not block_has_only_simulatable_csrs(asm_lines))
            or block_has_mem(asm_lines)
            or block_has_compressed(asm_lines)
            or not block_has_only_supported_scalar_mnems(asm_lines)
        ):
            return '\n'.join(fixed_lines).rstrip()
        regs = exec_block(asm_lines, PC_START=PC_START)
        out_lines = []
        for ln in fixed_lines:
            if not ln.strip().lower().startswith('# result:'):
                out_lines.append(ln)
                continue
            m = RESULT_RE2.match(ln)
            if not m:
                out_lines.append(ln)
                continue
            reg_name = m.group(1).strip()
            old_val_tok = m.group(2).strip()
            if '|' in old_val_tok:
                out_lines.append(normalize_result_line(ln))
                continue
            conditions = [x.strip() for x in re.split(r'\s*&\s*', old_val_tok)]
            new_conditions = []
            can_rewrite_all = True
            for cond in conditions:
                mm = re.match(r'^([A-Za-z][A-Za-z0-9]*)\s*=\s*(.+)$', cond)
                if mm:
                    lhs = mm.group(1).strip()
                    rhs = mm.group(2).strip()
                else:
                    lhs = reg_name
                    rhs = cond
                if not is_reg_name(lhs):
                    can_rewrite_all = False
                    break
                rid = reg_id(lhs)
                rhs_new = format_int_like(rhs, regs[rid])
                new_conditions.append(f'{lhs} = {rhs_new}')
            if can_rewrite_all:
                out_lines.append('# Result: ' + ' & '.join(new_conditions))
            else:
                out_lines.append(normalize_result_line(ln))
        return '\n'.join(out_lines).rstrip()
    except Exception as exc:
        if should_drop_block_due_to_bad_syntax(exc):
            return None
        raise

def compute_vlmax(sew: int, lmul_tok: str | None = None) -> int:
    vlen = 128
    lmul = 1
    if lmul_tok:
        tok = lmul_tok.lower().strip()
        if tok == 'm1':
            lmul = 1
        elif tok == 'm2':
            lmul = 2
        elif tok == 'm4':
            lmul = 4
        elif tok == 'm8':
            lmul = 8
        elif tok == 'mf2':
            lmul = 0.5
        elif tok == 'mf4':
            lmul = 0.25
        elif tok == 'mf8':
            lmul = 0.125
    return int((vlen * lmul) // sew)

def parse_vtype_lmul(parts):
    for tok in parts[1:]:
        tok_low = tok.lower().strip()
        if re.fullmatch(r'm(?:[1248]|f[248])', tok_low):
            return tok_low
    return 'm1'

def fix_vector_vm_operand_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    mnem = parts[0]
    if not mnem.startswith('v'):
        return line
    if len(parts) >= 2 and parts[-1].strip().lower() == 'vm':
        parts = parts[:-1]
        rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
        return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()
    return line

def fix_vadc_mask_operand_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    mnem = parts[0].lower()
    if mnem not in ('vadc.vim', 'vadc.vvm', 'vadc.vxm'):
        return line
    changed = False
    if len(parts) == 4:
        parts.append('v0')
        changed = True
    elif len(parts) >= 5:
        if parts[4].strip().lower() == 'v0.t':
            parts[4] = 'v0'
            changed = True
        elif parts[4].strip().lower() != 'v0':
            parts[4] = 'v0'
            changed = True
    if not changed:
        return line
    rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
    return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()

def fix_masked_vector_op_with_uninitialized_v0(lines):
    new_lines = []
    wrote_v0 = False
    for line in lines:
        parts = tokenize_asm(line)
        if not parts:
            new_lines.append(line)
            continue
        mnem = parts[0].lower()
        if instruction_writes_vector_register(parts, 'v0'):
            wrote_v0 = True
        if len(parts) >= 2 and parts[-1].strip().lower() == 'v0.t' and not wrote_v0:
            parts = parts[:-1]
            line = parts[0] + ' ' + ', '.join(parts[1:])
        new_lines.append(line)
    return new_lines

def fix_vector_mask_suffix_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    mnem = parts[0].lower()
    if mnem not in MASKABLE_MNEMS:
        return line
    changed = False
    if len(parts) >= 2 and parts[-1].strip().lower() == 'v0':
        parts[-1] = 'v0.t'
        changed = True
    if not changed:
        return line
    rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
    return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()

def normalize_vector_register_name(tok: str) -> str:
    tok = tok.strip()
    if tok.endswith('.t'):
        tok = tok[:-2]
    return tok

def is_vector_register(tok: str) -> bool:
    return re.fullmatch(r'v([0-9]|[12][0-9]|3[01])(?:\.t)?', tok.strip(), re.I) is not None

def parse_vector_register_index(tok: str) -> int | None:
    tok = tok.strip().lower()
    if tok.endswith('.t'):
        tok = tok[:-2]
    m = re.fullmatch(r'v([0-9]|[12][0-9]|3[01])', tok)
    if not m:
        return None
    return int(m.group(1))

def find_safe_scalar_temp_register(lines) -> str:
    used = set()
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        for tok in parts[1:]:
            tok2 = tok.strip().lower()
            if tok2 in REG_ALIAS:
                used.add(tok2)
    for cand in ['t6', 't5', 't4', 't3', 'a7', 'a6', 'a5']:
        if cand not in used:
            return cand
    return 't6'

def vector_register_written_before(lines, upto_idx: int, reg_name: str) -> bool:
    reg_name = normalize_vector_register_name(reg_name).lower()
    for i in range(upto_idx):
        parts = tokenize_asm(lines[i])
        if not parts or len(parts) < 2:
            continue
        mnem = parts[0].lower()
        dst = normalize_vector_register_name(parts[1]).lower() if is_vector_register(parts[1]) else None
        if dst == reg_name and (
            mnem.startswith('vmv.')
            or mnem.startswith('vle')
            or mnem.startswith('vl')
            or mnem.startswith('vadd')
            or mnem.startswith('vsub')
            or mnem.startswith('vand')
            or mnem.startswith('vor')
            or mnem.startswith('vxor')
            or mnem.startswith('vaadd')
            or mnem.startswith('vasub')
            or mnem.startswith('vadc')
            or mnem.startswith('vms')
            or mnem.startswith('vrgather')
            or mnem.startswith('vslide')
        ):
            return True
    return False

def needs_zero_init_for_vector_op(parts) -> str | None:
    if len(parts) < 3:
        return None
    mnem = parts[0].lower()
    vd = normalize_vector_register_name(parts[1])
    if not is_vector_register(vd):
        return None
    if mnem in {
        'vadd.vi', 'vand.vi',
        'vadd.vv', 'vand.vv', 'vsub.vv', 'vaadd.vv', 'vaaddu.vv', 'vasub.vv', 'vasubu.vv',
        'vadd.vx', 'vand.vx', 'vsub.vx', 'vaadd.vx', 'vaaddu.vx', 'vasub.vx', 'vasubu.vx',
    }:
        vs2 = normalize_vector_register_name(parts[2])
        if vd.lower() == vs2.lower():
            return vd
    return None

def FUNCTION_FIX_TEST_TEXT(TEXT_INPUT: str, PC_START: int = 0) -> str:
    blocks = split_blocks(TEXT_INPUT)
    fixed = [rewrite_block(b, PC_START=PC_START) for b in blocks]
    return '\n\n'.join(fixed).rstrip() + '\n'

def get_block_extension(block_text: str):
    for ln in block_text.splitlines():
        m = EXT_HEADER_RE.match(ln)
        if m:
            return m.group(1).strip()
    return None

def is_non_none_extension(ext: str) -> bool:
    return ext is not None and ext.strip().lower() != 'none'

def replace_invalid_register_aliases(text: str) -> str:
    mapping = {
        't7': 'a7',
        't8': 'x8',
        't9': 'x9',
        't10': 'x10',
        't11': 'x11',
        't12': 'x12',
        't13': 'x13',
    }
    for old, new in mapping.items():
        text = re.sub(rf'\b{re.escape(old)}\b', new, text)
    return text

def convert_binary_literals_to_decimal(text: str) -> str:
    def repl(match):
        value = int(match.group(1), 2)
        return str(value)
    return re.sub(r'\b0b([01_]+)\b', repl, text)

def fix_invalid_vector_4th_operand_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    mnem = parts[0].lower()
    if mnem not in MASKABLE_MNEMS:
        return line
    if len(parts) == 5:
        last = parts[4].strip().lower()
        if last not in ('v0.t', 'v0'):
            parts = parts[:4]
        elif last == 'v0':
            parts[4] = 'v0.t'
        rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
        return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()
    return line

def try_exec_vector_block(lines, PC_START: int = 0):
    try:
        return True, exec_vector_block(lines, PC_START=PC_START)
    except Exception:
        return False, None

def fix_li_label_to_la_in_line(line: str) -> str:
    if '#' in line:
        code, comment = line.split('#', 1)
        comment = '#' + comment
    else:
        code, comment = line, ''
    parts = tokenize_asm(code)
    if not parts:
        return line
    if parts[0].lower() != 'li':
        return line
    if len(parts) < 3:
        return line
    operand = parts[2].strip()
    if not is_numeric_token(operand):
        parts[0] = 'la'
        rebuilt = parts[0] + ' ' + ', '.join(parts[1:])
        return (rebuilt + (' ' if comment and not rebuilt.endswith(' ') else '') + comment).rstrip()
    return line

def block_has_only_supported_vector_mnems(lines):
    for ln in lines:
        parts = tokenize_asm(ln)
        if not parts:
            continue
        mnem = parts[0].lower()
        if mnem.startswith('v') and mnem not in SUPPORTED_VECTOR_MNEMS:
            return False
    return True

def fix_li_label_to_la_in_text(text: str) -> str:
    return '\n'.join(fix_li_label_to_la_in_line(line) for line in text.splitlines())
# FUNCTION - FINISH

# VARIABLE - BEGIN
SIMULATABLE_CSRS = {'satp'}

SAFE_BIN_OP = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.LShift: operator.lshift,
    ast.RShift: operator.rshift,
    ast.BitOr: operator.or_,
    ast.BitAnd: operator.and_,
    ast.BitXor: operator.xor,
}

SAFE_UNARY_OP = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
    ast.Invert: operator.invert,
}

PRIVILEGE_TRANSITION_CALLS = {
    'machine_to_supervisor',
    'machine_to_user',
    'machine_to_virtual',
    'supervisor_to_user',
    'machine_to_virtual_user',
    'vs_to_vu',
}

MAP_IMMEDIATE_RANGE = {
    'addi': (-2048, 2047), 'addiw': (-2048, 2047),
    'xori': (-2048, 2047), 'ori': (-2048, 2047), 'andi': (-2048, 2047),
    'slti': (-2048, 2047), 'sltiu': (-2048, 2047),
    'slli': (0, 63), 'srli': (0, 63), 'srai': (0, 63),
    'slliw': (0, 31), 'srliw': (0, 31), 'sraiw': (0, 31),
    'lui': (0, (1 << 20) - 1), 'auipc': (0, (1 << 20) - 1),
}

CSR_MNEMS = {
    'csrr', 'csrw', 'csrs', 'csrc',
    'csrrw', 'csrrs', 'csrrc',
    'csrrwi', 'csrrsi', 'csrrci',
}

REG_ALIAS = {
    **{f'x{i}': i for i in range(32)},
    'zero': 0, 'ra': 1, 'sp': 2, 'gp': 3, 'tp': 4,
    't0': 5, 't1': 6, 't2': 7,
    's0': 8, 'fp': 8, 's1': 9,
    'a0': 10, 'a1': 11, 'a2': 12, 'a3': 13, 'a4': 14, 'a5': 15, 'a6': 16, 'a7': 17,
    's2': 18, 's3': 19, 's4': 20, 's5': 21, 's6': 22, 's7': 23, 's8': 24, 's9': 25, 's10': 26, 's11': 27,
    't3': 28, 't4': 29, 't5': 30, 't6': 31,
}

MASK64 = (1 << 64) - 1
MASK32 = (1 << 32) - 1
_NUM_HEX_RE = re.compile(r'^\s*([+-]?)0[xX]([0-9a-fA-F_]+)\s*$')
_NUM_DEC_RE = re.compile(r'^\s*[-+]?\d+\s*$')
REGISTER_NAME_RE = re.compile(r'^\s*REGISTER\s*:\s*(.+?)\s*$', re.I)
REGISTER_NUMBER_RE = re.compile(r'^\s*NUMBER\s*:\s*(.+?)\s*$', re.I)
CSR_CANONICAL_NAME = build_csr_canonical_map(FILE_REGISTER_ALL)

MEM_MNEMS = {
    'lb','lbu','lh','lhu','lw','lwu','ld',
    'sb','sh','sw','sd',
    'flw','fsw','fld','fsd',
    'vle8.v', 'vle16.v', 'vle32.v', 'vle64.v',
    'vse8.v', 'vse16.v', 'vse32.v', 'vse64.v',
    'vsm.v',
}

VECTOR_IMM_RULES = [
    (re.compile(r'^vror\.vi$'), 0, 63),
    (re.compile(r'^(v(?:sll|srl|sra|ssll|ssrl|ssra|wsll)\.vi)$'), 0, 31),
    (re.compile(r'^(v(?:slidedown|slideup)\.vi)$'), 0, 31),
    (re.compile(r'^(vrgather\.vi)$'), 0, 31),
    (re.compile(r'^(v(?:nclip|nclipu)\.wi)$'), 0, 31),
    (re.compile(r'^(v(?:nsrl|nsra)\.wi)$'), 0, 31),
    (re.compile(r'^vaeskf[12]\.vi$'), 0, 31),
    (re.compile(r'^vsm3c\.vi$'), 0, 31),
    (re.compile(r'^vsm4k\.vi$'), 0, 31),
    (re.compile(r'^v.+\.vi(m)?$'), -16, 15),
]

COMPRESSED_IMM_RULES = [
    (re.compile(r'^c\.(?:addi|addiw|andi|li)$'), -32, 31),
    (re.compile(r'^c\.(?:slli|srli|srai)$'), 0, 63),
    (re.compile(r'^c\.lui$'), -(1 << 17), (1 << 17) - 1),
]

RESULT_RE = re.compile(
    r'^\s*#\s*Result\s*:\s*([a-zA-Z0-9]+)\s*=\s*([^\s#]+)',
    re.I
)

SUPPORTED_SCALAR_MNEMS = {
    # Existing handled
    'li', 'lui', 'auipc',
    'addi', 'andi', 'ori', 'xori', 'slti', 'sltiu', 'addiw',
    'slli', 'srli', 'srai', 'slliw', 'srliw', 'sraiw',
    'add', 'sub', 'and', 'or', 'xor', 'sll', 'srl', 'sra', 'slt', 'sltu',
    'andn', 'orn', 'xnor',
    'addw', 'subw', 'sllw', 'srlw', 'sraw',
    'mul', 'mulh', 'mulhsu', 'mulhu', 'div', 'divu', 'rem', 'remu',
    'mulw', 'divw', 'divuw', 'remw', 'remuw',
    'csrr', 'csrw',
    # Newly added (Section B)
    'add.uw', 'sh1add', 'sh2add', 'sh3add',
    'sh1add.uw', 'sh2add.uw', 'sh3add.uw', 'slli.uw',
    'clz', 'ctz', 'cpop', 'clzw', 'ctzw', 'cpopw',
    'max', 'maxu', 'min', 'minu',
    'rol', 'ror', 'rolw', 'rorw', 'rori', 'roriw',
    'sext.b', 'sext.h', 'sext.w', 'zext.h',
    'rev8', 'orc.b',
    'bclr', 'bset', 'binv', 'bext',
    'bclri', 'bseti', 'binvi', 'bexti',
    'clmul', 'clmulh', 'clmulr',
    'czero.eqz', 'czero.nez',
    'nop', 'mv', 'not', 'neg', 'negw', 'seqz', 'snez', 'sltz', 'sgtz',
    # Compressed
    'c.add', 'c.addi', 'c.addiw', 'c.addw', 'c.sub', 'c.subw',
    'c.and', 'c.andi', 'c.or', 'c.xor',
    'c.slli', 'c.srli', 'c.srai', 'c.lui', 'c.not', 'c.mul',
}

RESULT_RE2 = re.compile(r'^\s*#\s*Result\s*:\s*([a-zA-Z0-9]+)\s*=\s*(.+?)\s*$', re.I)

VECTOR_MEM_MNEMS = {
    'vle8.v', 'vle16.v', 'vle32.v', 'vle64.v',
    'vse8.v', 'vse16.v', 'vse32.v', 'vse64.v',
    'vle8ff.v', 'vle16ff.v', 'vle32ff.v', 'vle64ff.v',
    'vlm.v', 'vsm.v',
}

MASKABLE_MNEMS = {
    'vadd.vv', 'vadd.vx', 'vadd.vi',
    'vand.vv', 'vand.vx', 'vand.vi',
    'vsub.vv', 'vsub.vx',
    'vaadd.vv', 'vaadd.vx',
    'vaaddu.vv', 'vaaddu.vx',
    'vasub.vv', 'vasub.vx',
    'vasubu.vv', 'vasubu.vx',
    'vssra.vv', 'vssra.vx', 'vssra.vi',
    'vssrl.vv', 'vssrl.vx', 'vssrl.vi',
    'vsadd.vv', 'vsadd.vx', 'vsadd.vi',
    'vssub.vv', 'vssub.vx',
}

EXT_HEADER_RE = re.compile(
    r'^\s*#\s*(?:\d+\s*#\s*)?Extension\s+Must\s+Be\s+Active\s*:\s*([^\s#]+)\s*$',
    re.I
)

SUPPORTED_VECTOR_MNEMS = {
    'li', 'addi', 'csrw', 'csrwi', 'vsetvli',
    'vmset.m', 'vmclr.m',
    'vmv.v.x', 'vmv.v.i', 'vmv.s.x', 'vmv.x.s',
    'vfirst.m',
    'vslidedown.vi',
    'vmsne.vi', 'vmslt.vi', 'vmsltu.vi',
    'vadc.vim',
    'vadd.vi', 'vand.vi',
    'vadd.vv', 'vand.vv', 'vsub.vv', 'vaadd.vv', 'vaaddu.vv', 'vasub.vv', 'vasubu.vv',
    'vadd.vx', 'vand.vx', 'vsub.vx', 'vaadd.vx', 'vaaddu.vx', 'vasub.vx', 'vasubu.vx',
    'vssra.vv', 'vssra.vx', 'vssra.vi',
    'vssrl.vv', 'vssrl.vx', 'vssrl.vi',
    'vsadd.vv', 'vsadd.vx', 'vsadd.vi',
    'vssub.vv', 'vssub.vx',
}

VECTOR_TOKEN_FIX_MAP = {
    'vxsrm': 'vxrm',
    'vdup.vx': 'vmv.v.x',
    'vtrue.v': 'vmset.m',
    'vaddi.vx': 'vadd.vx',
    'vaddi.vi': 'vadd.vi',
    'vandi.vx': 'vand.vx',
    'vsubi.vx': 'vsub.vx',
    'vadc.vim.v': 'vadc.vim',
    'vadc.vvm.v': 'vadc.vvm',
    'vadc.vxm.v': 'vadc.vxm',
    'vmclear.m': 'vmclr.m',
    'vmset.v': 'vmset.m',
    'vmclr.v': 'vmclr.m',
}

fixed_blocks = []
fixed_blocks_ext = []
dropped_blocks = []
PC_INITIAL = 0x800001f2
# VARIABLE - FINISH

# CODE - BEGIN
with open(the_input, 'r', encoding='utf-8') as io:
        TEXT = io.read().strip()

TEXT = replace_invalid_register_aliases(TEXT)
TEXT = convert_binary_literals_to_decimal(TEXT)
TEXT = fix_li_label_to_la_in_text(TEXT)
TEXT = fix_la_expression_to_li_in_text(TEXT)
blocks = split_blocks(TEXT)

for b in blocks:
    fixed_b = rewrite_block(b, PC_START=PC_INITIAL)
    if fixed_b is None:
        dropped_blocks.append(b)
        continue
    fixed_blocks.append(fixed_b)
    ext = get_block_extension(b)
    if is_non_none_extension(ext):
        fixed_blocks_ext.append(fixed_b)

NEW_ALL = '\n\n'.join(fixed_blocks).rstrip()

with open(the_output, 'w', encoding='utf-8') as io:
    io.write(NEW_ALL)
# CODE - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 3 - CREATE - 3_TEST_CASE_REVISED_1.txt - END

0.07 second(s)

====
DONE
====


In [5]:
###########################
###########################
###########################

"""
Normalize expected results in RISC-V assembly test cases by converting complex CSR and vector register conditions into general-purpose register checks, handling trap results, and removing non-ASCII assembly lines.
"""

from pathlib import Path
import re

FILE_INPUT_OUTPUT = Path('6_TEST_CASE/3_TEST_CASE_REVISED_1.txt')

LIST_VECTOR_REGISTER = [f'v{i}' for i in range(32)]
SET_VECTOR_REGISTER = set(LIST_VECTOR_REGISTER)
SET_CSR_REGISTER = {
    'vl', 'vtype', 'vlenb', 'vstart', 'vxsat', 'vxrm', 'vcsr',
    'mstatus', 'sstatus', 'ustatus',
    'mepc', 'sepc', 'uepc',
    'mcause', 'scause', 'ucause',
    'mtval', 'stval', 'utval',
    'mtvec', 'stvec', 'utvec',
    'mie', 'sie', 'uie',
    'mip', 'sip', 'uip',
    'medeleg', 'mideleg',
    'menvcfg', 'senvcfg',
    'mcountinhibit', 'mcounteren', 'scounteren',
    'mscratch', 'sscratch', 'uscratch',
    'mcycle', 'minstret', 'cycle', 'instret',
    'time',
}

def FUNCTION_GET_CSR_USED_IN_ASSEMBLY(LIST_ASSEMBLY_LINE):
    SET_CSR_USED = set()

    for STRING_LINE in LIST_ASSEMBLY_LINE:
        STRING_CODE = STRING_LINE.split('#', 1)[0].strip()
        STRING_CODE = ' '.join(STRING_CODE.split())

        MATCH_CSR_INSTRUCTION = re.match(
            r'^(?:csrr|csrw|csrs|csrc|csrwi|csrsi|csrci|csrrw|csrrs|csrrc|csrrwi|csrrsi|csrrci)\b\s+(.*)$',
            STRING_CODE
        )

        if not MATCH_CSR_INSTRUCTION:
            continue

        LIST_OPERAND = [EACH_ITEM.strip() for EACH_ITEM in MATCH_CSR_INSTRUCTION.group(1).split(',')]

        if len(LIST_OPERAND) >= 2:
            STRING_OPCODE = STRING_CODE.split()[0]

            if STRING_OPCODE in {'csrw', 'csrs', 'csrc', 'csrwi', 'csrsi', 'csrci'}:
                SET_CSR_USED.add(LIST_OPERAND[0])
            else:
                SET_CSR_USED.add(LIST_OPERAND[1])

    return SET_CSR_USED

def FUNCTION_HAS_PRIVILEGE_TRANSITION_CALL(LIST_ASSEMBLY_LINE):
    SET_PRIVILEGE_CALL = {
        'machine_to_supervisor',
        'machine_to_user',
        'supervisor_to_user',
    }

    for STRING_LINE in LIST_ASSEMBLY_LINE:
        STRING_CODE = STRING_LINE.split('#', 1)[0].strip()
        STRING_CODE = ' '.join(STRING_CODE.split())

        MATCH_CALL = re.fullmatch(r'call\s+([A-Za-z_][A-Za-z0-9_]*)', STRING_CODE)

        if MATCH_CALL and MATCH_CALL.group(1) in SET_PRIVILEGE_CALL:
            return True

    return False
    
def FUNCTION_IS_ASCII(STRING_TEXT: str) -> bool:
    try:
        STRING_TEXT.encode('ascii')
        return True
    except UnicodeEncodeError:
        return False

def FUNCTION_NORMALIZE_INT(STRING_VALUE: str) -> str:
    STRING_VALUE = STRING_VALUE.strip()
    INT_VALUE = int(STRING_VALUE, 0)
    return hex(INT_VALUE) if INT_VALUE >= 10 else str(INT_VALUE)

def FUNCTION_GET_CURRENT_SEW(LIST_ASSEMBLY_LINE):
    STRING_SEW = None

    for STRING_LINE in LIST_ASSEMBLY_LINE:
        STRING_CODE = STRING_LINE.split('#', 1)[0].strip()

        MATCH = re.search(r'\be(\d+)\b', STRING_CODE)
        if MATCH:
            INT_SEW = int(MATCH.group(1))
            if INT_SEW in {8, 16, 32, 64}:
                STRING_SEW = f'e{INT_SEW}'

    return STRING_SEW or 'e8'

def FUNCTION_GET_VECTOR_STORE_AND_LOAD(STRING_VECTOR_REGISTER: str, STRING_EXPECTED: str, STRING_SEW: str, INT_INDEX: int):
    MAP_SEW = {
        'e8': {
            'store': 'vse8.v',
            'load': 'lbu',
            'byte': 1,
        },
        'e16': {
            'store': 'vse16.v',
            'load': 'lhu',
            'byte': 2,
        },
        'e32': {
            'store': 'vse32.v',
            'load': 'lwu',
            'byte': 4,
        },
        'e64': {
            'store': 'vse64.v',
            'load': 'ld',
            'byte': 8,
        },
    }

    DICT_INFO = MAP_SEW.get(STRING_SEW, MAP_SEW['e8'])
    INT_OFFSET = INT_INDEX * DICT_INFO['byte']

    LIST_ASSEMBLY = [
        f'{DICT_INFO["store"]} {STRING_VECTOR_REGISTER}, (a2)',
        f'{DICT_INFO["load"]} a0, {INT_OFFSET}(a2)',
    ]

    return LIST_ASSEMBLY, f'# Result: a0 = {STRING_EXPECTED}'

def FUNCTION_CONVERT_ONE_RESULT(STRING_RESULT_BODY: str, LIST_ASSEMBLY_LINE):
    STRING_RESULT_BODY = STRING_RESULT_BODY.strip()

    MATCH_BIT = re.fullmatch(
        r'([A-Za-z_][A-Za-z0-9_]*)\s*\[\s*(\d+)\s*\]\s*=\s*(-?(?:0x[0-9A-Fa-f]+|\d+))',
        STRING_RESULT_BODY
    )

    if MATCH_BIT:
        STRING_CSR = MATCH_BIT.group(1)
        STRING_BIT = MATCH_BIT.group(2)
        STRING_EXPECTED = FUNCTION_NORMALIZE_INT(MATCH_BIT.group(3))

        LIST_ASSEMBLY = [
            f'csrr a1, {STRING_CSR}',
            f'srli a0, a1, {STRING_BIT}',
            'andi a0, a0, 1',
        ]

        return LIST_ASSEMBLY, f'# Result: a0 = {STRING_EXPECTED}'

    MATCH_BIT_RANGE = re.fullmatch(
        r'([A-Za-z_][A-Za-z0-9_]*)\s*\[\s*(\d+)\s*:\s*(\d+)\s*\]\s*=\s*(-?(?:0x[0-9A-Fa-f]+|\d+))',
        STRING_RESULT_BODY
    )

    if MATCH_BIT_RANGE:
        STRING_CSR = MATCH_BIT_RANGE.group(1)
        INT_BIT_A = int(MATCH_BIT_RANGE.group(2))
        INT_BIT_B = int(MATCH_BIT_RANGE.group(3))
        INT_LSB = min(INT_BIT_A, INT_BIT_B)
        INT_MSB = max(INT_BIT_A, INT_BIT_B)
        INT_WIDTH = INT_MSB - INT_LSB + 1
        INT_MASK = (1 << INT_WIDTH) - 1
        STRING_EXPECTED = FUNCTION_NORMALIZE_INT(MATCH_BIT_RANGE.group(4))

        LIST_ASSEMBLY = [
            f'csrr a1, {STRING_CSR}',
            f'srli a0, a1, {INT_LSB}',
            f'andi a0, a0, {INT_MASK}',
        ]

        return LIST_ASSEMBLY, f'# Result: a0 = {STRING_EXPECTED}'

    MATCH_SHIFT_MASK = re.fullmatch(
        r'\(\s*([A-Za-z_][A-Za-z0-9_]*)\s*>>\s*(\d+)\s*\)\s*&\s*(0x[0-9A-Fa-f]+|\d+)\s*=\s*(-?(?:0x[0-9A-Fa-f]+|\d+))',
        STRING_RESULT_BODY
    )

    if MATCH_SHIFT_MASK:
        STRING_CSR = MATCH_SHIFT_MASK.group(1)
        STRING_SHIFT = MATCH_SHIFT_MASK.group(2)
        STRING_MASK = FUNCTION_NORMALIZE_INT(MATCH_SHIFT_MASK.group(3))
        STRING_EXPECTED = FUNCTION_NORMALIZE_INT(MATCH_SHIFT_MASK.group(4))

        LIST_ASSEMBLY = [
            f'csrr a1, {STRING_CSR}',
            f'srli a0, a1, {STRING_SHIFT}',
            f'andi a0, a0, {STRING_MASK}',
        ]

        return LIST_ASSEMBLY, f'# Result: a0 = {STRING_EXPECTED}'

    MATCH_CSR_AND_MASK = re.fullmatch(
        r'([A-Za-z_][A-Za-z0-9_]*)\s*&\s*(0x[0-9A-Fa-f]+|\d+)\s*=\s*(-?(?:0x[0-9A-Fa-f]+|\d+))',
        STRING_RESULT_BODY
    )

    if MATCH_CSR_AND_MASK:
        STRING_CSR = MATCH_CSR_AND_MASK.group(1)
        STRING_MASK = FUNCTION_NORMALIZE_INT(MATCH_CSR_AND_MASK.group(2))
        STRING_EXPECTED = FUNCTION_NORMALIZE_INT(MATCH_CSR_AND_MASK.group(3))

        LIST_ASSEMBLY = [
            f'csrr a1, {STRING_CSR}',
            f'andi a0, a1, {STRING_MASK}',
        ]

        return LIST_ASSEMBLY, f'# Result: a0 = {STRING_EXPECTED}'

    MATCH_REGISTER_OR_CSR_VALUE = re.fullmatch(
        r'((?:0x[0-9A-Fa-f]+)|(?:[A-Za-z_][A-Za-z0-9_]*))\s*=\s*(-?(?:0x[0-9A-Fa-f]+|\d+))',
        STRING_RESULT_BODY
    )
    
    if MATCH_REGISTER_OR_CSR_VALUE:
        STRING_NAME = MATCH_REGISTER_OR_CSR_VALUE.group(1)
        STRING_EXPECTED = FUNCTION_NORMALIZE_INT(MATCH_REGISTER_OR_CSR_VALUE.group(2))
    
        BOOL_NUMERIC_CSR = re.fullmatch(r'0x[0-9A-Fa-f]+', STRING_NAME) is not None
        SET_CSR_USED = FUNCTION_GET_CSR_USED_IN_ASSEMBLY(LIST_ASSEMBLY_LINE)
        BOOL_CSR_USED_IN_ASSEMBLY = STRING_NAME in SET_CSR_USED
    

    MATCH_VECTOR_SHIFT_EXPR = re.fullmatch(
        r'(v(?:[0-9]|[12][0-9]|3[01]))\s*\[\s*(\d+)\s*\]\s*=\s*'
        r'(v(?:[0-9]|[12][0-9]|3[01]))\s*\[\s*(\d+)\s*\]\s*<<\s*'
        r'(v(?:[0-9]|[12][0-9]|3[01]))\s*\[\s*(\d+)\s*\]',
        STRING_RESULT_BODY
    )

    if MATCH_VECTOR_SHIFT_EXPR:
        STRING_DEST_VECTOR = MATCH_VECTOR_SHIFT_EXPR.group(1)
        INT_DEST_INDEX = int(MATCH_VECTOR_SHIFT_EXPR.group(2))
        STRING_SOURCE_VECTOR = MATCH_VECTOR_SHIFT_EXPR.group(3)
        INT_SOURCE_INDEX = int(MATCH_VECTOR_SHIFT_EXPR.group(4))
        STRING_SHIFT_VECTOR = MATCH_VECTOR_SHIFT_EXPR.group(5)
        INT_SHIFT_INDEX = int(MATCH_VECTOR_SHIFT_EXPR.group(6))

        STRING_SEW = FUNCTION_GET_CURRENT_SEW(LIST_ASSEMBLY_LINE)

        MAP_SEW = {
            'e8': {
                'store': 'vse8.v',
                'load': 'lbu',
                'byte': 1,
                'shift_mask': 7,
                'value_mask': '0xff',
            },
            'e16': {
                'store': 'vse16.v',
                'load': 'lhu',
                'byte': 2,
                'shift_mask': 15,
                'value_mask': '0xffff',
            },
            'e32': {
                'store': 'vse32.v',
                'load': 'lwu',
                'byte': 4,
                'shift_mask': 31,
                'value_mask': '0xffffffff',
            },
            'e64': {
                'store': 'vse64.v',
                'load': 'ld',
                'byte': 8,
                'shift_mask': 63,
                'value_mask': None,
            },
        }

        DICT_INFO = MAP_SEW.get(STRING_SEW, MAP_SEW['e8'])

        INT_DEST_OFFSET = INT_DEST_INDEX * DICT_INFO['byte']
        INT_SOURCE_OFFSET = INT_SOURCE_INDEX * DICT_INFO['byte']
        INT_SHIFT_OFFSET = INT_SHIFT_INDEX * DICT_INFO['byte']

        LIST_ASSEMBLY = [
            f'{DICT_INFO["store"]} {STRING_DEST_VECTOR}, (a2)',
            f'{DICT_INFO["store"]} {STRING_SOURCE_VECTOR}, (a3)',
            f'{DICT_INFO["store"]} {STRING_SHIFT_VECTOR}, (a4)',
            f'{DICT_INFO["load"]} a0, {INT_DEST_OFFSET}(a2)',
            f'{DICT_INFO["load"]} a1, {INT_SOURCE_OFFSET}(a3)',
            f'{DICT_INFO["load"]} a5, {INT_SHIFT_OFFSET}(a4)',
            f'andi a5, a5, {DICT_INFO["shift_mask"]}',
            'sll a1, a1, a5',
        ]

        if DICT_INFO['value_mask'] is not None:
            LIST_ASSEMBLY.append(f'andi a1, a1, {DICT_INFO["value_mask"]}')

        return LIST_ASSEMBLY, '# Result: a0 = a1'

    MATCH_VECTOR_INDEX = re.fullmatch(
        r'(v(?:[0-9]|[12][0-9]|3[01]))\s*\[\s*(\d+)\s*\]\s*=\s*(-?(?:0x[0-9A-Fa-f]+|\d+))',
        STRING_RESULT_BODY
    )

    if MATCH_VECTOR_INDEX:
        STRING_VECTOR_REGISTER = MATCH_VECTOR_INDEX.group(1)
        INT_INDEX = int(MATCH_VECTOR_INDEX.group(2))
        STRING_EXPECTED = FUNCTION_NORMALIZE_INT(MATCH_VECTOR_INDEX.group(3))
        STRING_SEW = FUNCTION_GET_CURRENT_SEW(LIST_ASSEMBLY_LINE)

        return FUNCTION_GET_VECTOR_STORE_AND_LOAD(
            STRING_VECTOR_REGISTER=STRING_VECTOR_REGISTER,
            STRING_EXPECTED=STRING_EXPECTED,
            STRING_SEW=STRING_SEW,
            INT_INDEX=INT_INDEX,
        )

    return None, None

def FUNCTION_REMAP_RESULT_ASSEMBLY_REGISTER(LIST_ASSEMBLY, STRING_TARGET_REGISTER):
    LIST_OUTPUT = []

    for STRING_ASM in LIST_ASSEMBLY:
        STRING_ASM = re.sub(r'\ba0\b', STRING_TARGET_REGISTER, STRING_ASM)
        LIST_OUTPUT.append(STRING_ASM)

    return LIST_OUTPUT

def FUNCTION_CONVERT_RESULT_LINE(STRING_LINE: str, LIST_ASSEMBLY_LINE):
    MATCH_RESULT = re.match(r'^(\s*)#\s*Result\s*:\s*(.*?)\s*$', STRING_LINE)

    if not MATCH_RESULT:
        return None

    STRING_INDENT = MATCH_RESULT.group(1)
    STRING_BODY = MATCH_RESULT.group(2).strip()

    MATCH_MCAUSE = re.fullmatch(
        r'mcause\s*=\s*(-?(?:0x[0-9A-Fa-f]+|\d+))',
        STRING_BODY,
        re.I
    )
    
    if MATCH_MCAUSE:
        STRING_EXPECTED = FUNCTION_NORMALIZE_INT(MATCH_MCAUSE.group(1))
        return [STRING_INDENT + f'# Result: mcause = {STRING_EXPECTED} || t5 = {STRING_EXPECTED}']

    if '||' in STRING_BODY:
        return None

    LIST_PART = [EACH_PART.strip() for EACH_PART in STRING_BODY.split('&&')]

    LIST_NEW_ASSEMBLY = []
    LIST_NEW_RESULT = []

    LIST_TARGET_REGISTER = ['a0', 'a3', 'a4', 'a5', 'a6', 'a7']

    if len(LIST_PART) > len(LIST_TARGET_REGISTER):
        return None

    for INT_INDEX, STRING_PART in enumerate(LIST_PART):
        LIST_ASSEMBLY, STRING_NEW_RESULT = FUNCTION_CONVERT_ONE_RESULT(
            STRING_PART,
            LIST_ASSEMBLY_LINE
        )

        if LIST_ASSEMBLY is None:
            return None

        STRING_TARGET_REGISTER = LIST_TARGET_REGISTER[INT_INDEX]
        LIST_REMAPPED_ASSEMBLY = FUNCTION_REMAP_RESULT_ASSEMBLY_REGISTER(
            LIST_ASSEMBLY,
            STRING_TARGET_REGISTER
        )

        LIST_NEW_ASSEMBLY.extend(LIST_REMAPPED_ASSEMBLY)
        LIST_NEW_RESULT.append(
            STRING_NEW_RESULT.replace('# Result: a0', STRING_TARGET_REGISTER)
        )

    LIST_OUTPUT = [STRING_INDENT + EACH_LINE for EACH_LINE in LIST_NEW_ASSEMBLY]
    LIST_OUTPUT.append(STRING_INDENT + '# Result: ' + ' && '.join(LIST_NEW_RESULT))

    return LIST_OUTPUT

def FUNCTION_SHOULD_DROP_ASSEMBLY_LINE(STRING_LINE: str) -> bool:
    STRING_STRIPPED = STRING_LINE.strip()

    if not STRING_STRIPPED:
        return False

    if re.match(r'^\s*#\s*Result\s*:', STRING_LINE):
        return False

    if not FUNCTION_IS_ASCII(STRING_LINE):
        return True

    STRING_CODE = STRING_LINE.split('#', 1)[0].strip()

    if not STRING_CODE:
        return False

    LIST_TOKEN = re.findall(r'[^\s,()]+', STRING_CODE)

    for STRING_TOKEN in LIST_TOKEN:
        if not FUNCTION_IS_ASCII(STRING_TOKEN):
            return True

    return False

def FUNCTION_REWRITE_TEST_CASE_TEXT(STRING_TEXT: str) -> str:
    LIST_LINE = STRING_TEXT.replace('\r\n', '\n').replace('\r', '\n').split('\n')
    LIST_OUTPUT = []

    BOOL_INSIDE_ASSEMBLY = False
    LIST_CURRENT_ASSEMBLY_LINE = []

    INT_DROPPED_NON_ASCII = 0
    INT_CONVERTED_RESULT = 0
    INT_SKIPPED_RESULT = 0

    for STRING_LINE in LIST_LINE:
        if re.match(r'^\s*#\s+\d+\s+#\s+Extension Must Be Active:', STRING_LINE):
            BOOL_INSIDE_ASSEMBLY = False
            LIST_CURRENT_ASSEMBLY_LINE = []
            LIST_OUTPUT.append(STRING_LINE)
            continue

        if STRING_LINE.strip() == '##### ASSEMBLY #####':
            BOOL_INSIDE_ASSEMBLY = True
            LIST_CURRENT_ASSEMBLY_LINE = []
            LIST_OUTPUT.append(STRING_LINE)
            continue

        if BOOL_INSIDE_ASSEMBLY:
            if FUNCTION_SHOULD_DROP_ASSEMBLY_LINE(STRING_LINE):
                INT_DROPPED_NON_ASCII += 1
                continue

            if re.match(r'^\s*#\s*Result\s*:', STRING_LINE):
                LIST_CONVERTED = FUNCTION_CONVERT_RESULT_LINE(
                    STRING_LINE,
                    LIST_CURRENT_ASSEMBLY_LINE
                )

                if LIST_CONVERTED is not None:
                    LIST_OUTPUT.extend(LIST_CONVERTED)
                    INT_CONVERTED_RESULT += 1
                else:
                    LIST_OUTPUT.append(STRING_LINE)
                    INT_SKIPPED_RESULT += 1

                BOOL_INSIDE_ASSEMBLY = False
                LIST_CURRENT_ASSEMBLY_LINE = []
                continue

            LIST_CURRENT_ASSEMBLY_LINE.append(STRING_LINE)
            LIST_OUTPUT.append(STRING_LINE)
            continue

        LIST_OUTPUT.append(STRING_LINE)

    STRING_REVISED = '\n'.join(LIST_OUTPUT)

    print(f'Converted # Result lines: {INT_CONVERTED_RESULT}')
    print(f'Skipped # Result lines: {INT_SKIPPED_RESULT}')
    print(f'Dropped non-ASCII assembly/prose lines: {INT_DROPPED_NON_ASCII}')

    return STRING_REVISED

STRING_ORIGINAL = FILE_INPUT_OUTPUT.read_text(encoding='utf-8')
STRING_REVISED = FUNCTION_REWRITE_TEST_CASE_TEXT(STRING_ORIGINAL)

FILE_BACKUP = FILE_INPUT_OUTPUT.with_suffix(FILE_INPUT_OUTPUT.suffix + '.backup')
FILE_BACKUP.write_text(STRING_ORIGINAL, encoding='utf-8', newline='\n')
FILE_INPUT_OUTPUT.write_text(STRING_REVISED, encoding='utf-8', newline='\n')

print(f'Done. Backup saved to: {FILE_BACKUP}')
print(f'Run this to inspect changes:')
print(f'diff -u {FILE_BACKUP} {FILE_INPUT_OUTPUT}')

Converted # Result lines: 9
Skipped # Result lines: 157
Dropped non-ASCII assembly/prose lines: 163
Done. Backup saved to: 6_TEST_CASE/3_TEST_CASE_REVISED_1.txt.backup
Run this to inspect changes:
diff -u 6_TEST_CASE/3_TEST_CASE_REVISED_1.txt.backup 6_TEST_CASE/3_TEST_CASE_REVISED_1.txt


In [6]:
# 4 - CREATE - 3_TEST_CASE_REVISED_2.txt & 3_TEST_CASE_REVISED_2_NON_XV6.txt - START

"""
Prepare numbered RISC-V assembly test cases for xv6 and non-xv6 test harnesses, adapt register and CSR result conditions for non-xv6 execution, and distribute the test case files to the repair and emulator execution pipelines.
"""

# TIME START - BEGIN
import time
TIME_START = time.time()
# TIME START - FINISH

# IMPORT - BEGIN
from itertools import count
from pathlib import Path
import re
import shutil
# IMPORT - FINISH

# PATH - BEGIN
FILE_TEST_CASE = Path('6_TEST_CASE/3_TEST_CASE_REVISED_1.txt')
FILE_TEST_CASE_2 = Path('6_TEST_CASE/3_TEST_CASE_REVISED_2.txt') # xv6 normal
FILE_TEST_CASE_3 = Path('6_TEST_CASE/3_TEST_CASE_REVISED_2_NON_XV6.txt') # non xv6 special
FOLDER_REPAIR = Path('7_REPAIR_TEST_CASE/TEXT')
# PATH - FINISH

# FUNCTION - BEGIN
def FUNCTION_ADD_NUMBER_TO_HEADERS(text: str) -> str:
    counter = count(1)
    return REGEX_HEADER.sub(lambda match_value: f'# {next(counter)} {match_value.group(1)}', text)

def FUNCTION_CREATE_NON_XV6_TEXT(text: str) -> str:
    def FUNCTION_PARSE_INT(string_value: str):
        string_value = string_value.strip()
        try:
            return int(string_value, 0)
        except:
            return None

    MAP_GPR_ALIAS_TO_CANONICAL = {
        'zero': 'x0',  'x0': 'x0',
        'ra': 'x1',    'x1': 'x1',
        'sp': 'x2',    'x2': 'x2',
        'gp': 'x3',    'x3': 'x3',
        'tp': 'x4',    'x4': 'x4',
        't0': 'x5',    'x5': 'x5',
        't1': 'x6',    'x6': 'x6',
        't2': 'x7',    'x7': 'x7',
        's0': 'x8',    'fp': 'x8', 'x8': 'x8',
        's1': 'x9',    'x9': 'x9',
        'a0': 'x10',   'x10': 'x10',
        'a1': 'x11',   'x11': 'x11',
        'a2': 'x12',   'x12': 'x12',
        'a3': 'x13',   'x13': 'x13',
        'a4': 'x14',   'x14': 'x14',
        'a5': 'x15',   'x15': 'x15',
        'a6': 'x16',   'x16': 'x16',
        'a7': 'x17',   'x17': 'x17',
        's2': 'x18',   'x18': 'x18',
        's3': 'x19',   'x19': 'x19',
        's4': 'x20',   'x20': 'x20',
        's5': 'x21',   'x21': 'x21',
        's6': 'x22',   'x22': 'x22',
        's7': 'x23',   'x23': 'x23',
        's8': 'x24',   'x24': 'x24',
        's9': 'x25',   'x25': 'x25',
        's10': 'x26',  'x26': 'x26',
        's11': 'x27',  'x27': 'x27',
        't3': 'x28',   'x28': 'x28',
        't4': 'x29',   'x29': 'x29',
        't5': 'x30',   'x30': 'x30',
        't6': 'x31',   'x31': 'x31',
    }

    MAP_CANONICAL_TO_ABI = {
        'x5': 't0', 'x6': 't1', 'x7': 't2',
        'x28': 't3', 'x29': 't4', 'x30': 't5', 'x31': 't6',
        'x10': 'a0', 'x11': 'a1', 'x12': 'a2', 'x13': 'a3',
        'x14': 'a4', 'x15': 'a5', 'x16': 'a6', 'x17': 'a7',
        'x9': 's1', 'x18': 's2', 'x19': 's3', 'x20': 's4',
        'x21': 's5', 'x22': 's6', 'x23': 's7', 'x24': 's8',
        'x25': 's9', 'x26': 's10', 'x27': 's11', 'x8': 's0',
    }

    LIST_TEMP_POOL = [
        'x31', 'x29', 'x28', 'x7', 'x6', 'x5',      # t6, t4, t3, t2, t1, t0
        'x17', 'x16', 'x15', 'x14', 'x13', 'x12', 'x11', 'x10',   # a7..a0
        'x27', 'x26', 'x25', 'x24', 'x23', 'x22', 'x21', 'x20', 'x19', 'x18', 'x9', 'x8'  # s11..s0,s1
    ]

    def FUNCTION_CANONICALIZE_GPR(string_gpr: str) -> str:
        return MAP_GPR_ALIAS_TO_CANONICAL.get(string_gpr, string_gpr)

    def FUNCTION_ALLOCATE_TEMP(set_used_gpr, set_reserved_temp):
        for each_temp_canonical in LIST_TEMP_POOL:
            if each_temp_canonical in set_used_gpr:
                continue
            if each_temp_canonical in set_reserved_temp:
                continue
            set_reserved_temp.add(each_temp_canonical)
            return MAP_CANONICAL_TO_ABI[each_temp_canonical]
        raise ValueError(
            f'No free temp register available. '
            f'used={sorted(set_used_gpr)} reserved={sorted(set_reserved_temp)}'
        )

    list_block = text.strip().split('\n\n')
    list_block_output = []

    for each_block in list_block:
        list_line = each_block.split('\n')
        list_line_output = []

        for each_line in list_line:
            stripped_line = each_line.strip()

            if stripped_line.startswith('# Result:'):
                string_prefix = '# Result:'
                string_body = stripped_line[len(string_prefix):].strip()
                if '|' in string_body:
                    string_separator = '|'
                elif '&' in string_body:
                    string_separator = '&'
                else:
                    string_separator = None
                
                if string_separator:
                    list_condition = [item.strip() for item in string_body.split(string_separator)]
                else:
                    list_condition = [string_body.strip()]
                list_condition_parsed = []
                set_used_gpr = set()

                for each_condition in list_condition:
                    match_value = re.match(
                        r'^([A-Za-z_][A-Za-z0-9_]*(?:\[\d+(?::\d+)?\])?)\s*(!=|=)\s*(.+)$',
                        each_condition
                    )
                    if not match_value:
                        list_condition_parsed.append({'kind': 'raw', 'text': each_condition})
                        continue

                    string_lhs = match_value.group(1).strip()
                    string_op = match_value.group(2).strip()
                    string_rhs = match_value.group(3).strip()

                    match_field = re.match(r'^([A-Za-z_][A-Za-z0-9_]*)\[(\d+)(?::(\d+))?\]$', string_lhs)
                    if match_field:
                        string_base_name = match_field.group(1)
                        int_bit_1 = int(match_field.group(2))
                        int_bit_2 = int(match_field.group(3)) if match_field.group(3) is not None else int_bit_1
                        
                        int_msb = max(int_bit_1, int_bit_2)
                        int_lsb = min(int_bit_1, int_bit_2)
                    
                        if string_base_name in SET_NORMAL_REGISTERS:
                            set_used_gpr.add(FUNCTION_CANONICALIZE_GPR(string_base_name))
                            list_condition_parsed.append({
                                'kind': 'gpr_field',
                                'lhs': string_lhs,
                                'gpr': string_base_name,
                                'msb': int_msb,
                                'lsb': int_lsb,
                                'op': string_op,
                                'rhs': string_rhs,
                                'text': each_condition
                            })
                        else:
                            list_condition_parsed.append({
                                'kind': 'csr_field',
                                'lhs': string_lhs,
                                'csr': string_base_name,
                                'msb': int_msb,
                                'lsb': int_lsb,
                                'op': string_op,
                                'rhs': string_rhs,
                                'text': each_condition
                            })

                flag_need_edit = any(each_item['kind'] in {'csr', 'csr_field', 'gpr_field'} for each_item in list_condition_parsed)
                if not flag_need_edit:
                    list_line_output.append(each_line)
                    continue

                set_reserved_temp = set()
                map_csr_to_temp = {}
                list_extra_instruction = []
                list_condition_output = []

                for each_item in list_condition_parsed:
                    if each_item['kind'] == 'raw':
                        list_condition_output.append(each_item['text'])
                        continue

                    if each_item['kind'] == 'gpr':
                        list_condition_output.append(each_item['text'])
                        continue

                    if each_item['kind'] == 'csr':
                        string_csr_name = each_item['csr']
                        if string_csr_name == 'mcause':
                            list_condition_output.append(f't5 {each_item["op"]} {each_item["rhs"]}')
                            continue

                        if string_csr_name not in map_csr_to_temp:
                            map_csr_to_temp[string_csr_name] = FUNCTION_ALLOCATE_TEMP(set_used_gpr, set_reserved_temp)
                            list_extra_instruction.append(f'csrr {map_csr_to_temp[string_csr_name]}, {string_csr_name}')

                        list_condition_output.append(f'{map_csr_to_temp[string_csr_name]} {each_item["op"]} {each_item["rhs"]}')
                        continue

                    if each_item['kind'] == 'gpr_field':
                        string_gpr_name = each_item['gpr']
                        int_msb = each_item['msb']
                        int_lsb = each_item['lsb']
                        int_width = int_msb - int_lsb + 1
                        int_mask = (1 << int_width) - 1
                    
                        string_temp_value = FUNCTION_ALLOCATE_TEMP(set_used_gpr, set_reserved_temp)
                        list_extra_instruction.append(f'mv {string_temp_value}, {string_gpr_name}')
                    
                        if int_lsb != 0:
                            list_extra_instruction.append(f'srli {string_temp_value}, {string_temp_value}, {int_lsb}')
                    
                        if int_width == 1:
                            list_extra_instruction.append(f'andi {string_temp_value}, {string_temp_value}, 0x1')
                        else:
                            if int_mask <= 0xfff:
                                list_extra_instruction.append(f'andi {string_temp_value}, {string_temp_value}, {hex(int_mask)}')
                            else:
                                string_temp_mask = FUNCTION_ALLOCATE_TEMP(set_used_gpr, set_reserved_temp)
                                list_extra_instruction.append(f'li {string_temp_mask}, {hex(int_mask)}')
                                list_extra_instruction.append(f'and {string_temp_value}, {string_temp_value}, {string_temp_mask}')
                    
                        list_condition_output.append(f'{string_temp_value} {each_item["op"]} {each_item["rhs"]}')
                        continue

                    if each_item['kind'] == 'csr_field':
                        string_csr_name = each_item['csr']
                        int_msb = each_item['msb']
                        int_lsb = each_item['lsb']
                        int_width = int_msb - int_lsb + 1
                        int_mask = (1 << int_width) - 1

                        if int_width == 1 and int_msb == 63 and int_lsb == 63:
                            string_temp_mask = FUNCTION_ALLOCATE_TEMP(set_used_gpr, set_reserved_temp)
                            string_temp_value = FUNCTION_ALLOCATE_TEMP(set_used_gpr, set_reserved_temp)
                            list_extra_instruction.append(f'li {string_temp_mask}, 1 << 63')
                            list_extra_instruction.append(f'csrr {string_temp_value}, {string_csr_name}')
                            list_extra_instruction.append(f'and {string_temp_mask}, {string_temp_mask}, {string_temp_value}')

                            int_rhs = FUNCTION_PARSE_INT(each_item['rhs'])
                            if int_rhs is not None:
                                string_rhs_output = hex((1 << 63) if int_rhs != 0 else 0x0)
                            else:
                                string_rhs_output = each_item['rhs']

                            list_condition_output.append(f'{string_temp_mask} {each_item["op"]} {string_rhs_output}')
                            continue

                        string_temp_value = FUNCTION_ALLOCATE_TEMP(set_used_gpr, set_reserved_temp)
                        list_extra_instruction.append(f'csrr {string_temp_value}, {string_csr_name}')

                        if int_lsb != 0:
                            list_extra_instruction.append(f'srli {string_temp_value}, {string_temp_value}, {int_lsb}')

                        if int_width == 1:
                            list_extra_instruction.append(f'andi {string_temp_value}, {string_temp_value}, 0x1')
                        else:
                            if int_mask <= 0xfff:
                                list_extra_instruction.append(f'andi {string_temp_value}, {string_temp_value}, {hex(int_mask)}')
                            else:
                                string_temp_mask = FUNCTION_ALLOCATE_TEMP(set_used_gpr, set_reserved_temp)
                                list_extra_instruction.append(f'li {string_temp_mask}, {hex(int_mask)}')
                                list_extra_instruction.append(f'and {string_temp_value}, {string_temp_value}, {string_temp_mask}')

                        list_condition_output.append(f'{string_temp_value} {each_item["op"]} {each_item["rhs"]}')
                        continue

                for each_instruction in list_extra_instruction:
                    list_line_output.append(each_instruction)

                string_join = f' {string_separator} ' if string_separator else ''
                list_line_output.append(f'{string_prefix} ' + (string_join.join(list_condition_output) if string_separator else list_condition_output[0]))
                continue

            list_line_output.append(each_line)

        list_block_output.append('\n'.join(list_line_output))

    return '\n\n'.join(list_block_output).strip()
# FUNCTION - FINISH

# VARIABLE - BEGIN
REGEX_HEADER = re.compile(r'(?m)^(#\s*Extension Must Be Active:.*)$')
REGEX_RESULT_EQUAL = re.compile(r'^(#\s*Result:\s*)([A-Za-z_][A-Za-z0-9_]*)\s*=\s*(.+)$')

SET_NORMAL_REGISTERS = {
    'zero', 'ra', 'sp', 'gp', 'tp',
    't0', 't1', 't2', 't3', 't4', 't5', 't6',
    's0', 's1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11',
    'a0', 'a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7',
    'fp',
    'x0', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7', 'x8', 'x9', 'x10', 'x11', 'x12', 'x13', 'x14', 'x15',
    'x16', 'x17', 'x18', 'x19', 'x20', 'x21', 'x22', 'x23', 'x24', 'x25', 'x26', 'x27', 'x28', 'x29', 'x30', 'x31',
    'ft0', 'ft1', 'ft2', 'ft3', 'ft4', 'ft5', 'ft6', 'ft7', 'ft8', 'ft9', 'ft10', 'ft11',
    'fs0', 'fs1', 'fs2', 'fs3', 'fs4', 'fs5', 'fs6', 'fs7', 'fs8', 'fs9', 'fs10', 'fs11',
    'fa0', 'fa1', 'fa2', 'fa3', 'fa4', 'fa5', 'fa6', 'fa7',
    'f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15',
    'f16', 'f17', 'f18', 'f19', 'f20', 'f21', 'f22', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'f30', 'f31',
    'v0', 'v1', 'v2', 'v3', 'v4', 'v5', 'v6', 'v7', 'v8', 'v9', 'v10', 'v11', 'v12', 'v13', 'v14', 'v15',
    'v16', 'v17', 'v18', 'v19', 'v20', 'v21', 'v22', 'v23', 'v24', 'v25', 'v26', 'v27', 'v28', 'v29', 'v30', 'v31',
}

STRING_TEXT_RAW = FILE_TEST_CASE.read_text(encoding='utf-8').replace('\r\n', '\n')
STRING_TEXT_RAW = re.sub(r'^(# Result:.+?)\s*//\s*', r'\1 # ', STRING_TEXT_RAW, flags=re.MULTILINE)
STRING_TEXT_XV6 = FUNCTION_ADD_NUMBER_TO_HEADERS(STRING_TEXT_RAW)
STRING_TEXT_NON_XV6 = FUNCTION_CREATE_NON_XV6_TEXT(STRING_TEXT_XV6)
# VARIABLE - FINISH

# CODE - BEGIN
FILE_TEST_CASE_2.write_text(STRING_TEXT_XV6, encoding='utf-8', newline='\n')
FILE_TEST_CASE_3.write_text(STRING_TEXT_NON_XV6, encoding='utf-8', newline='\n')
(FOLDER_REPAIR / '1_REPAIR_TEST_CASE_INPUT.txt').write_text(STRING_TEXT_XV6, encoding='utf-8')
(FOLDER_REPAIR / 'REPAIR_TEST_CASE_RAW.txt').write_text(STRING_TEXT_XV6, encoding='utf-8')
STRING_REGISTER_TEXT = Path('5_REGISTER/1_OUTPUT/1_REGISTER_ALL.txt').read_text(encoding='utf-8').strip()
(FOLDER_REPAIR / 'REPAIR_TEST_CASE_REGISTER.txt').write_text(STRING_REGISTER_TEXT, encoding='utf-8')
# CODE - FINISH

# BONUS - BEGIN
SRC = FILE_TEST_CASE_2
DST = Path('8_EMULATOR/1_QEMU_XV6/TEXT/1_QEMU_XV6_INPUT.txt')
shutil.copy(SRC, DST)
SRC = FILE_TEST_CASE_2
DST = Path('8_EMULATOR/2_QEMU/TEXT/1_QEMU_INPUT.txt')
shutil.copy(SRC, DST)
DST = Path('8_EMULATOR/3_SPIKE/TEXT/1_SPIKE_INPUT.txt')
shutil.copy(SRC, DST)
DST = Path('8_EMULATOR/4_RVVM/TEXT/1_RVVM_INPUT.txt')
shutil.copy(SRC, DST)
DST = Path('8_EMULATOR/5_RENODE/TEXT/1_RENODE_INPUT.txt')
shutil.copy(SRC, DST)
# BONUS - FINISH

# TIME END - BEGIN
TIME_END = time.time()
DURATION_IN_SECOND = TIME_END - TIME_START
DAYS = int(DURATION_IN_SECOND // 86400)
DURATION_IN_SECOND %= 86400
HOURS = int(DURATION_IN_SECOND // 3600)
DURATION_IN_SECOND %= 3600
MINUTES = int(DURATION_IN_SECOND // 60)
DURATION_IN_SECOND %= 60
PARTS = []
if DAYS:
    PARTS.append(f'{DAYS} day(s)')
if HOURS:
    PARTS.append(f'{HOURS} hour(s)')
if MINUTES:
    PARTS.append(f'{MINUTES} minute(s)')
if DURATION_IN_SECOND or not PARTS:
    PARTS.append(f'{DURATION_IN_SECOND:.2f} second(s)')
print(' '.join(PARTS))
# TIME END - FINISH

# DONE - BEGIN
print('\n====\nDONE\n====')
# DONE - FINISH

# 4 - CREATE - 3_TEST_CASE_REVISED_2.txt & 3_TEST_CASE_REVISED_2_NON_XV6.txt - END

0.00 second(s)

====
DONE
====
